# Care Finder India — Project Notebook

**Healthcare facility search with service-specific evidence checks**

This notebook brings together our initial data exploration, record checks, service-matching experiments, and evaluation results. The final Streamlit application source is included at the end and is also available as `app.py` in the repository.

### How to use this notebook

We originally developed and ran these phases as separate Databricks notebooks. Some sections reuse variable names, rely on different intermediate tables or files, or need extra packages for the NLP experiments. **This is a record of our work and results, not a notebook designed to run top to bottom in one session.** Run individual sections in their original Databricks environment with the required setup, rather than assuming that one uninterrupted run will work.

The saved outputs are from earlier experiment runs and have not been regenerated as part of the notebook merge.

### Data access

The project uses the Virtue Foundation facility dataset through Databricks, including the source table `databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities` and the processed application table `workspace.default.care_finder_facilities`. Some evaluations read intermediate CSV files stored under `/Volumes/workspace/default/care_finder_files/`. The five evaluation CSV files are included in the repository's `data/` folder. You can rerun the **saved evaluation metrics** locally using the self-contained section below (requires pandas, but not Databricks). The earlier data exploration and experiment-generation steps still require access to Databricks and the specified tables. See `README.md` for details. Phase 5 labels were drafted with AI assistance and all 140 were manually reviewed by a team member. This review evaluates recorded text evidence, **not independent verification of facility services**.

### Important limitation

Recorded service claims and coordinates are not the same as confirmed current hospital readiness. Care Finder is a discovery tool, not a live service-availability checker or an emergency-dispatch system.

## Project stages

1. Initial data exploration and first search prototype
2. Phase 1: service evidence audit
3. Phase 2: record-quality review
4. Phase 2 follow-up: claim attribution checks
5. Phase 3: context-aware matching and NLP pilots
6. Phase 4: method evaluation
7. Phase 5: additional validation
8. Final Streamlit app source

The NLP and evaluation work was exploratory. The user-facing app relies on rule-based service matching and recorded location checks; experimental scores are not clinical validation.

## Run the saved evaluation results without Databricks

The `data/` folder contains the reviewed Phase 4 sample, the Phase 5 provisional-label sample, and the saved predictions. The following two cells can be run locally or in Databricks **without access to the original facility table**, as long as the CSV files are beside this notebook in `data/` and pandas is installed. Set `CARE_FINDER_DATA_DIR` if your files are stored somewhere else.

These calculations reproduce metrics from **saved predictions**, not an independent clinical or service-availability validation. Phase 5 labels were drafted with AI assistance and all 140 were subsequently reviewed manually. They are not clinically verified hospital-service labels. Other notebook sections remain separate Databricks experiments.

In [ ]:
from pathlib import Path
import os
import pandas as pd

# By default, read the CSV files from the data folder in this repository.
DATA_DIR = Path(os.environ.get("CARE_FINDER_DATA_DIR", "data"))
if not DATA_DIR.exists():
    print(f"Data folder not found: {DATA_DIR.resolve()}")
    print("Set CARE_FINDER_DATA_DIR to the folder containing the evaluation CSVs.")
else:
    print("Evaluation files:", DATA_DIR.resolve())

In [ ]:
def show_scores(reference, predicted, title):
    # Treat direct evidence as the positive class.
    actual = reference.eq("direct").astype(int)
    predicted = predicted.astype(int)
    tp = int(((actual == 1) & (predicted == 1)).sum())
    fp = int(((actual == 0) & (predicted == 1)).sum())
    fn = int(((actual == 1) & (predicted == 0)).sum())
    tn = int(((actual == 0) & (predicted == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    print(f"{title}: precision={precision:.3f}, recall={recall:.3f}, F1={f1:.3f}")
    print(f"  TP={tp} FP={fp} FN={fn} TN={tn}")

phase4 = pd.read_csv(DATA_DIR / "care_finder_phase4_review_labeled.csv")
phase5 = pd.read_csv(DATA_DIR / "care_finder_phase5_predictions.csv")

print("Phase 4: evaluation against the supplied review labels")
show_scores(phase4["gold_label"], phase4["regex_match"], "Regex baseline")
print("\nPhase 5: evaluation against AI-assisted, manually reviewed labels")
show_scores(phase5["gold_label"], phase5["nli_pred"], "NLI")
show_scores(phase5["gold_label"], phase5["hybrid_pred"], "Hybrid")

print("\nNote: these are text-label metrics, not confirmed hospital service accuracy.")

---

## Initial exploration and first search prototype

First look at the Virtue Foundation data, service columns, coordinate checks, and early search tests.

*Source notebook: `New Notebook 2026-11-21 19_38_14 (1).ipynb`*

In [0]:
# Load the hospital dataset
facilities_df = spark.table(
    "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities"
)

# Check the number of records
print("Total facilities:", facilities_df.count())

# Display all column names and data types
facilities_df.printSchema()

Total facilities: 10088
root
 |-- unique_id: string (nullable = true)
 |-- source_types: string (nullable = true)
 |-- source_ids: string (nullable = true)
 |-- source_content_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- organization_type: string (nullable = true)
 |-- content_table_id: string (nullable = true)
 |-- phone_numbers: string (nullable = true)
 |-- officialPhone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- websites: string (nullable = true)
 |-- officialWebsite: string (nullable = true)
 |-- yearEstablished: string (nullable = true)
 |-- acceptsVolunteers: string (nullable = true)
 |-- facebookLink: string (nullable = true)
 |-- address_line1: string (nullable = true)
 |-- address_line2: string (nullable = true)
 |-- address_line3: string (nullable = true)
 |-- address_city: string (nullable = true)
 |-- address_stateOrRegion: string (nullable = true)
 |-- address_zipOrPostcode: string (nullable = true)
 |-- address_country: s

In [0]:
# Show important information from 5 hospitals
selected_cols = [
    "unique_id",
    "name",
    "address_city",
    "address_stateOrRegion",
    "latitude",
    "longitude",
    "specialties",
    "capability",
    "procedure",
    "equipment",
    "description",
    "source_urls"
]

display(facilities_df.select(*selected_cols).limit(5))

unique_id,name,address_city,address_stateOrRegion,latitude,longitude,specialties,capability,procedure,equipment,description,source_urls
fa8ffca9-cb88-4ce6-b4f5-04ee5be45592,Dr Satish Jindal's Child Care Clinic,Bathinda,Punjab,30.197689056396484,74.9603271484375,"[""pediatrics"",""familyMedicine"",""familyMedicine"",""pediatrics""]",null,"[""Provides routine pediatric consultations"",""Administers pediatric vaccinations"",""Performs routine pediatric check-ups"",""Provides nutritional guidance"",""Offers tele-consultation services""]","[""Stethoscope is used by clinicians during pediatric examinations""]","Offers consultations, day care services, and vaccinations.","[""http://drsatishjindal.com/"",""https://www.facebook.com/1369953236450419"",null]"
fac0cc70-81a9-462d-bced-5ca40257b027,Sanjivani Multi Speciality Hospital,Chengannur,Alappuzha,59.949684143066406,-38.26263427734375,"[""internalMedicine"",""internalMedicine"",""hospiceAndPalliativeInternalMedicine"",""internalMedicine"",""internalMedicine"",""internalMedicine"",""internalMedicine"",""internalMedicine"",""internalMedicine"",""hospiceAndPalliativeInternalMedicine"",""internalMedicine"",""internalMedicine"",""anesthesia"",""cardiology"",""dentistry"",""dermatology"",""otolaryngology"",""gastroenterology"",""endocrinologyAndDiabetesAndMetabolism"",""internalMedicine"",""generalSurgery"",""nephrology"",""neurology"",""oralAndMaxillofacialSurgery"",""ophthalmology"",""orthopedicSurgery"",""pulmonology"",""physicalMedicineAndRehabilitation"",""psychiatry"",""radiology"",""urology"",""dietetics"",""painManagementInterventional"",""neonatologyPerinatalMedicine"",""gynecologyAndObstetrics"",""pediatrics"",""reproductiveEndocrinologyAndInfertility"",""breastImaging"",""emergencyMedicine"",""anesthesia"",""cardiology"",""dental"",""dermatology"",""ent"",""gastroenterology"",""endocrinology"",""generalMedicine"",""generalSurgery"",""nephrology"",""neurology""]","[""Provides outpatient, inpatient, and home care services"",""Morphine is available on site"",""Staff includes Dr. Darshana Pillai and Dr. Girish Sitharam"",""Located in Kollakadavu, Chengannur, Alappuzha district, Kerala"",""Multispeciality hospital"",""Open 24 Hrs"",""Has 17 years of operation in healthcare"",""Appears on Venmony map labeled as Sanjivani Multispeciality Hospital in Venmony, Kerala, India"",""NABH Entry Level accreditation"",""ISO 9001-2015 certified"",""Has more than 100 beds"",""Has 60 doctors"",""Has 30 departments"",""Centre of Excellence in Fertility Medicine"",""Centre of Excellence in Neonatology"",""Centre of Excellence in Obstetrics & Gynecology"",""Centre of Excellence in Pediatrics"",""NICU Level III"",""24/7 Emergency Department"",""Ambulance service available 24/7"",""24 Hour Services: Emergency Department, CT Scan, Ambulance, Mammogram, Pharmacy, Laboratory, Radiology, Blood Bank"",""Has 24/7 Emergency Department"",""Offers 24/7 CT scanning"",""Offers 24/7 mammography services"",""Provides ambulance service"",""Provides on-site laboratory services"",""Maintains a Blood Bank"",""Has Department of Radiology"",""Provides Physiotherapy services"",""Has 3 parking spaces"",""Free WIFI on premises"",""Every floor reception"",""Center of Excellence in Fertility Medicine"",""Center of Excellence in Neonatology"",""Center of Excellence in Obstetrics & Gynecology"",""Center of Excellence in Pediatrics"",""Departments include General Medicine, Obstetrics & Gynecology, Anaesthesia, Cardiology, Dermatology, Endocrinology, ENT, Nephrology & Dialysis, General & Laparoscopic Surgery, Neurology, Ophthalmology, Neonatology, Pediatrics, Psychiatry & Psychology, Pulmonology, Orthopedics, Pain Management Interventional, Physical Rehabilitation, Maxillo Facial Surgery, Radiology, Urology"",""Dermatology department offers medical, pediatric, surgical and cosmetic dermatology and treats melanoma, acne, lupus, bullous pemphigoid and pemphigus vulgaris"",""Dermatology department doctors include Dr. Susan Kuruvila a

In [0]:

from pyspark.sql import functions as F

# Columns needed for the first search tests.
selected_cols = [
    "name", "address_city", "address_stateOrRegion",
    "latitude", "longitude", "description",
    "specialties", "capability", "procedure",
    "equipment", "source_urls"
]

# Count missing or empty values
results = []

for col in selected_cols:
    missing = facilities_df.filter(
        F.col(col).isNull() |
        (F.trim(F.col(col).cast("string")) == "") |
        (F.lower(F.trim(F.col(col).cast("string"))) == "null")
    ).count()

    results.append((col, missing, round(missing / facilities_df.count() * 100, 2)))

missing_df = spark.createDataFrame(
    results, ["column", "missing_count", "missing_pct"]
)

display(missing_df)

# Identify coordinates outside an approximate India bounding box
invalid_coords = facilities_df.filter(
    F.col("latitude").isNotNull() &
    F.col("longitude").isNotNull() &
    (
        ~F.col("latitude").between(6, 38) |
        ~F.col("longitude").between(68, 98)
    )
)

print("Coordinates outside approximate India bounds:", invalid_coords.count())

display(
    invalid_coords.select(
        "name", "address_city", "address_stateOrRegion",
        "latitude", "longitude"
    ).limit(10)
)


column,missing_count,missing_pct
name,65,0.64
address_city,64,0.63
address_stateOrRegion,63,0.62
latitude,118,1.17
longitude,118,1.17
description,82,0.81
specialties,116,1.15
capability,122,1.21
procedure,146,1.45
equipment,225,2.23


Coordinates outside approximate India bounds: 6


name,address_city,address_stateOrRegion,latitude,longitude
Cura Imaging & Gastro Clinic,Nagpur,Maharashtra,2.9529342651367188,41.38715744018555
Hzb Arogyam Multispeciality Hospital,Hazaribagh,Jharkhand,46.07322692871094,106.171875
The Family Tree Hospital,Tirupati,Andhra Pradesh,32.96258544921875,7.479839324951172
Sanjivani Multi Speciality Hospital,Chengannur,Alappuzha,59.949684143066406,-38.26263427734375
Krishna Hospital Multispeciality,Lucknow,Uttar Pradesh,-81.706298828125,26.953109741210938
Cardia Health Care,Noida,Uttar Pradesh,7.710991382598877,109.6875


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType

# Columns containing medical information as JSON strings
service_cols = [
    "specialties",
    "capability",
    "procedure",
    "equipment",
    "source_urls"
]

# Parse the service lists into new columns.
df_clean = facilities_df

for col in service_cols:
    df_clean = df_clean.withColumn(
        col + "_list",
        F.from_json(F.col(col), ArrayType(StringType()))
    )

# Check a few parsed rows.
display(
    df_clean.select(
        "name",
        "capability_list",
        "procedure_list",
        "equipment_list"
    ).limit(5)
)

# Count rows where parsing failed or the list is empty
for col in service_cols:
    parsed = col + "_list"

    empty_count = df_clean.filter(
        F.col(parsed).isNull() |
        (F.size(F.col(parsed)) == 0)
    ).count()

    print(col, "missing/empty lists:", empty_count)


name,capability_list,procedure_list,equipment_list
Dr Satish Jindal's Child Care Clinic,null,"List(Provides routine pediatric consultations, Administers pediatric vaccinations, Performs routine pediatric check-ups, Provides nutritional guidance, Offers tele-consultation services)",List(Stethoscope is used by clinicians during pediatric examinations)
Sanjivani Multi Speciality Hospital,"List(Provides outpatient, inpatient, and home care services, Morphine is available on site, Staff includes Dr. Darshana Pillai and Dr. Girish Sitharam, Located in Kollakadavu, Chengannur, Alappuzha district, Kerala, Multispeciality hospital, Open 24 Hrs, Has 17 years of operation in healthcare, Appears on Venmony map labeled as Sanjivani Multispeciality Hospital in Venmony, Kerala, India, NABH Entry Level accreditation, ISO 9001-2015 certified, Has more than 100 beds, Has 60 doctors, Has 30 departments, Centre of Excellence in Fertility Medicine, Centre of Excellence in Neonatology, Centre of Excellence in Obstetrics & Gynecology, Centre of Excellence in Pediatrics, NICU Level III, 24/7 Emergency Department, Ambulance service available 24/7, 24 Hour Services: Emergency Department, CT Scan, Ambulance, Mammogram, Pharmacy, Laboratory, Radiology, Blood Bank, Has 24/7 Emergency Department, Offers 24/7 CT scanning, Offers 24/7 mammography services, Provides ambulance service, Provides on-site laboratory services, Maintains a Blood Bank, Has Department of Radiology, Provides Physiotherapy services, Has 3 parking spaces, Free WIFI on premises, Every floor reception, Center of Excellence in Fertility Medicine, Center of Excellence in Neonatology, Center of Excellence in Obstetrics & Gynecology, Center of Excellence in Pediatrics, Departments include General Medicine, Obstetrics & Gynecology, Anaesthesia, Cardiology, Dermatology, Endocrinology, ENT, Nephrology & Dialysis, General & Laparoscopic Surgery, Neurology, Ophthalmology, Neonatology, Pediatrics, Psychiatry & Psychology, Pulmonology, Orthopedics, Pain Management Interventional, Physical Rehabilitation, Maxillo Facial Surgery, Radiology, Urology, Dermatology department offers medical, pediatric, surgical and cosmetic dermatology and treats melanoma, acne, lupus, bullous pemphigoid and pemphigus vulgaris, Dermatology department doctors include Dr. Susan Kuruvila and Dr. Ann Willie Thomas, 24/7 Emergency Department with 24-hour emergency care, Centre of Excellence in Fertility Medicine, Centre of Excellence in Neonatology, Centre of Excellence in Obstetrics & Gynecology, Centre of Excellence in Pediatrics, Ophthalmology department with glaucoma specialist, Sanjivani Multi Speciality Hospital is a multispeciality hospital, Provides 24/7 emergency services, Located in Kollakadavu, Chengannur, Alappuzha district, Located opposite Alacode Temple, Canteen is located within Sanjivani Multispeciality Hospital in Venmony, Kerala, India)","List(Offers Intra Uterine Insemination (IUI) as a fertility treatment, Offers In Vitro Fertilization (IVF) as a fertility treatment, Offers Intra Cytoplasmic Sperm Injection (ICSI) as a fertility treatment, Offers CT scan for diagnostic imaging, Offers Mammogram for breast imaging, Provides CT scanning, Provides mammography services, CT Scan (diagnostic imaging) available, Mammogram (mammography) available, Computed Tomography (CT) scanning is available on-site, Mammography is available on-site, General and laparoscopic surgical procedures are offered)","List(CT scanner, Mammography equipment, Radiology department imaging equipment, Ambulance vehicles for patient transport, On-site CT scanner, On-site mammography equipment, Radiology department facilities, On-site CT scanner, On-site mammography unit, On-site laboratory equipment, On-site blood bank)"
Shaurya Hospital,"List(Shaurya Hospital offers 24/7 emergency care, Shaurya Hospital is a multispecialty hospital, Shaurya Hospital provides on-site diagnostic services including X-ray and blood laboratory, Shaurya Hospital of

specialties missing/empty lists: 118
capability missing/empty lists: 144
procedure missing/empty lists: 872
equipment missing/empty lists: 2408
source_urls missing/empty lists: 118


In [0]:

import re
from pyspark.sql import functions as F

# Supported services and common terms used in hospital records
service_terms = {
    "Dialysis": [
        r"\bdialysis\b",
        r"\bhemodialysis\b",
        r"\bhaemodialysis\b"
    ],
    "NICU": [
        r"\bnicu\b",
        r"\bneonatal\s+(?:intensive\s+care|icu)\b"
    ],
    "Emergency C-Section": [
        r"\bemergency\s+(?:c[\s-]?section|caesarean|cesarean|lscs)\b",
        r"\b(?:c[\s-]?section|caesarean|cesarean)\s+(?:delivery|section|surgery)\b"
    ],
    "MRI": [
        r"\bmri\b",
        r"\bmagnetic\s+resonance\s+imaging\b"
    ],
    "Trauma Care": [
        r"\btrauma\s+(?:care|surgery|center|centre|services)\b",
        r"\bpolytrauma\b"
    ],
    "Burn Care": [
        r"\bburn\s+(?:care|unit|centre|center|treatment)\b",
        r"\bburns?\s+unit\b"
    ]
}

# Combine individual service claims into a single searchable list
df_services = df_clean.withColumn(
    "evidence_list",
    F.concat(
        F.coalesce(F.col("capability_list"), F.array().cast("array<string>")),
        F.coalesce(F.col("procedure_list"), F.array().cast("array<string>")),
        F.coalesce(F.col("equipment_list"), F.array().cast("array<string>"))
    )
)

# Function to find supporting statements for a service
def find_evidence(claims, service):
    if not claims or service not in service_terms:
        return []

    patterns = service_terms[service]
    matches = []

    for claim in claims:
        if not claim:
            continue

        text = str(claim)

        if any(re.search(p, text, re.IGNORECASE) for p in patterns):
            matches.append(text)

    return list(dict.fromkeys(matches))

# Test matching on the actual facilities
sample = df_services.select(
    "name", "evidence_list"
).limit(100).collect()

for row in sample:
    matches = find_evidence(row["evidence_list"], "Dialysis")

    if matches:
        print("\nHospital:", row["name"])
        print("Evidence:", matches[:3])



Hospital: Sanjivani Multi Speciality Hospital
Evidence: ['Departments include General Medicine, Obstetrics & Gynecology, Anaesthesia, Cardiology, Dermatology, Endocrinology, ENT, Nephrology & Dialysis, General & Laparoscopic Surgery, Neurology, Ophthalmology, Neonatology, Pediatrics, Psychiatry & Psychology, Pulmonology, Orthopedics, Pain Management Interventional, Physical Rehabilitation, Maxillo Facial Surgery, Radiology, Urology']

Hospital: Paras Health, Gurugram
Evidence: ['Hemodialysis is offered', 'Peritoneal dialysis is offered']

Hospital: Veda Hospital & Kidney Centre
Evidence: ['Hemodialysis', 'Peritoneal dialysis', 'Provides hemodialysis and plasmapheresis services']

Hospital: India Kidney Hospital and Dialysis Centre
Evidence: ['Specializes in kidney care and dialysis services', 'Specializes in nephrology and dialysis', 'Provides dialysis treatment']

Hospital: Chavan Hospital
Evidence: ['The hospital provides FREE & Paid Dialysis Center']


In [0]:

import re

# Statements suggesting a service may not be available
negative_patterns = [
    r"\bnot (?:available|offered|provided)\b",
    r"\b(?:no|without|lacks?)\s+(?:dialysis|hemodialysis|haemodialysis)\b",
    r"\b(?:dialysis|hemodialysis|haemodialysis)\s+(?:is\s+)?(?:unavailable|discontinued|suspended)\b"
]

# Phrases indicating an explicit dialysis service
strong_patterns = [
    r"\b(?:offers?|provides?|performs?)\b.{0,50}\b(?:dialysis|hemodialysis|haemodialysis)\b",
    r"\b(?:dialysis|hemodialysis|haemodialysis)\b.{0,30}\b(?:offered|available|provided|treatment|services?|cent(?:er|re))\b",
    r"\b(?:hemodialysis|haemodialysis|peritoneal dialysis)\b"
]

def classify_dialysis_evidence(claims):
    matches = find_evidence(claims, "Dialysis")

    if not matches:
        return "No direct evidence", [], []

    positive = []
    negative = []

    for claim in matches:
        text = claim.lower()

        if any(re.search(p, text) for p in negative_patterns):
            negative.append(claim)
        else:
            positive.append(claim)

    if negative and positive:
        level = "Conflicting"
    elif negative:
        level = "Negative evidence"
    elif any(
        re.search(p, claim.lower())
        for claim in positive
        for p in strong_patterns
    ):
        level = "Strong"
    else:
        level = "Moderate"

    return level, positive, negative


# Test on the five hospitals we identified
hospital_names = [
    "Sanjivani Multi Speciality Hospital",
    "Paras Health, Gurugram",
    "Veda Hospital & Kidney Centre",
    "India Kidney Hospital and Dialysis Centre",
    "Chavan Hospital"
]

test_rows = (
    df_services
    .filter(F.col("name").isin(hospital_names))
    .select(
        "name", "evidence_list",
        "latitude", "longitude"
    )
    .collect()
)

for row in test_rows:
    level, positive, negative = classify_dialysis_evidence(
        row["evidence_list"]
    )

    lat = row["latitude"]
    lon = row["longitude"]

    valid_location = (
        lat is not None and lon is not None
        and 6 <= lat <= 38
        and 68 <= lon <= 98
    )

    print("\nHospital:", row["name"])
    print("Evidence level:", level)
    print("Coordinates within India bounds:", valid_location)
    print("Supporting statements:", positive[:2])
    print("Contradictory statements:", negative[:2])



Hospital: Sanjivani Multi Speciality Hospital
Evidence level: Strong
Coordinates within India bounds: True
Supporting statements: ['Offers kidney dialysis services', 'Dialysis units']
Contradictory statements: []

Hospital: Chavan Hospital
Evidence level: Strong
Coordinates within India bounds: True
Supporting statements: ['Provides dialysis services, including free dialysis for underprivileged patients']
Contradictory statements: []

Hospital: Sanjivani Multi Speciality Hospital
Evidence level: Moderate
Coordinates within India bounds: False
Supporting statements: ['Departments include General Medicine, Obstetrics & Gynecology, Anaesthesia, Cardiology, Dermatology, Endocrinology, ENT, Nephrology & Dialysis, General & Laparoscopic Surgery, Neurology, Ophthalmology, Neonatology, Pediatrics, Psychiatry & Psychology, Pulmonology, Orthopedics, Pain Management Interventional, Physical Rehabilitation, Maxillo Facial Surgery, Radiology, Urology']
Contradictory statements: []

Hospital: Paras

In [0]:

from pyspark.sql import functions as F

# Hospitals that appeared more than once
hospital_names = [
    "Sanjivani Multi Speciality Hospital",
    "Chavan Hospital"
]

# Compare identity, location, and source information
duplicate_check = (
    df_services
    .filter(F.col("name").isin(hospital_names))
    .select(
        "unique_id",
        "name",
        "address_city",
        "address_stateOrRegion",
        "address_line1",
        "latitude",
        "longitude",
        "cluster_id",
        "officialWebsite",
        "source_urls"
    )
    .orderBy("name", "address_city")
)

display(duplicate_check)

# Check how widespread repeated names are
name_counts = (
    facilities_df
    .filter(F.col("name").isNotNull())
    .groupBy("name")
    .count()
    .filter(F.col("count") > 1)
)

print("Names appearing more than once:", name_counts.count())

print(
    "Records with repeated names:",
    name_counts.agg(F.sum("count")).first()[0]
)


unique_id name address_city address_stateOrRegion address_line1 latitude longitude cluster_id officialWebsite source_urls 027972c1-8235-4690-95d4-0cbff3bcb3bb Chavan Hospital Mumbai Maharashtra Dattamandir Cross Road No-1 19.208019256591797 72.83660888671875 027972c1-8235-4690-95d4-0cbff3bcb3bb chavanhospital.org ["https://www.chavanhospital.org/service/dental/","https://www.chavanhospital.org/","https://www.chavanhospital.org/service/ophthalmology/",null] 3e4097c5-e8d4-434a-ba26-1ddf4ae9ae63 Chavan Hospital Pune Maharashtra Plot Number 54, Stadium Road, Vastu Udyog 18.060657501220703 75.7997055053711 08f6081c1360598603e0e9740a8fe97a chavanhospital.com ["https://chavanhospital.com/","http://www.onefivenine.com/india/villages/Nashik/Niphad/Lasalgaon","http://www.onefivenine.com/india/Places/Taluk/Jalgaon/Chalisgaon/Hospital","http://www.chavanhospital.com/","https://www.onefivenine.com/india/Places/Taluk/Jalgaon/Chalisgaon/Hospital","https://www.practo.com/pune/hospital/chavan-eye-hospital-pimpri","https://www.practo.com/pune/hospital/sharad-orthopaedic-hospital-sinhagad-road","http://www.onefivenine.com/india/Places/Taluk/Satara/Karad/Hospital","https://www.docindia.org/india/mh/sangli/ayurveda","https://www.chavanhospital.com/doctor-details.php","http://www.onefivenine.com/india/Places/Taluk/Kolhapur/Radhanagari/Hospital","https://www.practo.com/pune/hospital/vinod-memorial-multispeciality-hospital-vishrantwadi/about","https://www.justdial.com/Mumbai/Chavan-Hospital-Near-Dhanukarwadi-Opposite-Lic-Office-Kandivali-West/022P8403616_BZDET","http://www.onefivenine.com/india/Places/Taluk/Kolhapur/Chandgad/Hospital","https://www.myupchar.com/en/doctors/pune/pimpri-chinchwad/ophthalmology/dr-nabanita-neogi-baidya-62482","https://www.practo.com/pune/clinic/varad-joint-spine-clinic-wakad","https://www.hexahealth.com/pune/doctor/dr-nabnita-baidya-ophthalmologist","https://www.practo.com/pune/doctor/dr-priya-mangaonkar-ophthalmologist","https://www.practo.com/pune/clinic/the-knee-klinik-mc-viman-nagar","https://www.practo.com/pune/clinic/nityanand-institute-medical-science-rahabilitation-centrer-wadogaon-maval/psychiatrist-ps-39","https://www.practo.com/pune/doctor/dr-kaahushik-sh-ophthalmologist","https://www.practo.com/pune/hospital/lifeline-hospital-aundh-1/services","https://www.practo.com/pune/hospital/ruby-hall-clinic-wanowarie","https://dialysis.therenalproject.com/dialysis-center-in-nashik/","https://www.practo.com/pune/doctor/dr-bhagyesh-kulkarni-general-physician","https://www.magicbricks.com/flats-for-rent-in-poonam-sagar-mumbai-pppfr","https://dialysis.therenalproject.com/dialysis-center-in-pune/","https://www.99acres.com/morwadi-pune-overview-piffid","https://www.justdial.com/Mumbai/Dr-Subhash-Parek-Chavan-Hospital-Near-Dhanukarwadi-Opposite-Lic-Office-Kandivali-West/022PXX22-XX22-240422220037-R1E3_BZDET","https://www.practo.com/pimpri-chinchwad/hospital/asg-eye-hospital-11-sector-21-pimpri-chinchwad/services","http://www.onefivenine.com/india/Places/Taluk/Kolhapur/Kagal/Hospital","https://www.policyx.com/health-insurance/network-hospitals/future-generali-pune-maharashtra/","https://dialysis.therenalproject.com/dialysis-center-in-borivali/","https://www.practo.com/pune/clinic/ivf-nest-fertility-clinic-wakad-1/infertility-specialist-ps-329","https://mapcarta.com/N7632306106","https://www.proptiger.com/nashik/rane-nagar-59679/hospitals","https://www.proptiger.com/nashik/anandwalli-gaon-53064/hospitals","https://www.practo.com/pune/hospital/suryaprabha-nursing-home-paud-road-2/doctors","https://www.practo.com/pune/doctor/dr-amol-dilip-joshi-ear-nose-throat-ent-specialist","https://mapcarta.com/N7653308089","https://www.practo.com/pune/hospital/dr-kachale-multispecialty-hospital-narayangaon/services","https://www.proptiger.com/nashik/panchavati-60196/hospitals","https://www.facebook.com/275200755884878","https://www.practo.com/pune/clinic/wellness-health-clinic-1-nehrunagar","https://chotu.com/local/dental-clinics-in-nagpur-mah

Names appearing more than once: 320
Records with repeated names: 823


In [0]:

from pyspark.sql import functions as F

# Keep separate facility rows for now.
df_ready = df_services

# Flag coordinates missing or outside approximate India bounds
df_ready = df_ready.withColumn(
    "location_status",
    F.when(
        F.col("latitude").isNull() |
        F.col("longitude").isNull(),
        "Missing coordinates"
    ).when(
        ~F.col("latitude").between(6, 38) |
        ~F.col("longitude").between(68, 98),
        "Outside India bounds"
    ).otherwise("Within India bounds - not city verified")
)

# Count how many facilities fall into each category
display(
    df_ready.groupBy("location_status")
    .count()
    .orderBy(F.desc("count"))
)

# Identify names that occur more than once
name_counts = (
    df_ready
    .filter(F.col("name").isNotNull())
    .groupBy("name")
    .agg(F.count("*").alias("name_count"))
)

df_ready = df_ready.join(
    name_counts,
    on="name",
    how="left"
).withColumn(
    "repeated_name",
    F.coalesce(F.col("name_count") > 1, F.lit(False))
)

# Check the suspicious hospitals
display(
    df_ready.filter(
        F.col("name").isin(
            "Sanjivani Multi Speciality Hospital",
            "Chavan Hospital"
        )
    ).select(
        "unique_id",
        "name",
        "address_city",
        "latitude",
        "longitude",
        "location_status",
        "repeated_name"
    )
)


location_status,count
Within India bounds - not city verified,9964
Missing coordinates,118
Outside India bounds,6


unique_id,name,address_city,latitude,longitude,location_status,repeated_name
6f7bd0c4-2d30-4a7f-a2c0-f29b18ff8443,Sanjivani Multi Speciality Hospital,Surat,20.91231346130371,70.36070251464844,Within India bounds - not city verified,true
3e4097c5-e8d4-434a-ba26-1ddf4ae9ae63,Chavan Hospital,Pune,18.060657501220703,75.7997055053711,Within India bounds - not city verified,true
fac0cc70-81a9-462d-bced-5ca40257b027,Sanjivani Multi Speciality Hospital,Chengannur,59.949684143066406,-38.26263427734375,Outside India bounds,true
027972c1-8235-4690-95d4-0cbff3bcb3bb,Chavan Hospital,Mumbai,19.208019256591797,72.83660888671875,Within India bounds - not city verified,true


In [0]:

import math
import json
import re
from pyspark.sql import functions as F

# Example search
service = "Dialysis"
city = "Patna"
state = "Bihar"

# Approximate coordinates of central Patna
user_lat = 25.5941
user_lon = 85.1376

# Calculate straight-line distance in kilometers
def distance_km(lat1, lon1, lat2, lon2):
    r = 6371

    a1 = math.radians(lat1)
    a2 = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(a1) * math.cos(a2)
        * math.sin(dlon / 2) ** 2
    )

    return 2 * r * math.asin(min(1, math.sqrt(a)))

# Select facilities recorded in Patna, Bihar
candidates = (
    df_ready
    .filter(F.lower(F.trim("address_city")) == city.lower())
    .filter(
        F.lower(F.trim("address_stateOrRegion")) == state.lower()
    )
    .filter(
        F.col("location_status")
        == "Within India bounds - not city verified"
    )
    .select(
        "unique_id", "name", "address_city",
        "address_stateOrRegion", "latitude",
        "longitude", "evidence_list", "source_urls"
    )
    .collect()
)

print("Facilities recorded in Patna:", len(candidates))

results = []

for row in candidates:
    level, positive, negative = classify_dialysis_evidence(
        row["evidence_list"]
    )

    # Require strong or moderate positive evidence
    if level not in ["Strong", "Moderate"]:
        continue

    dist = distance_km(
        user_lat, user_lon,
        row["latitude"], row["longitude"]
    )

    # Initial city-coordinate consistency screen
    if dist > 60:
        continue

    evidence_score = 70 if level == "Strong" else 40
    distance_score = 30 * max(0, 1 - dist / 60)
    score = round(evidence_score + distance_score, 1)

    # Parse source links
    try:
        sources = json.loads(row["source_urls"] or "[]")
        sources = [s for s in sources if isinstance(s, str)]
    except (ValueError, TypeError):
        sources = []

    results.append({
        "id": row["unique_id"],
        "hospital": row["name"],
        "distance_km": round(dist, 1),
        "evidence_level": level,
        "ranking_score": score,
        "supporting_text": positive[:3],
        "source_urls": sources[:3]
    })

# Rank eligible facilities
results.sort(
    key=lambda x: (-x["ranking_score"], x["distance_km"])
)

print("Eligible facilities:", len(results))

for i, hospital in enumerate(results[:5], 1):
    print(f"\n{i}. {hospital['hospital']}")
    print("Distance:", hospital["distance_km"], "km")
    print("Evidence:", hospital["evidence_level"])
    print("Ranking score:", hospital["ranking_score"])
    print("Supporting text:", hospital["supporting_text"])
    print("Source URLs:", hospital["source_urls"])


Facilities recorded in Patna: 135
Eligible facilities: 23

1. Nectar Hospital
Distance: 1.5 km
Evidence: Strong
Ranking score: 99.2
Supporting text: ['Performs dialysis treatment', 'Provides dialysis treatment in a dedicated dialysis unit', 'Dialysis unit']
Source URLs: ['https://www.facebook.com/TheNectarHospital/', 'https://www.justdial.com/Patna/Nectar-Hospital-Near-Chandan-Hero-Pc-Colony-Pc-Colony/0612PX612-X612-220503221309-S2K2_BZDET', 'http://www.nectarhospital.in/']

2. Advanced Neuro Hospital
Distance: 1.7 km
Evidence: Strong
Ranking score: 99.2
Supporting text: ['Offers a multi-speciality hospital with departments including neurology, neuro-surgery, maxillofacial surgery, gastroenterology, laparoscopy, urology, general surgery, hematology-oncology, endocrinology, nephrology, electrophysiology, hemodialysis, and echocardiography', 'Nephrology treatments including hemodialysis', 'Hemodialysis is provided']
Source URLs: ['https://www.convicare.com/neurologists/advanced-neuro-hos

In [0]:

from pyspark.sql import functions as F

# Count facility records with direct text matches for each service
all_rows = df_ready.select(
    "unique_id", "evidence_list"
).collect()

service_counts = []

for service in service_terms:
    count = 0

    for row in all_rows:
        matches = find_evidence(
            row["evidence_list"], service
        )

        if matches:
            count += 1

    service_counts.append((service, count))

# Display service coverage
coverage_df = spark.createDataFrame(
    service_counts,
    ["service", "matching_facilities"]
)

display(
    coverage_df.orderBy(
        F.desc("matching_facilities")
    )
)


service,matching_facilities
Dialysis,1036
MRI,941
Trauma Care,920
NICU,721
Emergency C-Section,362
Burn Care,62


In [0]:

import math
import json
import re
from pyspark.sql import functions as F

# Starting locations for testing
city_locations = {
    "Patna": (25.5941, 85.1376, "Bihar"),
    "Jaipur": (26.9124, 75.7873, "Rajasthan"),
    "Lucknow": (26.8467, 80.9462, "Uttar Pradesh"),
    "Pune": (18.5204, 73.8567, "Maharashtra"),
    "Ahmedabad": (23.0225, 72.5714, "Gujarat"),
    "Delhi": (28.6139, 77.2090, "Delhi")
}

def get_evidence(claims, service):
    matches = find_evidence(claims, service)

    if not matches:
        return "No direct evidence", []

    # Reuse our dialysis-specific classifier
    if service == "Dialysis":
        level, positive, negative = classify_dialysis_evidence(claims)

        if negative:
            return "Conflicting", []

        return level, positive

    # Conservative first-pass negation check
    negative_terms = [
        r"\bnot available\b",
        r"\bnot offered\b",
        r"\bnot provided\b",
        r"\bunavailable\b",
        r"\bdiscontinued\b",
        r"\bno longer\b"
    ]

    positive = []
    negative = []

    for claim in matches:
        if any(re.search(p, claim, re.I) for p in negative_terms):
            negative.append(claim)
        else:
            positive.append(claim)

    if negative:
        return "Conflicting", []

    if not positive:
        return "No direct evidence", []

    # For now, all non-dialysis text matches
    # remain unverified candidate evidence.
    return "Candidate match", positive


def search_hospitals(service, city, top_n=5):
    if service not in service_terms:
        print("Unsupported service")
        return []

    if city not in city_locations:
        print("City not available in this test version")
        return []

    user_lat, user_lon, state = city_locations[city]

    # Use city and state together to reduce location ambiguity
    rows = (
        df_ready
        .filter(F.lower(F.trim("address_city")) == city.lower())
        .filter(F.lower(F.trim("address_stateOrRegion")) == state.lower())
        .filter(
            F.col("location_status")
            == "Within India bounds - not city verified"
        )
        .select(
            "unique_id", "name", "latitude", "longitude",
            "evidence_list", "source_urls"
        )
        .collect()
    )

    results = []

    for row in rows:
        level, evidence = get_evidence(
            row["evidence_list"], service
        )

        if level in ["No direct evidence", "Conflicting"]:
            continue

        dist = distance_km(
            user_lat, user_lon,
            row["latitude"], row["longitude"]
        )

        # Flag coordinates that seem too far from the selected city.
        if dist > 60:
            continue

        # Evidence-based score for exploratory ranking
        evidence_points = {
            "Strong": 70,
            "Moderate": 40,
            "Candidate match": 30
        }.get(level, 0)

        distance_points = 30 * max(0, 1 - dist / 60)
        score = round(evidence_points + distance_points, 1)

        try:
            urls = json.loads(row["source_urls"] or "[]")
            urls = [u for u in urls if isinstance(u, str)]
        except (ValueError, TypeError):
            urls = []

        results.append({
            "id": row["unique_id"],
            "hospital": row["name"],
            "service": service,
            "city": city,
            "distance_km": round(dist, 1),
            "evidence_level": level,
            "score": score,
            "evidence": evidence[:3],
            "source_urls": urls[:3]
        })

    results.sort(
        key=lambda x: (-x["score"], x["distance_km"], x["id"])
    )

    print("Service:", service)
    print("Location:", city)
    print("Candidates:", len(rows))
    print("Matched records after location screening:", len(results))

    for i, hospital in enumerate(results[:top_n], 1):
        print(f"\n{i}. {hospital['hospital']}")
        print("Distance:", hospital["distance_km"], "km")
        print("Evidence level:", hospital["evidence_level"])
        print("Prototype score:", hospital["score"])
        print("Supporting text:", hospital["evidence"])

    return results[:top_n]


# Test three different searches
dialysis_results = search_hospitals("Dialysis", "Patna")
nicu_results = search_hospitals("NICU", "Pune")
trauma_results = search_hospitals("Trauma Care", "Ahmedabad")


Service: Dialysis
Location: Patna
Candidates: 135
Matched records after location screening: 23

1. Nectar Hospital
Distance: 1.5 km
Evidence level: Strong
Prototype score: 99.2
Supporting text: ['Performs dialysis treatment', 'Provides dialysis treatment in a dedicated dialysis unit', 'Dialysis unit']

2. Advanced Neuro Hospital
Distance: 1.7 km
Evidence level: Strong
Prototype score: 99.2
Supporting text: ['Offers a multi-speciality hospital with departments including neurology, neuro-surgery, maxillofacial surgery, gastroenterology, laparoscopy, urology, general surgery, hematology-oncology, endocrinology, nephrology, electrophysiology, hemodialysis, and echocardiography', 'Nephrology treatments including hemodialysis', 'Hemodialysis is provided']

3. Satyavrat Hospital
Distance: 1.8 km
Evidence level: Strong
Prototype score: 99.1
Supporting text: ['Hemodialysis unit available', 'Departments include Orthopedics, Neurology, Gynaecology, Anesthesia, Critical Care, Laparoscopic Surgery,

In [0]:

import re

# Explicit service evidence patterns
strong_service_patterns = {
    "Dialysis": [
        r"\b(?:provides?|offers?|performs?)\b.{0,80}\bdialysis\b",
        r"\b(?:dialysis|hemodialysis|haemodialysis)\b.{0,45}\b(?:unit|center|centre|services?|treatment|available|offered|provided|performed)\b",
        r"\b(?:hemodialysis|haemodialysis|peritoneal dialysis)\b"
    ],
    "NICU": [
        r"\bnicu\b.{0,45}\b(?:available|on.site|unit|facilities|beds|cots|capacity)\b",
        r"\b(?:has|provides?|offers?|includes?)\b.{0,65}\bnicu\b",
        r"\bneonatal intensive care unit\b"
    ],
    "MRI": [
        r"\bmri\b.{0,45}\b(?:available|scanner|machine|services?|facility|on.site)\b",
        r"\b(?:offers?|provides?|performs?|has)\b.{0,60}\bmri\b",
        r"\bmagnetic resonance imaging\b.{0,35}\b(?:available|services?|facility)\b"
    ],
    "Trauma Care": [
        r"\b(?:provides?|offers?|performs?)\b.{0,60}\btrauma\b",
        r"\btrauma\b.{0,40}\b(?:care|surgery|center|centre|services?|unit)\b",
        r"\bpolytrauma\b.{0,35}\b(?:surgery|treatment|care)\b"
    ],
    "Burn Care": [
        r"\b(?:provides?|offers?|treats?)\b.{0,60}\bburns?\b",
        r"\bburns?\b.{0,45}\b(?:care|unit|center|centre|treatment|ward)\b"
    ],
    "Emergency C-Section": [
        r"\bemergency\b.{0,50}\b(?:c.section|cesarean|caesarean|lscs)\b",
        r"\b(?:c.section|cesarean|caesarean|lscs)\b.{0,50}\bemergency\b"
    ]
}

# Statements that suggest unavailability or uncertainty
negative_service_patterns = [
    r"\bnot\s+(?:available|offered|provided|performed)\b",
    r"\b(?:unavailable|discontinued|suspended)\b",
    r"\bno\s+(?:nicu|mri|dialysis|trauma|burn|cesarean|caesarean|c.section)\b",
    r"\bno longer\b"
]

# Supporting-context language that doesn't establish availability
weak_service_patterns = [
    r"\b(?:received|donated|support program|training program)\b",
    r"\b(?:planned|proposed|under construction|coming soon)\b"
]

def classify_service_evidence(claims, service):
    matches = find_evidence(claims, service)

    if not matches:
        return "No evidence", [], []

    strong = []
    moderate = []
    weak = []
    negative = []

    for claim in matches:
        text = claim.lower()

        if any(re.search(p, text) for p in negative_service_patterns):
            negative.append(claim)

        elif any(re.search(p, text) for p in weak_service_patterns):
            weak.append(claim)

        elif any(
            re.search(p, text)
            for p in strong_service_patterns.get(service, [])
        ):
            strong.append(claim)

        else:
            moderate.append(claim)

    if negative:
        return "Conflicting", strong + moderate, negative

    if strong:
        return "Strong", strong + moderate, weak

    if moderate:
        return "Moderate", moderate, weak

    return "Weak", weak, []


# Test against hospitals from our actual search results
test_hospitals = [
    ("Dhanwantari Hospital", "NICU"),
    ("Deenanath Mangeshkar Hospital and Research Center", "NICU"),
    ("Inamdar Multispecialty Hospital", "NICU"),
    ("Shaurya Hospital", "Trauma Care"),
    ("Nectar Hospital", "Dialysis")
]

for hospital, service in test_hospitals:
    rows = (
        df_ready
        .filter(F.col("name") == hospital)
        .select("name", "evidence_list")
        .collect()
    )

    for row in rows:
        level, evidence, warnings = classify_service_evidence(
            row["evidence_list"], service
        )

        print("\nHospital:", row["name"])
        print("Service:", service)
        print("Evidence level:", level)
        print("Supporting evidence:", evidence[:2])
        print("Warnings:", warnings[:2])



Hospital: Dhanwantari Hospital
Service: NICU
Evidence level: Strong
Supporting evidence: ['Combined NICU and PICU capacity of 12 cots', 'NICU and PICU units']
Warnings: []

Hospital: Deenanath Mangeshkar Hospital and Research Center
Service: NICU
Evidence level: Weak
Supporting evidence: ['Deenanath Mangeshkar Hospital has received neonatal equipment and staff training as part of PALAV Neonatal Intensive Care and Equipment Support program']
Warnings: []

Hospital: Inamdar Multispecialty Hospital
Service: NICU
Evidence level: Strong
Supporting evidence: ['Neonatal Intensive Care Unit (NICU) available', 'ICU, CCU, NICU, and PICU facilities']
Warnings: []

Hospital: Shaurya Hospital
Service: Trauma Care
Evidence level: Strong
Supporting evidence: ['Trauma surgery services', 'Trauma surgery for complex fractures and polytrauma']
Warnings: []

Hospital: Nectar Hospital
Service: Dialysis
Evidence level: Strong
Supporting evidence: ['Performs dialysis treatment', 'Provides dialysis treatment

In [0]:

import json
import re
from pyspark.sql import functions as F

def clean_urls(raw):
    try:
        urls = json.loads(raw or "[]")
    except (ValueError, TypeError):
        return []

    cleaned = []

    for url in urls:
        if not isinstance(url, str):
            continue

        # Convert Markdown-style links into regular URLs
        match = re.search(r"\]\((https?://[^)]+)\)", url)
        if match:
            url = match.group(1)

        if url.startswith(("http://", "https://")):
            if url not in cleaned:
                cleaned.append(url)

    return cleaned


def search_hospitals(service, city, top_n=5):
    if service not in service_terms:
        print("Unsupported service")
        return []

    if city not in city_locations:
        print("Unsupported city")
        return []

    user_lat, user_lon, state = city_locations[city]

    rows = (
        df_ready
        .filter(
            F.lower(F.trim(F.col("address_city"))) == city.lower()
        )
        .filter(
            F.lower(F.trim(F.col("address_stateOrRegion"))) == state.lower()
        )
        .filter(
            F.col("location_status")
            == "Within India bounds - not city verified"
        )
        .select(
            "unique_id", "name", "latitude", "longitude",
            "evidence_list", "source_urls", "officialWebsite",
            "phone_numbers", "capacity"
        )
        .collect()
    )

    results = []

    for row in rows:
        level, evidence, warnings = classify_service_evidence(
            row["evidence_list"], service
        )

        # Do not recommend facilities with insufficient evidence
        if level not in ["Strong", "Moderate"]:
            continue

        dist = distance_km(
            user_lat, user_lon,
            row["latitude"], row["longitude"]
        )

        # Provisional city-coordinate consistency check
        if dist > 60:
            continue

        # This score is for sorting matches, not a probability.
        evidence_points = 70 if level == "Strong" else 40
        distance_points = 30 * max(0, 1 - dist / 60)

        score = round(evidence_points + distance_points, 1)

        results.append({
            "id": row["unique_id"],
            "hospital": row["name"],
            "service": service,
            "city": city,
            "latitude": row["latitude"],
            "longitude": row["longitude"],
            "distance_km": round(dist, 1),
            "evidence_level": level,
            "ranking_score": score,
            "supporting_text": evidence[:3],
            "warnings": warnings[:3],
            "source_urls": clean_urls(row["source_urls"])[:5],
            "official_website": row["officialWebsite"],
            "phone_numbers": row["phone_numbers"],
            "reported_capacity": row["capacity"],
            "location_warning": (
                "Coordinates pass preliminary screening "
                "but have not been independently verified."
            )
        })

    results.sort(
        key=lambda x: (
            -x["ranking_score"],
            x["distance_km"],
            x["id"]
        )
    )

    print("Service:", service)
    print("Location:", city)
    print("Facilities screened:", len(rows))
    print("Eligible candidates:", len(results))

    for i, hospital in enumerate(results[:top_n], 1):
        print(f"\n{i}. {hospital['hospital']}")
        print("Distance:", hospital["distance_km"], "km")
        print("Evidence:", hospital["evidence_level"])
        print("Ranking score:", hospital["ranking_score"])
        print("Supporting text:", hospital["supporting_text"])

    return results[:top_n]


# Test the updated engine
nicu_results = search_hospitals("NICU", "Pune")


Service: NICU
Location: Pune
Facilities screened: 270
Eligible candidates: 10

1. Dhanwantari Hospital
Distance: 1.0 km
Evidence: Strong
Ranking score: 99.5
Supporting text: ['Combined NICU and PICU capacity of 12 cots', 'NICU and PICU units', 'NICU with 3 phototherapy machines (conventional and LED)']

2. Inamdar Multispecialty Hospital
Distance: 5.0 km
Evidence: Strong
Ranking score: 97.5
Supporting text: ['Neonatal Intensive Care Unit (NICU) available', 'ICU, CCU, NICU, and PICU facilities', 'Critical Care Services including ICU, CCU, NICU, and PICU']

3. Sahyadri Hospitals
Distance: 8.2 km
Evidence: Strong
Ranking score: 95.9
Supporting text: ['NICU on-site', 'Neonatal Intensive Care Unit (NICU) facilities']

4. Kohakade Hospital
Distance: 9.5 km
Evidence: Strong
Ranking score: 95.2
Supporting text: ['Has 35 beds with 7 ICU/NICU beds']

5. Vatsalya Mother & Child Care Hospital
Distance: 11.1 km
Evidence: Strong
Ranking score: 94.4
Supporting text: ['NICU with 20 beds', 'Dedicated N

In [0]:

# Find available catalogs and our current workspace context
print("Current catalog:", spark.sql("SELECT current_catalog()").first()[0])
print("Current schema:", spark.sql("SELECT current_schema()").first()[0])

display(spark.sql("SHOW CATALOGS"))


Current catalog: workspace
Current schema: default


catalog
databricks_virtue_foundation_dataset_dais_2026
samples
system
workspace


In [0]:

# Look at the schemas available in the workspace.
display(spark.sql("SHOW SCHEMAS IN workspace"))

# Check our current catalog and schema
print("Current catalog:", spark.sql("SELECT current_catalog()").first()[0])
print("Current schema:", spark.sql("SELECT current_schema()").first()[0])


databaseName
default
information_schema


Current catalog: workspace
Current schema: default


In [0]:

table_name = "workspace.default.care_finder_facilities"

# Don't overwrite a table if it already exists
if spark.catalog.tableExists(table_name):
    print("Table already exists:", table_name)
else:
    # Save our prepared hospital data as a Delta table
    app_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(table_name)

    print("Table created:", table_name)

# Verify the saved data
saved_df = spark.table(table_name)

print("Saved records:", saved_df.count())
display(saved_df.select(
    "name",
    "address_city",
    "location_status"
).limit(5))


Table already exists: workspace.default.care_finder_facilities
Saved records: 10088


name,address_city,location_status
Aravind Eye Hospital,Hyderabad,Within India bounds - not city verified
"Fortis Hospital, Gurugram",Gurgaon,Within India bounds - not city verified
Fortis Hospital Anandapur,Kolkata,Within India bounds - not city verified
Wockhardt Hospital Nagpur,Nagpur,Within India bounds - not city verified
"RAM HOSPITAL & RESEARCH CENTRE, KANPUR",Kanpur,Within India bounds - not city verified


In [0]:

from pyspark.sql import functions as F

# Select hospital records in Lucknow
lucknow = df_ready.filter(
    F.lower(F.trim(F.col("address_city"))) == "lucknow"
)

# Search all evidence statements
terms = (
    r"(?i)caesarean|cesarean|c.section|"
    r"\blscs\b|obstetric|maternity"
)

matches = lucknow.filter(
    F.exists(
        F.col("evidence_list"),
        lambda x: x.rlike(terms)
    )
)

print("Lucknow facility records:", lucknow.count())
print("Records mentioning obstetric or C-section terms:", matches.count())

display(
    matches.select(
        "name",
        "address_city",
        "evidence_list"
    ).limit(10)
)


Lucknow facility records: 155
Records mentioning obstetric or C-section terms: 43


name address_city evidence_list Fatima Hospital Lucknow List(Fatima Hospital is located in Lucknow, Uttar Pradesh., Has 30 years in healthcare, Has 35 doctors available, Accepts Ayushman Card government health insurance, Accepts Future Generali India Insurance, Fatima Hospital is located in Noida, Uttar Pradesh, India, Fatima Hospital employs Dr. Mrityunjay Pandey as Sr. Consultant in Pediatrics, Fatima Hospital has gynecology and obstetrics services with Dr. Shuchi Gupta (Gynecologist and Obstetrician with MS in Obstetrics and Gynecology), Accepts cash payments, Fatima Hospital is located in Lucknow, India, Fatima Hospital employs a Senior Consultant Obstetrician & Gynecologist (the Revive Clinic doctor described in the content works at Fatima Hospital), Fatima Hospital is a hospital located in Lucknow, India, Fatima Hospital is listed under Hospitals in Lucknow on Joonsquare, Opened Fatima Hospital in 1995, Located in Gorakhpur, Uttar Pradesh, India, Has 25 inpatient beds, 60 additional beds under construction, Operated by the Diocese of Gorakhpur, Has 39 health care providers, Seven full-time doctors, Three on-call doctors, Five nursing sisters, Twelve dispensary attendants, A cardiac physician visits Fatima Hospital three times per year from Delhi, Hospital director is Father Jose Manjiyil, C.S.T, Fatima Hospital is a Future Generali cashless network hospital in Gorakhpur., Has 340 beds., Part of the Diocese of Varanasi Hospital and Health Centres network., Administrator is Sr. Lily, MSJ., Medical Superintendent is Dr. (Sr.) Jude, MSJ., Fatima Hospital was established in 1958 in Lucknow by Bishop Conrard De Vito., The first missionary doctor associated with Fatima Hospital was Dr. Raffaela Scolletta, a specialist in Obstetrics & Gynecology., Medical Sisters of St. Joseph from Kerala were invited to serve the people of Lucknow through Fatima Hospital., Fatima Clinic was established in 1958 and Fatima Hospital & Nursing College Lucknow houses a nursing college., Specialty departments include Nephrology, Pulmonary Medicine, and Obstetrics & Gynecology., Fatima Nursing College offers GNM and BSc Nursing programs; a DNB programme is offered., The hospital lists departments including Medicine, Surgery, Obstetrics & Gynecology, Pediatrics, Pead. Surgery (Pediatric Surgery), Anestheist, Orthopedic, ENT, Eye, Neuro, Radiology, Physiotherapy, Skin, Chest Specialist, Dental, Urology, Pathology, Ayurveda, Emergency, Psychiatry, and DNB., Located in Lucknow., Multi-specialty hospital, Offers dental care and surgical services, Has 4 doctors available, Fatima Hospital has a General Physician on staff, Dr. Gaurav Jain is a General Physician associated with Fatima Hospital, Dr. Gaurav Jain holds MBBS, MD (Medicine), and Diploma (Cardiology), Opens at 09:00 AM, Has 30 years in healthcare, Eye hospital located in Belthara, Ballia (region), Parent hospital: Fatima Hospital, Affiliated hospitals include Nur Manzil, St. Joseph's Hospital Gomti Nagar, St. Joseph's Community Health Centre Mehmoodabad (Sitapur District) Rural, and Shanti Niketan Community Health Centre Dasauli Urban, Fatima Clinic was established in 1958 by Bishop Conrard De Vito (then Bishop of Lucknow) with Dr. Raffaela Scolletta, a specialist in Obstetrics & Gynecology, as the first missionary doctor, Fatima Hospital - no procedures listed in the provided content., , No explicit medical procedures listed for Fatima Hospital in the provided content, Performs cesarean births, Provides normal births, Artificial Teeth Implant, , , Offers in vitro fertilization (IVF) services, Provides dialysis treatment, Offers maternity and obstetric care, Provides emergency medicine services, Offers plastic surgery and cosmetic surgery, Provides ophthalmology (eye) care, Offers orthopedics and neurosurgery services, Provides cardiology and cardiothoracic surgery services, Provides general surgery and gynecology procedures, , Dermatology consultations provided by Dr. Naveen Kumar at Fatima Hospita

In [0]:

import re
from pyspark.sql import functions as F

# Lucknow facilities
lucknow_rows = (
    df_ready
    .filter(
        F.lower(F.trim(F.col("address_city"))) == "lucknow"
    )
    .select("unique_id", "name", "evidence_list")
    .collect()
)

csection_pattern = (
    r"\b(?:c[\s-]?section|cesarean|caesarean|lscs)\b"
)

emergency_pattern = (
    r"\b(?:emergency care|emergency department|"
    r"emergency services?|24/7 emergency|"
    r"24x7 emergency)\b"
)

direct_pattern = (
    r"\bemergency.{0,50}(?:c[\s-]?section|"
    r"cesarean|caesarean|lscs)\b|"
    r"\b(?:c[\s-]?section|cesarean|caesarean|"
    r"lscs).{0,50}emergency\b"
)

summary = {
    "Direct evidence": 0,
    "Combined evidence": 0,
    "C-section only": 0,
    "Emergency only": 0,
    "Neither": 0
}

combined_examples = []

for row in lucknow_rows:
    claims = row["evidence_list"] or []

    csection = [
        x for x in claims
        if x and re.search(csection_pattern, x, re.I)
    ]

    emergency = [
        x for x in claims
        if x and re.search(emergency_pattern, x, re.I)
    ]

    direct = [
        x for x in claims
        if x and re.search(direct_pattern, x, re.I)
    ]

    if direct:
        category = "Direct evidence"
    elif csection and emergency:
        category = "Combined evidence"
    elif csection:
        category = "C-section only"
    elif emergency:
        category = "Emergency only"
    else:
        category = "Neither"

    summary[category] += 1

    if category == "Combined evidence":
        combined_examples.append({
            "hospital": row["name"],
            "csection": csection[0],
            "emergency": emergency[0]
        })

print("Lucknow records analyzed:", len(lucknow_rows))

for category, count in summary.items():
    print(category + ":", count)

print("\nExamples requiring verification:")

for item in combined_examples[:5]:
    print("\nHospital:", item["hospital"])
    print("C-section:", item["csection"])
    print("Emergency:", item["emergency"])


Lucknow records analyzed: 155
Direct evidence: 0
Combined evidence: 9
C-section only: 2
Emergency only: 35
Neither: 109

Examples requiring verification:

Hospital: Nova Hospital
C-section: Performs cesarean sections (LSCS) as part of obstetric care
Emergency: Nova Hospital provides 24/7 emergency services

Hospital: City Hospital
C-section: Obstetric delivery including normal and cesarean deliveries
Emergency: Provides accident and emergency department services

Hospital: Maa Hospital
C-section: Normal Delivery and Cesarean Section performed at Max Super Speciality Hospital, Noida
Emergency: Offers 24/7 emergency care

Hospital: Krishna Medical Centre
C-section: Obstetrics and gynaecology services including ANC, normal delivery, C-section and high-risk pregnancy care
Emergency: 24/7 emergency care

Hospital: St.Joseph Hospital Lucknow
C-section: Caesarean Section
Emergency: 24x7 emergency services


---

## Phase 1 — Service evidence audit

Review service claims and test the first rule-based approach.

*Source notebook: `care_finder_phase1_evidence_audit (1).ipynb`*

**Before running this section:** It was tested separately. Restart the Python session or check existing variables if you are rerunning the analysis after an earlier stage.

# Care Finder India — Evidence Audit (Phase 1)

This notebook tests a **conservative rule baseline** against real Virtue Foundation records. It does not claim to be ML/NLI or independent verification. Do not deploy a new ranking before manually reviewing errors.

In [0]:
import json
import re
from collections import Counter
from pyspark.sql import functions as F

TABLE = "workspace.default.care_finder_facilities"  # Change only if your Unity Catalog table differs
df = spark.table(TABLE)
print(f"Facility records: {df.count():,}")
print("Columns:", df.columns)
df.printSchema()

Facility records: 10,088
Columns: ['unique_id', 'name', 'address_line1', 'address_city', 'address_stateOrRegion', 'address_zipOrPostcode', 'latitude', 'longitude', 'description', 'specialties_list', 'capability_list', 'procedure_list', 'equipment_list', 'evidence_list', 'source_urls_list', 'officialWebsite', 'phone_numbers', 'capacity', 'location_status', 'repeated_name']
root
 |-- unique_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- address_line1: string (nullable = true)
 |-- address_city: string (nullable = true)
 |-- address_stateOrRegion: string (nullable = true)
 |-- address_zipOrPostcode: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- description: string (nullable = true)
 |-- specialties_list: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- capability_list: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- procedure_list: array (nullabl

In [0]:
required = {"unique_id", "name", "address_city", "address_stateOrRegion", "latitude", "longitude", "evidence_list"}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"The app source expects fields that are missing: {missing}")
print("Original dataset fields available beyond current app fields:")
app_fields = required | {"address_line1", "source_urls_list", "officialWebsite", "repeated_name"}
print(sorted(set(df.columns) - app_fields))

# See how much source text and location information is present.
counts = df.select([F.count(F.when(F.col(c).isNotNull(), 1)).alias(c) for c in required])
display(counts)

Original dataset fields available beyond current app fields:
['address_zipOrPostcode', 'capability_list', 'capacity', 'description', 'equipment_list', 'location_status', 'phone_numbers', 'procedure_list', 'specialties_list']


unique_id,address_city,latitude,name,address_stateOrRegion,longitude,evidence_list
10088,10030,9970,10034,10030,9970,10088


## Rule-based claim baseline

Run this cell to define the tested evidence classifier locally in the notebook.

In [0]:
"""Care Finder India evidence engine, phase 1.

Conservative, testable claim classification for the existing Virtue Foundation
facility records. This is a rule baseline, NOT an NLP model or verification.

Next phase: benchmark a semantic retrieval/NLI model against labeled claims.
"""
from __future__ import annotations

import re
from dataclasses import dataclass, asdict
from typing import Iterable

# Deliberately keep specific terms separate from broader related concepts.
SERVICE_TERMS = {
    "Dialysis": [r"\bdialysis\b", r"\bha?emodialysis\b", r"\bperitoneal dialysis\b"],
    "NICU": [r"\bnicu\b", r"\bneonatal intensive care\b"],
    "MRI": [r"\bmri\b", r"\bmagnetic resonance imaging\b"],
    "Trauma Care": [r"\btrauma (?:care|cent(?:er|re)|unit|surgery|services?)\b", r"\bpolytrauma\b"],
    "Burn Care": [r"\bburns? (?:care|unit|ward|treatment|cent(?:er|re))\b"],
    "Emergency Department": [r"\bemergency (?:department|room|care|unit|services?|medicine)\b", r"\bcasualty department\b"],
    "Emergency C-Section": [r"\b(?:emergency\s+(?:c[ -]?section|caesarean|cesarean)|(?:c[ -]?section|caesarean|cesarean).{0,40}emergency)\b"],
}

# These verbs indicate a claimed CURRENT facility offering when used near
# a relevant service expression. They do not prove the claim is true.
CURRENT_CLAIM = re.compile(
    r"\b(?:provide(?:s|d)?|offer(?:s|ed)?|perform(?:s|ed)?|conduct(?:s|ed)?|available|operational|"
    r"currently|equipped|dedicated|established|functional|in.house|"
    r"running|service(?:s)?|unit|department|scanner|machine)\b", re.I
)
PLANNED = re.compile(
    r"\b(?:plans? to|planned|proposed|coming soon|will (?:open|start|offer|provide)|"
    r"under construction|expected to|to be launched|future|soon to open)\b", re.I
)
UNAVAILABLE = re.compile(
    r"\b(?:not available|unavailable|no longer|does not (?:offer|provide|have|perform)|"
    r"do not (?:offer|provide|have|perform)|not (?:offered|provided|operational)|"
    r"discontinued|suspended|closed down|without)\b", re.I
)
GENERIC = re.compile(
    r"\b(?:specialties include|departments? include|services? include|"
    r"list of specialties|listed specialties|research|training|"
    r"studies|information about)\b", re.I
)

@dataclass(frozen=True)
class ClaimResult:
    service: str
    claim: str
    label: str  # direct | supporting | weak | planned | contradictory | unrelated
    reason: str
    method: str = "conservative_rules_v1"


def _clauses(text: str) -> list[str]:
    """Split only on firm sentence boundaries; keep the original claim visible."""
    return [part.strip() for part in re.split(r"[.;\n]+", text) if part.strip()]


def classify_claim(service: str, claim: str) -> ClaimResult:
    """Classify a recorded statement, not the real operational status."""
    if service not in SERVICE_TERMS:
        raise ValueError(f"Unsupported service: {service}")
    text = str(claim or "").strip()
    if not text:
        return ClaimResult(service, text, "unrelated", "Empty statement")

    pattern = re.compile("|".join(f"(?:{term})" for term in SERVICE_TERMS[service]), re.I)
    relevant = [part for part in _clauses(text) if pattern.search(part)]
    if not relevant:
        return ClaimResult(service, text, "unrelated", "No requested-service wording")

    # A long statement can contain both positive and negative sentences.
    # Treat that as contradictory rather than silently selecting one.
    labels = set()
    for part in relevant:
        if UNAVAILABLE.search(part) or re.search(r"\bno\s+(?:\w+\s+){0,3}(?:" + pattern.pattern + r")", part, re.I):
            labels.add("contradictory")
        elif PLANNED.search(part):
            labels.add("planned")
        elif GENERIC.search(part):
            labels.add("weak")
        elif CURRENT_CLAIM.search(part):
            labels.add("direct")
        else:
            labels.add("supporting")

    if "contradictory" in labels:
        return ClaimResult(service, text, "contradictory", "Statement indicates unavailability or conflicting status")
    if "planned" in labels:
        return ClaimResult(service, text, "planned", "Statement refers to a planned or future service")
    if "direct" in labels:
        return ClaimResult(service, text, "direct", "Statement explicitly claims a current service or unit")
    if "supporting" in labels:
        return ClaimResult(service, text, "supporting", "Relevant wording without a clear current-service assertion")
    return ClaimResult(service, text, "weak", "General listing or background mention")


def summarize_service(service: str, claims: Iterable[str]) -> dict:
    """Return auditable status from ALL statements for a facility and service.

    Negative and positive evidence coexisting is flagged; a model should not
    use only the most favorable statement.
    """
    records = [classify_claim(service, statement) for statement in claims]
    relevant = [result for result in records if result.label != "unrelated"]
    labels = {result.label for result in relevant}
    positive = any(label in labels for label in {"direct", "supporting"})
    negative = "contradictory" in labels
    if negative and positive:
        status = "conflicting_claims"
    elif negative:
        status = "unavailable_claim"
    elif "direct" in labels:
        status = "direct_claim"
    elif "supporting" in labels:
        status = "supporting_only"
    elif "planned" in labels:
        status = "future_only"
    elif "weak" in labels:
        status = "weak_only"
    else:
        status = "no_evidence"
    return {
        "service": service,
        "status": status,
        "claims": [asdict(result) for result in relevant],
        "should_recommend": status == "direct_claim",  # Conservative default
        "disclaimer": "Dataset claims only; current availability not independently verified.",
    }


def patient_explanation(summary: dict, distance_km: float, location_concern: bool = False) -> str:
    """Patient-facing wording; no unsupported claims of verification."""
    status = summary["status"]
    if status == "direct_claim":
        intro = "The hospital record says this service is offered."
    elif status == "supporting_only":
        intro = "The hospital record mentions this service, but does not clearly confirm it is offered."
    elif status == "conflicting_claims":
        intro = "The record contains conflicting information about this service."
    else:
        intro = "The available record does not establish that this service is currently offered."
    message = f"{intro} The recorded location is {distance_km:.1f} km away in a straight line."
    if location_concern:
        message += " The facility's location details may be inconsistent; confirm its location before traveling."
    message += " Contact the hospital to confirm current availability and patient acceptance."
    return message


In [0]:
# Review a small sample first; synthetic examples are only for checking the code.
rows = (df.select("unique_id", "name", "address_city", "evidence_list")
        .where(F.col("evidence_list").isNotNull())
        .orderBy(F.rand(seed=615))
        .limit(150)
        .collect())

def parse_claims(raw):
    if isinstance(raw, (list, tuple)):
        return [str(x) for x in raw if x is not None]
    if isinstance(raw, str):
        try:
            value = json.loads(raw)
            return [str(x) for x in value] if isinstance(value, list) else []
        except (TypeError, ValueError):
            return []
    return []

out = []
for row in rows:
    claims = parse_claims(row["evidence_list"])
    for service in SERVICE_TERMS:
        result = summarize_service(service, claims)
        if result["status"] != "no_evidence":
            out.append({
                "facility_id": str(row["unique_id"]),
                "facility": row["name"],
                "city": row["address_city"],
                "service": service,
                "status": result["status"],
                "supporting_claims": [x["claim"] for x in result["claims"]][:3],
            })
print("Matches for manual review:", len(out))
print("Status distribution:", Counter(x["status"] for x in out))
if out:
    display(spark.createDataFrame([
        {**x, "supporting_claims": " | ".join(x["supporting_claims"])}
        for x in out
    ]))

Matches for manual review: 101
Status distribution: Counter({'direct_claim': 68, 'supporting_only': 31, 'weak_only': 2})


city,facility,facility_id,service,status,supporting_claims
"Andheri West, Mumbai","Smile Children Clinic, Andheri, Mumbai | Best Children Specialist Doctor | Pediatrician I Vaccination",e870fdd8-0b58-4ba5-ad7d-1105df50eb2f,MRI,supporting_only,"Brain MRI is included as part of the comprehensive health assessment | Comprehensive child health assessment including blood tests, genetic tests, eye check-up, hearing assessment, and brain MRI"
"Andheri West, Mumbai","Smile Children Clinic, Andheri, Mumbai | Best Children Specialist Doctor | Pediatrician I Vaccination",e870fdd8-0b58-4ba5-ad7d-1105df50eb2f,Emergency Department,direct_claim,Provides pediatric emergency care
Rajkot,Premdhara Hospital,081eb377-1666-43c5-b54d-10a7a12c6978,Emergency Department,direct_claim,On-site ambulance for emergency services
New Delhi,City X-ray and Scan Clinic -Tilak Nagar,319833de-7a02-4fca-b143-3fd0122c2e7d,MRI,direct_claim,MRI | MRI | Performs MRI scans
Ludhiana,"Global Heart And Super Speciality Hospital, Lalbagh",ae2c85db-e120-4203-85ad-49309fa673d8,Dialysis,direct_claim,"Offers cardiology, cardiac surgery, nephrology including dialysis, neurology, neurosurgery, orthopedics including knee replacement, endovascular surgery, cosmetic and plastic surgery, neuro and spine surgery, non-invasive cardiology, radiology, gynecology and obstetrics, general medicine, and laparoscopic surgery | Nephrology including dialysis | Dialysis"
Ludhiana,"Global Heart And Super Speciality Hospital, Lalbagh",ae2c85db-e120-4203-85ad-49309fa673d8,MRI,supporting_only,"Advanced neurological diagnostic technologies including EEG, EMG, MRI, CT, and PET | EEG, EMG, MRI, CT scan, and PET scan capabilities"
Ludhiana,"Global Heart And Super Speciality Hospital, Lalbagh",ae2c85db-e120-4203-85ad-49309fa673d8,Trauma Care,supporting_only,24/7 emergency and trauma care | Round-the-clock emergency and trauma care | Trauma care
Ludhiana,"Global Heart And Super Speciality Hospital, Lalbagh",ae2c85db-e120-4203-85ad-49309fa673d8,Emergency Department,direct_claim,Around-the-clock emergency services | 24/7 emergency care
Bhopal,"Smile Gallery Dental Wellness Center Saurabh Nursing Home, 82 Marwari Road, Near Azad Market, Bhopal, M.P. 462001",a8c51d98-a31b-4234-acf6-0a2d4cf67cac,Emergency Department,direct_claim,"Provides a broad range of dental treatments including restorative, preventive, cosmetic, pediatric, and emergency care"
Muzaffarpur,Apex Children's Hospital,a7cb46eb-417e-4c10-aab7-119b76fc0507,NICU,direct_claim,"NICU program exists at Apex Children's Hospital | NICU provides round-the-clock care for premature and ill newborn infants | Public NICU staffing vacancy posted on May 29, 2024"


## Manual validation protocol

Review at least 50–100 **service claims**, not just facility records. Record the correct label (direct, supporting, weak, planned, contradictory, unrelated), record identity problems separately, and compare with the automatic label. Include challenging negatives and different services. Only after reviewing this sample should we run semantic search/NLI and compare precision/recall.

Next: inspect the newly revealed original columns, decide which are reliable source fields, and create an enriched table for the Streamlit app.

In [0]:

from pyspark.sql import functions as F

table_name = "workspace.default.care_finder_facilities"

df = spark.table(table_name)

# Facilities with potentially mismatched service claims
names = [
    "Aravind Eye Hospital",
    "Partha Dental Kothapet",
    "Smile Gallery Dental Wellness Center"
]

# Find records by facility name
pattern = "|".join(names)

suspicious = df.filter(
    F.col("name").rlike("(?i)" + pattern)
)

# Inspect the original dataset fields
columns = [
    "name",
    "address_city",
    "address_stateOrRegion",
    "location_status",
    "repeated_name",
    "description",
    "specialties_list",
    "capability_list",
    "procedure_list",
    "equipment_list",
    "evidence_list",
    "source_urls_list"
]

for row in suspicious.select(*columns).collect():
    print("\n" + "=" * 80)
    print("FACILITY:", row["name"])
    print("=" * 80)

    for col in columns[1:]:
        print(f"\n{col}:")
        print(row[col])



FACILITY: Aravind Eye Hospital

address_city:
Hyderabad

address_stateOrRegion:
Telangana

location_status:
Within India bounds - not city verified

repeated_name:
False

description:
Referral hospital outside the state listed under Assam Government schemes (AAN/Atal Amrit Abhiyan).

specialties_list:
['ophthalmology', 'internalMedicine', 'internalMedicine', 'internalMedicine', 'internalMedicine', 'internalMedicine', 'internalMedicine', 'internalMedicine', 'internalMedicine', 'internalMedicine', 'internalMedicine', 'internalMedicine', 'internalMedicine', 'anesthesia', 'burnAndTraumaPlasticSurgery', 'otolaryngology', 'gynecologyAndObstetrics', 'neonatologyPerinatalMedicine', 'internalMedicine', 'neurosurgery', 'orthopedicSurgery', 'pediatrics', 'pediatricSurgery', 'physicalMedicineAndRehabilitation', 'pulmonology', 'dermatology', 'generalSurgery', 'radiology', 'anesthesia', 'burnAndTraumaPlasticSurgery', 'otolaryngology', 'gynecologyAndObstetrics', 'neonatologyPerinatalMedicine', 'fami

---

## Phase 2 — Record quality

Look at repeated records, mixed sources, and possible facility attribution problems.

*Source notebook: `care_finder_phase2_record_quality (1).ipynb`*

**Before running this section:** It was tested separately. Restart the Python session or check existing variables if you are rerunning the analysis after an earlier stage.

# Care Finder India — Phase 2: Record-quality audit

Run separately from your existing cleaning notebook. This flags possible mixed hospital evidence, other-branch statements, and specialist-only emergencies. **Review flags are not verification.** The original data and Streamlit app remain unchanged.

In [0]:
import re
from urllib.parse import urlparse
from collections import Counter
from pyspark.sql import functions as F, types as T

SOURCE_TABLE = "workspace.default.care_finder_facilities"
OUTPUT_TABLE = "workspace.default.care_finder_record_quality_v2"
WRITE_RESULTS = False  # Inspect the audit first. Set True only after review.

In [0]:
# This is an initial check for possible record issues, not a verified identity match.
# Some fields combine details from several websites. The URL list does
# not tell us which website supports each claim, so these are just review flags.

GENERAL_EMERGENCY = re.compile(
    r"\b(?:emergency (?:department|room|medicine|services?|care)|"
    r"accident\s*(?:&|and)\s*emergency|casualty (?:department|unit))\b", re.I
)
SPECIALTY_EMERGENCY = re.compile(
    r"\b(?:dental|dentistry|eye|ocular|ophthalmic|pediatric|paediatric|"
    r"psychiatric|veterinary|skin)\b.{0,65}\bemergency\b|"
    r"\bemergency\b.{0,65}\b(?:dental|dentistry|eye|ocular|ophthalmic|"
    r"pediatric|paediatric|psychiatric|veterinary|skin)\b", re.I
)
NAMED_HOSPITAL = re.compile(
    r"\b([A-Z][A-Za-z&.'-]*(?:\s+[A-Z][A-Za-z&.'-]*){0,5}\s+"
    r"(?:Hospital|Hospitals|Medical Centre|Medical Center))\b"
)
BRANCH_AT = re.compile(
    r"\b(?:at|in|located at|located in)\s+"
    r"([A-Za-z][A-Za-z -]{2,32}?)\s+(?:clinic|branch)\b", re.I
)
CITY_TO_STATE = {
    "delhi": "delhi", "new delhi": "delhi", "mumbai": "maharashtra",
    "chennai": "tamil nadu", "kolkata": "west bengal",
    "hyderabad": "telangana", "patna": "bihar", "jaipur": "rajasthan",
    "lucknow": "uttar pradesh", "pune": "maharashtra",
    "bhopal": "madhya pradesh", "bangalore": "karnataka",
    "bengaluru": "karnataka",
}
# Domains of institution-specific government sources in the current sample.
# This is a REVIEW signal, not automatic proof the record is wrong.
FOREIGN_HOSPITAL_DOMAIN_STATES = {
    "igh.delhi.gov.in": "delhi", "lnjp.delhi.gov.in": "delhi",
    "gtbh.delhi.gov.in": "delhi", "dduh.delhi.gov.in": "delhi",
    "rgssh.delhi.gov.in": "delhi", "ggsgh.delhi.gov.in": "delhi",
    "lbsh.delhi.gov.in": "delhi", "gbpant.delhi.gov.in": "delhi",
}

def normalize(s):
    return re.sub(r"[^a-z0-9]+", " ", str(s or "").lower()).strip()

def tokens(s):
    stop = {"hospital", "hospitals", "medical", "centre", "center", "clinic",
            "health", "care", "the", "of", "and", "private", "limited", "pvt", "ltd"}
    return {w for w in normalize(s).split() if len(w) > 2 and w not in stop}

def arr(row, col):
    value = row[col] if col in row else None
    return [str(s).strip() for s in (value or []) if s is not None and str(s).strip()]

def analyze(row):
    name = str(row["name"] or "")
    city = normalize(row["address_city"])
    state = normalize(row["address_stateOrRegion"])
    address = normalize(row["address_line1"])
    own = tokens(name)
    evidence = list(dict.fromkeys(arr(row, "evidence_list")))
    # The original lists are often duplicated in evidence_list. Do not treat
    # repeated fields as independent sources or award points for their count.
    urls = list(dict.fromkeys(arr(row, "source_urls_list")))

    other_hospitals = set()
    foreign_branch = []
    specialty_only = []
    explicit_location = []

    for claim in evidence:
        for match in NAMED_HOSPITAL.finditer(claim):
            mentioned = match.group(1).strip()
            t = tokens(mentioned)
            if own and t and len(t & own) == 0:
                other_hospitals.add(mentioned)

        match = BRANCH_AT.search(claim)
        if match:
            branch = normalize(match.group(1))
            # Only a possible branch conflict when the branch label doesn't
            # occur in this facility's own name/address.
            if (branch and branch not in normalize(name)
                    and branch not in address and branch != city):
                foreign_branch.append(claim)

        if SPECIALTY_EMERGENCY.search(claim):
            specialty_only.append(claim)

        for named_city, named_state in CITY_TO_STATE.items():
            if named_state != state and re.search(
                r"\b(?:located|situated|based)\s+in\s+" + re.escape(named_city) + r"\b",
                claim, re.I
            ):
                explicit_location.append(claim)
                break

    foreign_domains = set()
    for u in urls:
        try:
            host = urlparse(u).hostname or ""
        except ValueError:
            continue
        if host in FOREIGN_HOSPITAL_DOMAIN_STATES and (
            FOREIGN_HOSPITAL_DOMAIN_STATES[host] != state
        ):
            foreign_domains.add(host)

    flags = []
    if len(other_hospitals) >= 2:
        flags.append("multiple_other_hospitals_mentioned")
    elif other_hospitals:
        flags.append("other_hospital_mentioned")
    if foreign_branch:
        flags.append("other_branch_mentioned")
    if specialty_only:
        flags.append("specialty_emergency_mentions")
    if explicit_location:
        flags.append("explicit_other_state_location")
    if len(foreign_domains) >= 2:
        flags.append("multiple_foreign_hospital_domains")
    if bool(row["repeated_name"]):
        flags.append("repeated_facility_name")
    location_status = str(row["location_status"] or "")
    if "not city verified" in location_status.lower():
        flags.append("coordinates_not_city_verified")

    # Only strong MULTIPLE independent-looking indicators cause hold-for-review.
    # This does not claim the hospital is false, or that source URLs map to claims.
    hold = (len(other_hospitals) >= 2 and len(foreign_domains) >= 2)
    status = "hold_for_review" if hold else (
        "needs_review" if any(f != "coordinates_not_city_verified" for f in flags)
        else "no_specific_conflict_detected"
    )
    return {
        "unique_id": str(row["unique_id"] or ""),
        "name": name,
        "city": str(row["address_city"] or ""),
        "status": status,
        "hold_for_review": hold,
        "flags": flags,
        "other_hospitals": sorted(other_hospitals)[:10],
        "foreign_domains": sorted(foreign_domains)[:10],
        "branch_examples": foreign_branch[:3],
        "specialty_emergency_examples": specialty_only[:3],
        "location_examples": explicit_location[:3],
        "claim_count": len(evidence),
    }

In [0]:
# Try a few examples before scanning the full table.

def demo_row(name, city="Hyderabad", state="Telangana", claims=None, urls=None):
    return dict(unique_id="test", name=name, address_city=city,
                address_stateOrRegion=state, address_line1="Kothapet",
                evidence_list=claims or [], source_urls_list=urls or [],
                repeated_name=False, location_status="Within India bounds - not city verified")

r1 = analyze(demo_row("Partha Dental Kothapet", claims=[
    "Provides dental emergency services at Attapur clinic"
]))
assert "other_branch_mentioned" in r1["flags"]
assert "specialty_emergency_mentions" in r1["flags"]
assert not r1["hold_for_review"]

r2 = analyze(demo_row("Aravind Eye Hospital", claims=[
    "Janakpuri Super Speciality Hospital provides care",
    "Indira Gandhi Hospital provides imaging",
    "Lok Nayak Hospital provides services",
], urls=["https://igh.delhi.gov.in/igh/services-and-facilities",
         "https://lnjp.delhi.gov.in/"]))
assert r2["hold_for_review"]

r3 = analyze(demo_row("Example Hospital", claims=["Offers dialysis treatment"]))
assert r3["status"] == "no_specific_conflict_detected"
print("Passed 3 record-quality smoke tests")

Passed 3 record-quality smoke tests


In [0]:
# Check the Unity Catalog records without changing them.
df = spark.table(SOURCE_TABLE)
needed = ["unique_id", "name", "address_city", "address_stateOrRegion",
          "address_line1", "evidence_list", "source_urls_list",
          "location_status", "repeated_name"]
assert not (set(needed) - set(df.columns)), "Missing expected table columns"

results = []
for row in df.select(*needed).toLocalIterator():
    results.append(analyze(row.asDict(recursive=True)))

print("Records audited:", len(results))
print("Review status:", dict(Counter(r["status"] for r in results)))
print("Top flags:", Counter(flag for r in results for flag in r["flags"]).most_common(12))

Records audited: 10088
Review status: {'hold_for_review': 1, 'no_specific_conflict_detected': 6300, 'needs_review': 3787}
Top flags: [('coordinates_not_city_verified', 9964), ('multiple_other_hospitals_mentioned', 1483), ('other_hospital_mentioned', 1314), ('repeated_facility_name', 823), ('specialty_emergency_mentions', 472), ('other_branch_mentioned', 405), ('explicit_other_state_location', 93), ('multiple_foreign_hospital_domains', 1)]


In [0]:
# Review examples and problematic facilities. No changes to live app rankings.
schema = T.StructType([
    T.StructField("unique_id", T.StringType()),
    T.StructField("name", T.StringType()),
    T.StructField("city", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("hold_for_review", T.BooleanType()),
    T.StructField("flags", T.ArrayType(T.StringType())),
    T.StructField("other_hospitals", T.ArrayType(T.StringType())),
    T.StructField("foreign_domains", T.ArrayType(T.StringType())),
    T.StructField("branch_examples", T.ArrayType(T.StringType())),
    T.StructField("specialty_emergency_examples", T.ArrayType(T.StringType())),
    T.StructField("location_examples", T.ArrayType(T.StringType())),
    T.StructField("claim_count", T.IntegerType()),
])
audit_df = spark.createDataFrame(results, schema=schema)
display(audit_df.groupBy("status").count().orderBy("status"))
display(audit_df.filter(F.col("name").rlike("(?i)Aravind Eye Hospital|Partha Dental Kothapet|Smile Gallery"))
        .select("name", "city", "status", "flags", "other_hospitals", "foreign_domains", "branch_examples"))
display(audit_df.filter(F.col("status") == "hold_for_review")
        .select("name", "city", "flags", "other_hospitals", "foreign_domains").limit(30))

status,count
hold_for_review,1
needs_review,3787
no_specific_conflict_detected,6300


name,city,status,flags,other_hospitals,foreign_domains,branch_examples
Aravind Eye Hospital,Hyderabad,hold_for_review,"List(multiple_other_hospitals_mentioned, multiple_foreign_hospital_domains, coordinates_not_city_verified)","List(Delhi Government Hospital, Delhi Government Hospitals, Janakpuri Super Speciality Hospital, Lok Nayak Hospital, Society Hospitals)","List(dduh.delhi.gov.in, gbpant.delhi.gov.in, ggsgh.delhi.gov.in, gtbh.delhi.gov.in, igh.delhi.gov.in, lbsh.delhi.gov.in, lnjp.delhi.gov.in, rgssh.delhi.gov.in)",List()
Partha Dental Kothapet,Hyderabad,needs_review,"List(other_branch_mentioned, specialty_emergency_mentions, coordinates_not_city_verified)",List(),List(),List(Provides dental emergency services at Attapur clinic)
Priyamvada Birla Aravind Eye Hospital,Kolkata,needs_review,"List(multiple_other_hospitals_mentioned, specialty_emergency_mentions, coordinates_not_city_verified)","List(Star Health Network Hospital, Star Health Network Hospitals, Universal Sompo Network Hospital, Universal Sompo Network Hospitals)",List(),List()
"Smile Gallery Dental Wellness Center Saurabh Nursing Home, 82 Marwari Road, Near Azad Market, Bhopal, M.P. 462001",Bhopal,needs_review,"List(other_hospital_mentioned, specialty_emergency_mentions, coordinates_not_city_verified)",List(Jawaharlal Nehru Cancer Hospital),List(),List()


name,city,flags,other_hospitals,foreign_domains
Aravind Eye Hospital,Hyderabad,"List(multiple_other_hospitals_mentioned, multiple_foreign_hospital_domains, coordinates_not_city_verified)","List(Delhi Government Hospital, Delhi Government Hospitals, Janakpuri Super Speciality Hospital, Lok Nayak Hospital, Society Hospitals)","List(dduh.delhi.gov.in, gbpant.delhi.gov.in, ggsgh.delhi.gov.in, gtbh.delhi.gov.in, igh.delhi.gov.in, lbsh.delhi.gov.in, lnjp.delhi.gov.in, rgssh.delhi.gov.in)"


In [0]:
# Change WRITE_RESULTS only if we decide to save the audit table.
# The output is a separate table, so the source stays untouched.
if WRITE_RESULTS:
    audit_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)
    print("Audit saved to:", OUTPUT_TABLE)
else:
    print("Preview only; original data and app are unchanged.")

Preview only; original data and app are unchanged.


In [0]:

from pyspark.sql import functions as F

# Sample records with other-hospital and branch flags.
# We want to understand which are genuine problems
# versus harmless references.

flags_to_review = [
    "multiple_other_hospitals_mentioned",
    "other_hospital_mentioned",
    "other_branch_mentioned",
    "explicit_other_state_location"
]

for flag in flags_to_review:

    print("\n" + "=" * 75)
    print("FLAG:", flag)
    print("=" * 75)

    sample = (
        audit_df
        .filter(F.array_contains(F.col("flags"), flag))
        .orderBy(F.rand(seed=615))
        .limit(8)
        .select(
            "name",
            "city",
            "status",
            "other_hospitals",
            "branch_examples",
            "location_examples"
        )
    )

    display(sample)

# Count combinations of flags to see which
# types of warnings commonly occur together.

display(
    audit_df
    .withColumn(
        "non_geo_flags",
        F.expr(
            "filter(flags, x -> "
            "x != 'coordinates_not_city_verified')"
        )
    )
    .withColumn(
        "flag_combination",
        F.concat_ws(", ", F.col("non_geo_flags"))
    )
    .groupBy("flag_combination")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)



FLAG: multiple_other_hospitals_mentioned


name,city,status,other_hospitals,branch_examples,location_examples
Fortis Hospital,Bangalore,needs_review,"List(Apollo Hospitals, Awarded Best Hospital, Outlook Hospital, Top Hospitals, Watchdoq's Find Hospitals)",List(),List()
Alpha Superspeciality Hospital,Wakad,needs_review,"List(Practo's Top Hospitals, Star Health Network Hospital, Top Hospitals, Top Skin Hospitals)",List(),List()
Shenoy Hospitals,Hyderabad,needs_review,"List(Gynecology Hospital, Nutritionist Hospitals, Top General Physician Hospitals, Top Hospital, Top Periodontics Hospital, Top Skin Hospitals)",List(),List()
Ankur Hospital,Karad City,needs_review,"List(Aditya Birla Health Insurance Network Hospital, Aditya Birla Network Hospital, Ankura Hospital, Atkur Hospital, Operates Ankura Hospital, Satara District Hospitals, Top Hospitals)",List(),List()
Shree Balaji Super Speciality Hospital,Nashik,needs_review,"List(Best Hospitals, Star Health Network Hospital, Top General Physician Hospital, Top General Surgery Hospital, Top Gynecology Hospitals, Top Hospitals)",List(),List()
Harsh Hospital,Pune,needs_review,"List(Harish Hospital, Inamdar Hospital, Open Hospital, Sahyadri Hospital)",List(),List()
MYRO Clinic,Jaipur,needs_review,"List(Cocoon Hospital, Eternal Hospital)",List(),List()
Manipal Hospitals,Ghaziabad,needs_review,"List(Columbia Asia Hospital, Our Top Hospitals, Top General Physician Hospitals, Top Gynecology Hospitals, Top Hospitals, Top Internal Medicine Hospitals, Top Orthopedic Hospitals)",List(),List()



FLAG: other_hospital_mentioned


name,city,status,other_hospitals,branch_examples,location_examples
Jyothi Hospital,Kuruda,needs_review,List(Jyoti Hospital),List(),List()
Chittaranjan National Cancer Institute Blood Centre,Kolkata,needs_review,List(Top Hospitals),List(),List()
Green View Nursing Home,Laxmipur,needs_review,List(Star Health Network Hospital),List(),List()
Viveka Hospitals,Nagpur,needs_review,List(Zenith Hospital),List(),List()
Sri Aurobindo Seva Kendra,Kolkata,needs_review,List(Star Health Network Hospital),List(ENT services are provided at the clinic by Dr. Sabyasachi Gon (ENT)),List()
Thiraviam Orthopaedic Hospital,Nagercoil,needs_review,List(Kanniyakumari District Public Utilities Hospitals),List(),List()
Somawanshi Hospital,Shrigonda,needs_review,List(Somvanshi Hospital),List(),List()
K. B. Haji Bachooali Charitable Ophthalmic and ENT Hospital,Mumbai,needs_review,List(KBHB Eye Hospital),List(),List()



FLAG: other_branch_mentioned


name,city,status,other_hospitals,branch_examples,location_examples
Rangani Medicare And Dental Hospital,Rajkot,needs_review,List(),"List(Dr. Rangani Mukeshkumar Nalinbhai serves as consultant physician at the clinic (MBBS, DNB - General Medicine))",List()
"Dentistree Dental Hospitals, Tambaram",Tambaram,needs_review,"List(Dentistry Hospital, Practo Top Hospitals, Practo's Top Hospitals, Top Hospital, Top Hospitals)",List(Provides dental imaging service at the clinic),List()
"Dr Batra’s® Homeopathy Clinic in Howrah, Kolkata",Howrah,needs_review,List(),List(Experienced and certified homeopathic doctors at Barasat clinic),List()
Hearing Aid Center in Maharshi Nagar Pune,Pune,needs_review,List(),"List(Hearing aids from Signia, Phonak, Widex, Resound, Starkey, and Oticon available at the clinic)",List()
Suwarna's Dental Clinic,Nagpur,needs_review,List(),List(Dr. Suwarna Upare is the dentist at the clinic),List()
Dr. Pasricha's E.N.T. & Eye Care Centre,Gurgaon,needs_review,List(),"List(Offers ophthalmology and ENT outpatient services at a multi-speciality clinic, No on-site diagnostic or imaging service at the clinic)",List()
Arora nursing home,Agra,needs_review,List(),List(Dr. Anjana Arora is the gynecologist practicing at the clinic),List()
ANBU HOSPITAL,Theni,needs_review,"List(ANN Hospital, Ann Hospital, Anu Hospital, Anu Hospitals, Manu Hospitals, Top Hospital, Top Pediatric Hospitals)",List(Dental examination and dental procedures performed at the on-site dental clinic (as evidenced by the dental chair setup)),List()



FLAG: explicit_other_state_location


name,city,status,other_hospitals,branch_examples,location_examples
OrthoSquare,New Panvel East,needs_review,"List(Practo's Navi Mumbai Hospitals, Top Dental Hospitals, Top Hospitals)",List(),"List(Located in Mumbai, Maharashtra, India, Orthosquare is a dental clinic located in Mumbai, Maharashtra, India, Orthosquare is a hospital located in Mumbai, Maharashtra, India)"
Nalam Hospital,Chennai,needs_review,"List(Care Health Insurance Network Hospital, Nalamm Hospital, Thiyagu Piles Hospital, Top Hospitals)",List(),"List(Located in Chennai, Nalamm Hospital PVT. Limited is located in Chennai, Located in Chennai, India)"
Shamshad Childrens Hospital,Hyderabad,needs_review,List(Kid Care Children's Hospital),List(Has 1 doctor listed in the clinic profile),"List(Located in Hyderabad, Telangana)"
Manipal Hospital,Vijayawada,needs_review,"List(Top Gynecology Hospitals, Top Hospital)",List(),"List(Located in Bangalore, India)"
My Family Dentist,Thiruvanmiyur,needs_review,List(),List(),List(Located in Chennai)
Nova IVF Fertility,Lucknow,needs_review,List(Top Hospitals),List(),"List(Located in Lucknow, Hazratganj)"
Dr Todkar Hospital,Pune,needs_review,"List(CitilCares Asia Multi Specialty Hospital, Dr L H Hiranandani Hospital, Hiranandani Hospital, Kem Hospital, Poona Hospital)",List(),List(Located in Pune)
Anand Hospital,Shirol,needs_review,"List(Best Hospitals, SBI General Network Hospital, Top Hospitals, Top Orthopedic Hospitals)",List(),List(Located in Mumbai)


flag_combination,count
,6300
other_hospital_mentioned,1071
multiple_other_hospitals_mentioned,1052
repeated_facility_name,343
specialty_emergency_mentions,304
"multiple_other_hospitals_mentioned, repeated_facility_name",270
other_branch_mentioned,241
"other_hospital_mentioned, repeated_facility_name",127
"other_hospital_mentioned, specialty_emergency_mentions",48
"multiple_other_hospitals_mentioned, specialty_emergency_mentions",45


---

## Phase 2 follow-up — Claim attribution

Check more specific attribution issues without changing the original dataset.

*Source notebook: `care_finder_phase2_v2 (1) (1).ipynb`*

**Before running this section:** It was tested separately. Restart the Python session or check existing variables if you are rerunning the analysis after an earlier stage.

# Care Finder Phase 2 v2 — Claim attribution audit
This is a preliminary check of possible attribution issues. It previews results by default. It does not measure clinical reliability and does not modify the source table.


In [0]:
# Care Finder India — Phase 2 v2: conservative claim attribution audit.
# Run as a SEPARATE notebook. Original source table remains unchanged.

import re
from collections import Counter
from urllib.parse import urlparse

from pyspark.sql import functions as F, types as T

SOURCE_TABLE = "workspace.default.care_finder_facilities"
OUTPUT_TABLE = "workspace.default.care_finder_record_quality_v2"
WRITE_RESULTS = False  # Only change after reviewing the output.

# Explicitly named institution-specific domains; not generic state portals.
INSTITUTION_DOMAINS = {
    "igh.delhi.gov.in", "lnjp.delhi.gov.in", "gtbh.delhi.gov.in",
    "dduh.delhi.gov.in", "rgssh.delhi.gov.in", "ggsgh.delhi.gov.in",
    "lbsh.delhi.gov.in", "gbpant.delhi.gov.in", "vmmc-sjh.mohfw.gov.in",
}

SERVICE = re.compile(
    r"\b(?:dialysis|ha?emodialysis|nicu|neonatal intensive|mri|magnetic resonance|"
    r"emergency|trauma|burns? care|burns? ward|c[ -]?section|caesarean|cesarean)\b", re.I
)
SCOPE = re.compile(r"\b(dental|dentistry|ophthalmic|eye|ocular|pediatric|paediatric|psychiatric)\b", re.I)
GENERIC_REFERENCE = re.compile(
    r"\b(?:top|best|award(?:ed)?|ranked|network|insurance|cashless|"
    r"empanelled|directory|listing|list of|nearby|compared with|"
    r"affiliated|collaborat(?:es|ion|ing)|formerly known as)\b", re.I
)
# Extract names only from targeted attribution constructions, not everywhere.
HOSPITAL_NAME = r"([A-Z][A-Za-z0-9&.'-]*(?:\s+[A-Z][A-Za-z0-9&.'-]*){0,6}\s+(?:Hospital|Hospitals|Medical Cent(?:er|re)))"
NAMED_PERFORMER = re.compile(
    r"\b" + HOSPITAL_NAME + r"\s+(?:provides?|offers?|performs?|operates?|has|runs)\b", re.I
)
SERVICE_AT_NAMED = re.compile(
    r"\b(?:at|by|from)\s+" + HOSPITAL_NAME + r"\b", re.I
)
BRANCH_NAME = re.compile(
    r"\b(?:at|in|from|by)\s+([A-Za-z][A-Za-z -]{2,35}?)\s+(?:clinic|branch)\b", re.I
)
# Area references need to be concrete and clearly tied to a physical site.
CITY_IN = re.compile(r"\b(?:located|situated|based)\s+in\s+([A-Za-z][A-Za-z ]{2,35})", re.I)
STOP_NAMES = {
    "best hospital", "top hospitals", "top hospital", "award best hospital",
    "network hospital", "network hospitals", "government hospitals",
    "society hospitals", "find hospitals", "general physician hospital",
}


def norm(value):
    return re.sub(r"[^a-z0-9]+", " ", str(value or "").casefold()).strip()


def identity_tokens(value):
    stop = {"hospital", "hospitals", "medical", "centre", "center", "clinic",
            "health", "care", "the", "of", "and", "pvt", "ltd", "private"}
    return {w for w in norm(value).split() if len(w) >= 3 and w not in stop}


def other_institution(mentioned, own_name):
    m = norm(mentioned)
    if m in STOP_NAMES or not m:
        return False
    ours, theirs = identity_tokens(own_name), identity_tokens(mentioned)
    # Conservative. Similar names may be aliases and should be checked separately.
    return bool(ours and theirs and ours.isdisjoint(theirs))


def unique_items(values):
    return list(dict.fromkeys(str(v).strip() for v in (values or []) if v and str(v).strip()))


def audit_record(row):
    name = str(row.get("name") or "")
    city = norm(row.get("address_city"))
    address = norm(row.get("address_line1"))
    state = norm(row.get("address_stateOrRegion"))
    claims = unique_items(row.get("evidence_list"))
    urls = unique_items(row.get("source_urls_list"))

    # Only service-relevant statements become candidates for exclusion.
    excluded_claims = []
    other_branch_claims = []
    restricted_emergency_claims = []
    other_hospital_claims = []
    location_claims = []
    benign_references = []
    affected_institutions = set()

    for claim in claims:
        service_match = bool(SERVICE.search(claim))

        if "emergency" in norm(claim).split() and SCOPE.search(claim):
            restricted_emergency_claims.append(claim)

        # Different hospital actively provides the service, or it occurs at Hospital B.
        external_names = []
        if service_match:
            for pattern in (NAMED_PERFORMER, SERVICE_AT_NAMED):
                external_names.extend(m.group(1).strip() for m in pattern.finditer(claim))

        if external_names and GENERIC_REFERENCE.search(claim):
            benign_references.append(claim)
        else:
            for named in external_names:
                if other_institution(named, name):
                    affected_institutions.add(norm(named))
                    other_hospital_claims.append(claim)
                    excluded_claims.append(claim)
                    break

        # Explicit other-branch attribution, e.g. 'at Attapur clinic'.
        # Require a service word to avoid excluding mundane branch references.
        for match in BRANCH_NAME.finditer(claim):
            branch = norm(match.group(1))
            if (service_match and branch and branch not in norm(name)
                    and branch not in address and branch != city):
                other_branch_claims.append(claim)
                excluded_claims.append(claim)
                break

        # Log but do not auto-exclude other-city statements.
        if service_match and CITY_IN.search(claim):
            city_mention = norm(CITY_IN.search(claim).group(1))
            if city and city not in city_mention and city_mention not in city:
                location_claims.append(claim)

    institutional_foreign = []
    for link in urls:
        try:
            host = (urlparse(link).hostname or "").casefold()
        except ValueError:
            continue
        if host in INSTITUTION_DOMAINS and state != "delhi":
            institutional_foreign.append(host)
    institutional_foreign = sorted(set(institutional_foreign))

    # Strong signal of multi-institution contamination, not proof of falsity.
    # Apply to Aravind-like cases with several unrelated institution sources
    # AND several separate facility references in the record.
    named_anywhere = set()
    for claim in claims:
        for match in re.finditer(HOSPITAL_NAME, claim, re.I):
            candidate = match.group(1)
            if GENERIC_REFERENCE.search(claim):
                continue
            if other_institution(candidate, name):
                named_anywhere.add(norm(candidate))

    severe_source_mix = (len(institutional_foreign) >= 3
                         and len(named_anywhere) >= 2)

    # A repeated name, restricted emergency claim, or a non-city-verified
    # coordinate is NOT by itself evidence of a wrong facility identity.
    flags = []
    if other_hospital_claims:
        flags.append("service_claim_attributed_to_other_hospital")
    if other_branch_claims:
        flags.append("service_claim_attributed_to_other_branch")
    if location_claims:
        flags.append("service_location_discrepancy_to_review")
    if severe_source_mix:
        flags.append("multiple_unrelated_institution_sources")
    if restricted_emergency_claims:
        flags.append("specialty_emergency_only_review_scope")

    if severe_source_mix:
        status = "hold_for_review"
    elif other_hospital_claims or other_branch_claims or location_claims:
        status = "needs_claim_review"
    else:
        status = "no_specific_attribution_conflict_detected"

    return {
        "unique_id": str(row.get("unique_id") or ""),
        "name": name,
        "city": str(row.get("address_city") or ""),
        "status": status,
        "flags": flags,
        "excluded_claims": unique_items(excluded_claims)[:30],
        "other_hospital_claims": unique_items(other_hospital_claims)[:5],
        "other_branch_claims": unique_items(other_branch_claims)[:5],
        "restricted_emergency_claims": unique_items(restricted_emergency_claims)[:5],
        "location_claims": unique_items(location_claims)[:5],
        "unrelated_institution_domains": institutional_foreign[:15],
        "coordinates_city_verified": "not city verified" not in str(row.get("location_status") or "").lower(),
        "repeated_name": bool(row.get("repeated_name")),
        "claim_count": len(claims),
    }


# ============================================================
# SMOKE TESTS (NOT A MEASURE OF REAL-WORLD ACCURACY)
# ============================================================

def fake(name, claims, city="Hyderabad", state="Telangana", urls=None):
    return dict(unique_id="test", name=name, address_city=city,
                address_stateOrRegion=state, address_line1="Kothapet",
                evidence_list=claims, source_urls_list=urls or [],
                location_status="Within India bounds - not city verified",
                repeated_name=False)

assert audit_record(fake("Alpha Hospital", ["Ranked among the Top Hospitals in India"]))["status"] == "no_specific_attribution_conflict_detected"
assert audit_record(fake("Smile Gallery Dental", ["Empanelled with Jawaharlal Nehru Cancer Hospital"]))["status"] == "no_specific_attribution_conflict_detected"
branch = audit_record(fake("Partha Dental Kothapet", ["Provides dental emergency services at Attapur clinic"]))
assert len(branch["other_branch_claims"]) == 1
assert len(branch["restricted_emergency_claims"]) == 1
assert branch["status"] == "needs_claim_review"
assert audit_record(fake("Example Hospital", ["Provides dialysis treatment"]))["status"] == "no_specific_attribution_conflict_detected"
foreign = audit_record(fake("Aravind Eye Hospital", [
    "Janakpuri Super Speciality Hospital is listed in Delhi",
    "Delhi hospital details about Lok Nayak Hospital",
    "Indira Gandhi Hospital provides MRI scans",
], urls=["https://igh.delhi.gov.in/igh/services-and-facilities",
         "https://lnjp.delhi.gov.in/", "https://gtbh.delhi.gov.in/"]))
assert foreign["status"] == "hold_for_review"
print("5 smoke tests passed")

# ============================================================
# AUDIT THE FULL DATASET
# ============================================================

columns = ["unique_id", "name", "address_city", "address_stateOrRegion",
           "address_line1", "evidence_list", "source_urls_list",
           "location_status", "repeated_name"]
source_df = spark.table(SOURCE_TABLE)
missing = sorted(set(columns) - set(source_df.columns))
assert not missing, f"Missing columns: {missing}"
results = [audit_record(row.asDict(recursive=True))
           for row in source_df.select(*columns).toLocalIterator()]
print("Records audited:", len(results))
print("Statuses:", dict(Counter(x["status"] for x in results)))
print("Flag counts:", Counter(f for x in results for f in x["flags"]).most_common())

schema = T.StructType([
    T.StructField("unique_id", T.StringType()),
    T.StructField("name", T.StringType()),
    T.StructField("city", T.StringType()),
    T.StructField("status", T.StringType()),
    T.StructField("flags", T.ArrayType(T.StringType())),
    T.StructField("excluded_claims", T.ArrayType(T.StringType())),
    T.StructField("other_hospital_claims", T.ArrayType(T.StringType())),
    T.StructField("other_branch_claims", T.ArrayType(T.StringType())),
    T.StructField("restricted_emergency_claims", T.ArrayType(T.StringType())),
    T.StructField("location_claims", T.ArrayType(T.StringType())),
    T.StructField("unrelated_institution_domains", T.ArrayType(T.StringType())),
    T.StructField("coordinates_city_verified", T.BooleanType()),
    T.StructField("repeated_name", T.BooleanType()),
    T.StructField("claim_count", T.IntegerType()),
])
result_df = spark.createDataFrame(results, schema)

print("STATUS COUNTS")
display(result_df.groupBy("status").count().orderBy("status"))

print("KNOWN TEST RECORDS")
display(result_df.filter(F.col("name").rlike(
    "(?i)Aravind Eye Hospital|Partha Dental Kothapet|Smile Gallery"
)).select("name", "city", "status", "flags", "other_hospital_claims",
          "other_branch_claims", "unrelated_institution_domains"))

print("SAMPLE FLAGGED CLAIMS")
display(result_df.filter(F.col("status") != "no_specific_attribution_conflict_detected")
        .orderBy(F.rand(seed=615))
        .select("name", "city", "status", "flags", "excluded_claims").limit(25))

print("SPECIALTY-ONLY EMERGENCY EXAMPLES")
display(result_df.filter(F.array_contains(F.col("flags"), "specialty_emergency_only_review_scope"))
        .orderBy(F.rand(seed=615))
        .select("name", "city", "restricted_emergency_claims").limit(12))

if WRITE_RESULTS:
    result_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(OUTPUT_TABLE)
    print("Saved separate audit table:", OUTPUT_TABLE)
else:
    print("Preview only. No original records or live rankings changed.")


5 smoke tests passed
Records audited: 10088
Statuses: {'hold_for_review': 1, 'no_specific_attribution_conflict_detected': 10035, 'needs_claim_review': 52}
Flag counts: [('specialty_emergency_only_review_scope', 537), ('service_claim_attributed_to_other_hospital', 38), ('service_claim_attributed_to_other_branch', 8), ('service_location_discrepancy_to_review', 6), ('multiple_unrelated_institution_sources', 1)]
STATUS COUNTS


status,count
hold_for_review,1
needs_claim_review,52
no_specific_attribution_conflict_detected,10035


KNOWN TEST RECORDS


name,city,status,flags,other_hospital_claims,other_branch_claims,unrelated_institution_domains
Aravind Eye Hospital,Hyderabad,hold_for_review,List(multiple_unrelated_institution_sources),List(),List(),"List(dduh.delhi.gov.in, gbpant.delhi.gov.in, ggsgh.delhi.gov.in, gtbh.delhi.gov.in, igh.delhi.gov.in, lbsh.delhi.gov.in, lnjp.delhi.gov.in, rgssh.delhi.gov.in, vmmc-sjh.mohfw.gov.in)"
Partha Dental Kothapet,Hyderabad,needs_claim_review,"List(service_claim_attributed_to_other_branch, specialty_emergency_only_review_scope)",List(),List(Provides dental emergency services at Attapur clinic),List()
Priyamvada Birla Aravind Eye Hospital,Kolkata,no_specific_attribution_conflict_detected,List(specialty_emergency_only_review_scope),List(),List(),List()
"Smile Gallery Dental Wellness Center Saurabh Nursing Home, 82 Marwari Road, Near Azad Market, Bhopal, M.P. 462001",Bhopal,no_specific_attribution_conflict_detected,List(specialty_emergency_only_review_scope),List(),List(),List()


SAMPLE FLAGGED CLAIMS


name,city,status,flags,excluded_claims
Anandalok Hospital,Kolkata,needs_claim_review,List(service_claim_attributed_to_other_hospital),List(Emergency & Trauma Care department is available at Anandaloke Hospital)
Parekh Hospital,Mumbai,needs_claim_review,List(service_location_discrepancy_to_review),List()
Narayana Hrudayalaya,Bengaluru,needs_claim_review,List(service_claim_attributed_to_other_hospital),"List(Sparsh Orthopaedic and Spine Hospital provides orthopedic procedures including arthroplasty, spinal surgery, trauma and emergency care)"
Aravind Eye Hospital,Hyderabad,hold_for_review,List(multiple_unrelated_institution_sources),List()
Dr Parag Hospital,Mandvi,needs_claim_review,List(service_claim_attributed_to_other_hospital),"List(Harsh Medical Centre provides 24 hours round the clock indoor and outdoor services for critical medical situations, i.e., 24/7 emergency care.)"
Sugam Hospitals,Chennai,needs_claim_review,List(service_claim_attributed_to_other_hospital),List(Provides 24/7 emergency care at Sugan Hospital)
Yadgire Super Specialty Hospital,Amravati,needs_claim_review,List(service_claim_attributed_to_other_hospital),"List(MRI services at District Hospital, Amravati are provided under NHM PPP with M/S Krishna Diagnostic Pvt Ltd., MoU with M/S Krishna Diagnostic Pvt Ltd dated 29 April 2024 for MRI services at District Hospital, Amravati., MRI machine installed at District Hospital, Amravati under NHM PPP.)"
Partha Dental Vizag,Vizag,needs_claim_review,"List(service_claim_attributed_to_other_branch, specialty_emergency_only_review_scope)",List(Provides dental emergency services at Tirupati clinic)
Srp Hospital,Aranthangi,needs_claim_review,List(service_claim_attributed_to_other_hospital),List(Offers dialysis services at SPMM Hospital.)
Vijay Hospital,Kolhapur,needs_claim_review,List(service_claim_attributed_to_other_hospital),"List(Has a dedicated dialysis unit at Vijan Hospital, Nashik, Dialysis is provided at Vijan Hospital, Nashik via a dedicated dialysis unit, On-site dialysis machines as part of the dialysis unit at Vijan Hospital, Nashik)"


SPECIALTY-ONLY EMERGENCY EXAMPLES


name,city,restricted_emergency_claims
Tathastu Eye Hospital,Ahmedabad,List(Provides 24/7 emergency eye care)
SARVDRISHTI EYE HOSPITAL PRIVATE LIMITED,Patna,List(24/7 emergency care for ocular trauma)
Teeth Care Multispeciality Dental Clinic,Kolkata,List(Provides dental emergency services)
Jabalpur Dental Clinic,Jabalpur,List(Provides 24/7 emergency dental support)
Kalyani Specialty Dental Clinic,Guntur,List(24-hour emergency dental clinic)
"Sarkar Multi-Speciality Hospital & IVF Centre, Agra",Agra,List(Provides pediatric and geriatric emergency care)
Dr White's Dental And Aesthetics,Bhubaneswar,"List(Emergency dental care available, Emergency dental care / urgent care)"
ASG Eye Hospital,Nashik,"List(Provides 24/7 emergency eye care, 24/7 emergency ophthalmic services)"
Dr. Dupare Dental Clinic,Nagpur,List(Provides emergency dental care)
Multispeciality Family Dental Care,Noida,"List(Offers emergency dental services, Emergency dental service available, Emergency Dental Service)"


Preview only. No original records or live rankings changed.


---

## Phase 3 — Context checks and NLP pilot

Test contextual checks, candidate retrieval, and optional language-model comparisons.

*Source notebook: `care_finder_phase3_pilot (1).ipynb`*

**Before running this section:** It was tested separately. Restart the Python session or check existing variables if you are rerunning the analysis after an earlier stage.

# Phase 3 — Context-aware audit + NLP pilot

This section explores records without changing the source table. It was originally run as a separate Databricks notebook. A model download may require outbound access and suitable libraries. Initial NLP results are exploratory—not verified hospital services.

In [0]:
import re
from collections import Counter
from difflib import SequenceMatcher
from urllib.parse import urlparse

SOURCE_TABLE = 'workspace.default.care_finder_facilities'
MAX_FACILITIES = 10088  # audited read-only; change to 200 for faster smoke runs
PILOT_FACILITIES = 100  # NLP is deliberately bounded
MAX_CLAIMS_PER_FACILITY = 12
RUN_EMBEDDINGS = True      # requires sentence-transformers + model download
RUN_NLI = True           # turn on after embeddings work; heavier model download

FOREIGN_INSTITUTION_DOMAINS = {
    'igh.delhi.gov.in', 'lnjp.delhi.gov.in', 'gtbh.delhi.gov.in',
    'dduh.delhi.gov.in', 'rgssh.delhi.gov.in', 'ggsgh.delhi.gov.in',
    'lbsh.delhi.gov.in', 'gbpant.delhi.gov.in', 'vmmc-sjh.mohfw.gov.in',
}
SERVICE = re.compile(r'\b(?:dialysis|ha?emodialysis|nicu|neonatal intensive|mri|magnetic resonance|emergency|trauma|burns?|c[ -]?section|caesarean|cesarean)\b', re.I)
HOSPITAL_NAME = r'([A-Z][A-Za-z0-9&.\'-]*(?:\s+[A-Z][A-Za-z0-9&.\'-]*){0,6}\s+(?:Hospitals?|Medical Cent(?:er|re)))'
NAMED_PERFORMER = re.compile(r'\b' + HOSPITAL_NAME + r'\s+(?:provides?|offers?|performs?|operates?|has|runs)\b', re.I)
SERVICE_AT_NAMED = re.compile(r'\b(?:at|by|from)\s+' + HOSPITAL_NAME + r'\b', re.I)
BRANCH = re.compile(r'\b(?:at|in|from)\s+([A-Z][A-Za-z\-]{2,30}(?:\s+[A-Z][A-Za-z\-]{2,30})?)\s+(?:clinic|branch)\b')
SPECIALTY_EMERGENCY = re.compile(r'\b(?:dental|dentistry|eye|ocular|ophthalmic|psychiatric|paediatric|pediatric)\s+(?:\w+\s+){0,2}emergency\b|\bemergency\s+(?:\w+\s+){0,2}(?:dental|dentistry|eye|ocular|ophthalmic|psychiatric|paediatric|pediatric)\b', re.I)
GENERAL_EMERGENCY = re.compile(r'\b(?:general|adult|accident|casualty)\b', re.I)
IGNORE_CONTEXT = re.compile(r'\b(?:top|best|ranked|insurance|cashless|network hospitals?|empanelled|directory|listing|nearby|affiliated|collaborat\w*|formerly known as|merger|acquired)\b', re.I)
GENERIC_NAMES = {'top hospitals', 'best hospital', 'network hospital', 'society hospitals', 'government hospitals', 'top hospital', 'find hospitals'}
STOP = {'hospital', 'hospitals', 'medical', 'centre', 'center', 'clinic', 'care', 'health', 'and', 'of', 'the', 'pvt', 'ltd', 'private', 'super', 'speciality', 'specialty'}


def norm(s):
    return re.sub(r'[^a-z0-9]+', ' ', str(s or '').casefold()).strip()


def unique_items(a):
    return list(dict.fromkeys(str(x).strip() for x in (a or []) if x and str(x).strip()))


def identity_tokens(s):
    return {t for t in norm(s).split() if len(t) > 2 and t not in STOP}


def is_possible_alias(own, mentioned):
    a, b = norm(own), norm(mentioned)
    aa, bb = ' '.join(sorted(identity_tokens(a))), ' '.join(sorted(identity_tokens(b)))
    if not aa or not bb:
        return False
    if len(identity_tokens(a) & identity_tokens(b)) > 0:
        return True
    # Handles close typos (Anandalok/Anandaloke; Sugam/Sugan),
    # but does NOT assert that they are the same real-world facility.
    if SequenceMatcher(None, aa, bb).ratio() >= (0.78 if len(aa.split()) == len(bb.split()) == 1 else 0.85):
        return True
    # GAIMS = Gujarat Adani Institute of Medical Sciences (example).
    words = [w for w in norm(own).split() if w not in {'of','the','and','hospital','hospitals','pvt','ltd'}]
    acronym = ''.join(w[0] for w in words if w)
    if len(acronym) >= 3 and acronym in identity_tokens(mentioned):
        return True
    return False


def is_other_hospital(own, mention):
    m = norm(mention)
    if not m or m in GENERIC_NAMES:
        return False
    return not is_possible_alias(own, mention)


def emergency_scope(claim):
    if not re.search(r'\bemergency\b', claim, re.I):
        return 'not_emergency'
    if re.search(r'\b(?:dentistry|ophthalmology|pediatrics|paediatrics)\s+(?:and|,)\s+emergency medicine\b', claim, re.I):
        return 'general_or_mixed'
    specialty = bool(SPECIALTY_EMERGENCY.search(claim))
    # If both adult/general and pediatric appear, do not label specialty-only.
    if specialty and not GENERAL_EMERGENCY.search(claim):
        return 'specialty_specific'
    if GENERAL_EMERGENCY.search(claim) or re.search(r'\bemergency (?:department|medicine)\b', claim, re.I):
        return 'general_or_mixed'
    return 'unspecified'


def audit_record(row):
    name = str(row.get('name') or '')
    own_city = norm(row.get('address_city'))
    own_address = norm(row.get('address_line1'))
    own_state = norm(row.get('address_stateOrRegion'))
    claims = unique_items(row.get('evidence_list'))
    links = unique_items(row.get('source_urls_list'))
    other_hospital, other_branch, specialty = [], [], []
    alias_candidates, review_locations = [], []

    for claim in claims:
        if emergency_scope(claim) == 'specialty_specific':
            specialty.append(claim)
        if not SERVICE.search(claim):
            continue
        # Only exclude a claim when the service is *explicitly* attributed to another institution.
        if not IGNORE_CONTEXT.search(claim):
            for pattern in (NAMED_PERFORMER, SERVICE_AT_NAMED):
                for match in pattern.finditer(claim):
                    mentioned = match.group(1).strip()
                    if is_possible_alias(name, mentioned):
                        alias_candidates.append(claim)
                    elif is_other_hospital(name, mentioned):
                        other_hospital.append(claim)
        # Named location required. Generic 'at the clinic' cannot qualify.
        for match in BRANCH.finditer(claim):
            branch = norm(match.group(1))
            if branch in ('the', 'a', 'an', 'this', 'our', 'same', 'local'):
                continue
            if (branch and branch != own_city and branch not in own_address
                    and branch not in norm(name)):
                other_branch.append(claim)
        # Location mismatches are *not inferred* from neighborhood/city labels.
        # Geographic comparison requires a verified geography mapping or source-specific address.

    foreign_domains = []
    for u in links:
        try:
            hostname = (urlparse(u).hostname or '').lower()
        except ValueError:
            continue
        if hostname in FOREIGN_INSTITUTION_DOMAINS and own_state != 'delhi':
            foreign_domains.append(hostname)
    foreign_domains = sorted(set(foreign_domains))
    named_other = set()
    for claim in claims:
        if IGNORE_CONTEXT.search(claim):
            continue
        for m in re.finditer(HOSPITAL_NAME, claim, re.I):
            if is_other_hospital(name, m.group(1)):
                named_other.add(norm(m.group(1)))
    severe = len(foreign_domains) >= 3 and len(named_other) >= 2
    excluded = unique_items(other_hospital + other_branch)
    status = ('hold_for_review' if severe else
              'needs_claim_review' if excluded else
              'no_specific_attribution_conflict_detected')
    return {
        'unique_id': str(row.get('unique_id') or ''), 'name': name,
        'city': str(row.get('address_city') or ''), 'status': status,
        'other_hospital_claims': unique_items(other_hospital)[:8],
        'other_branch_claims': unique_items(other_branch)[:8],
        'excluded_claims': excluded[:40],
        'alias_candidates_not_excluded': unique_items(alias_candidates)[:5],
        'specialty_emergency_claims': unique_items(specialty)[:6],
        'foreign_institution_domains': foreign_domains,
        'coordinates_city_verified': 'not city verified' not in str(row.get('location_status') or '').lower(),
        'repeated_name': bool(row.get('repeated_name')),
        'claim_count': len(claims),
    }

In [0]:
# Test the rules on a few known examples. These are not accuracy estimates.
def fake(name, evidence, city='Hyderabad', addr='Kothapet', urls=None):
    return dict(unique_id='test', name=name, address_city=city,
                address_line1=addr, address_stateOrRegion='Telangana',
                evidence_list=evidence, source_urls_list=urls or [],
                location_status='Within India bounds - not city verified',
                repeated_name=False)

assert is_possible_alias('Anandalok Hospital', 'Anandaloke Hospital')
assert is_possible_alias('Sugam Hospitals', 'Sugan Hospital')
assert is_possible_alias('Gujarat Adani Institute of Medical Sciences', 'GAIMS Hospital')
assert not audit_record(fake('St George Healthcare', ['Provides dialysis at the clinic']))['excluded_claims']
assert not audit_record(fake('Ortho Care', ['Operates an emergency unit at a multi speciality clinic']))['excluded_claims']
assert audit_record(fake('Partha Dental Kothapet', ['Provides dental emergency services at Attapur clinic']))['other_branch_claims']
assert emergency_scope('24x7 emergency care including adult and paediatric trauma') == 'general_or_mixed'
assert emergency_scope('Provides emergency dental care') == 'specialty_specific'
assert emergency_scope('Dentistry and Emergency Medicine departments') != 'specialty_specific'
assert audit_record(fake('Alpha Hospital', ['Beta Hospital provides dialysis treatment']))['other_hospital_claims']
assert audit_record(fake('Aravind Eye Hospital', [
    'Janakpuri Super Speciality Hospital is listed in Delhi',
    'Lok Nayak Hospital performs surgery',
    'Indira Gandhi Hospital provides MRI scans',
], urls=['https://igh.delhi.gov.in/', 'https://lnjp.delhi.gov.in/', 'https://gtbh.delhi.gov.in/']))['status'] == 'hold_for_review'
print('11 attribution/scope regression tests passed')

11 attribution/scope regression tests passed


In [0]:
# Review the dataset without writing any new tables.
from pyspark.sql import functions as F
source = spark.table(SOURCE_TABLE)
needed = ['unique_id', 'name', 'address_city', 'address_line1', 'address_stateOrRegion',
          'evidence_list', 'source_urls_list', 'location_status', 'repeated_name']
assert set(needed).issubset(source.columns), f'Missing columns: {set(needed)-set(source.columns)}'
rows = source.select(*needed).limit(MAX_FACILITIES).toLocalIterator()
reviews = [audit_record(r.asDict(recursive=True)) for r in rows]
print('Audited:', len(reviews))
print('Review statuses:', dict(Counter(r['status'] for r in reviews)))
for term in ['Aravind Eye Hospital', 'Partha Dental Kothapet', 'Anandalok Hospital',
             'Sugam Hospitals', 'Gujarat Adani Institute of Medical Sciences']:
    found = [r for r in reviews if term.lower() in r['name'].lower()]
    for r in found[:2]:
        print('\n', r['name'], '|', r['status'])
        print('Foreign domains:', r['foreign_institution_domains'][:5])
        print('Other-hospital claims:', r['other_hospital_claims'][:2])
        print('Other-branch claims:', r['other_branch_claims'][:2])
        print('Alias candidates:', r['alias_candidates_not_excluded'][:2])

Audited: 10088
Review statuses: {'hold_for_review': 1, 'no_specific_attribution_conflict_detected': 10024, 'needs_claim_review': 63}

 Aravind Eye Hospital | hold_for_review
Foreign domains: ['dduh.delhi.gov.in', 'gbpant.delhi.gov.in', 'ggsgh.delhi.gov.in', 'gtbh.delhi.gov.in', 'igh.delhi.gov.in']
Other-hospital claims: []
Other-branch claims: []
Alias candidates: []

 Priyamvada Birla Aravind Eye Hospital | no_specific_attribution_conflict_detected
Foreign domains: []
Other-hospital claims: []
Other-branch claims: []
Alias candidates: []

 Partha Dental Kothapet | needs_claim_review
Foreign domains: []
Other-hospital claims: []
Other-branch claims: ['Provides dental emergency services at Attapur clinic']
Alias candidates: []

 Anandalok Hospital | no_specific_attribution_conflict_detected
Foreign domains: []
Other-hospital claims: []
Other-branch claims: []
Alias candidates: ['Emergency & Trauma Care department is available at Anandaloke Hospital']

 Sugam Hospitals | no_specific_attr

In [0]:
# Try the NLP methods on a smaller sample rather than every record.
# Candidate evidence is drawn across multiple original fields; repeated text is deduplicated.
# These fields are all claims, not independent validation sources.
import pandas as pd
PILOT_COLUMNS = ['unique_id', 'name', 'address_city', 'address_line1', 'address_stateOrRegion', 'source_urls_list', 'description',
                 'capability_list', 'procedure_list', 'equipment_list', 'evidence_list']
pilot_rows = (source.select(*PILOT_COLUMNS)
              .orderBy(F.rand(seed=615)).limit(PILOT_FACILITIES).collect())
claim_rows = []
for r in pilot_rows:
    d = r.asDict(recursive=True)
    audit = audit_record(d)
    excluded = set(audit['excluded_claims'])
    claims = unique_items([d.get('description')] +
                          sum([d.get(k) or [] for k in ['capability_list','procedure_list','equipment_list','evidence_list']], []))
    # Prioritize service-specific evidence while retaining a few generic negatives.
    prioritized = [c for c in claims if SERVICE.search(c)] + [c for c in claims if not SERVICE.search(c)]
    for claim in prioritized[:MAX_CLAIMS_PER_FACILITY]:
        if claim in excluded:
            continue
        if len(claim) < 12:
            continue
        claim_rows.append(dict(facility_id=d['unique_id'], facility=d['name'],
                               claim=claim, emergency_scope=emergency_scope(claim)))
pilot_df = pd.DataFrame(claim_rows)
print('Facilities in pilot:', len(pilot_rows), '| unique candidate passages:', len(pilot_df))
display(pilot_df.head(12))

Facilities in pilot: 100 | unique candidate passages: 1118


facility_id,facility,claim,emergency_scope
e3e0514a-1dc5-48fb-9827-e8cf904e3332,"Infinit Dental Solutions - Jayanagar 9th block, South Bangalore","Infinit Dental Solutions is a private dental clinic established in 2006 in Jayanagar 9th Block, South Bangalore, offering general dentistry, root canal treatment, dental implants, extractions, and related services.",not_emergency
e3e0514a-1dc5-48fb-9827-e8cf904e3332,"Infinit Dental Solutions - Jayanagar 9th block, South Bangalore",Partially equipped for minor dental emergencies,not_emergency
e3e0514a-1dc5-48fb-9827-e8cf904e3332,"Infinit Dental Solutions - Jayanagar 9th block, South Bangalore",Serves NRIs and overseas dental tourists,not_emergency
e3e0514a-1dc5-48fb-9827-e8cf904e3332,"Infinit Dental Solutions - Jayanagar 9th block, South Bangalore",ISO 9001-2008 certified,not_emergency
e3e0514a-1dc5-48fb-9827-e8cf904e3332,"Infinit Dental Solutions - Jayanagar 9th block, South Bangalore",Specialists in cosmetic dentistry and implant dentistry,not_emergency
e3e0514a-1dc5-48fb-9827-e8cf904e3332,"Infinit Dental Solutions - Jayanagar 9th block, South Bangalore",Specialists in maxillofacial prosthodontics and implant dentistry,not_emergency
e3e0514a-1dc5-48fb-9827-e8cf904e3332,"Infinit Dental Solutions - Jayanagar 9th block, South Bangalore","Specialists in orthodontics, periodontics, endodontics, pediatric dentistry, and oral & maxillofacial surgery",not_emergency
e3e0514a-1dc5-48fb-9827-e8cf904e3332,"Infinit Dental Solutions - Jayanagar 9th block, South Bangalore",Offers orthodontic treatments including Invisalign and ClearPath Orthodontics,not_emergency
e3e0514a-1dc5-48fb-9827-e8cf904e3332,"Infinit Dental Solutions - Jayanagar 9th block, South Bangalore","In-house prosthodontists specialized in smile design, implants, crowns, and bridges",not_emergency
e3e0514a-1dc5-48fb-9827-e8cf904e3332,"Infinit Dental Solutions - Jayanagar 9th block, South Bangalore","Offers comprehensive dental care including oral prophylaxis, dental hygiene, implantology, orthodontics, aesthetic dentistry, pediatric dentistry, root canal treatment, oral surgery, and anesthesia-assisted treatment",not_emergency


In [0]:
# Optional test using sentence embeddings (requires a model download).
# Similarity finds possible matches; it does not verify a hospital service.
SERVICE_QUERIES = {
    'Dialysis': 'The hospital currently provides dialysis or hemodialysis treatment.',
    'NICU': 'The hospital currently provides neonatal intensive care for newborn infants.',
    'MRI': 'The facility performs MRI scans for patients.',
    'Trauma Care': 'The hospital treats major trauma injuries and accident victims.',
    'Burn Care': 'The hospital treats patients with burn injuries.',
    'General Emergency': 'The hospital operates a general emergency department for patients.',
}
embedding_results = pd.DataFrame()
if RUN_EMBEDDINGS and not pilot_df.empty:
    try:
        import numpy as np
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')
        passages = pilot_df['claim'].tolist()
        qnames = list(SERVICE_QUERIES)
        texts = [SERVICE_QUERIES[k] for k in qnames]
        document_vectors = model.encode(passages, batch_size=32, normalize_embeddings=True, show_progress_bar=False)
        query_vectors = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        similarity = query_vectors @ document_vectors.T
        picks = []
        for i, service in enumerate(qnames):
            for j in np.argsort(-similarity[i])[:8]:
                rec = pilot_df.iloc[int(j)]
                picks.append({'service': service, 'similarity': round(float(similarity[i, j]), 3),
                              'facility': rec['facility'], 'claim': rec['claim'],
                              'scope': rec['emergency_scope']})
        embedding_results = pd.DataFrame(picks)
        display(embedding_results)
    except Exception as exc:
        print('Embedding pilot unavailable:', type(exc).__name__, str(exc)[:450])
        print('Install sentence-transformers in the notebook environment and retry.')
else:
    print('Embedding step disabled or no pilot claims.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

service,similarity,facility,claim,scope
Dialysis,0.76,"Ekana Hospital And Research Centre, Gomtinagar Vistaar",Provides dialysis treatment,not_emergency
Dialysis,0.76,Sharda Hospital,Provides dialysis treatment,not_emergency
Dialysis,0.76,Narinder Mohan Hospital & Research,Provides dialysis treatment,not_emergency
Dialysis,0.755,"Ekana Hospital And Research Centre, Gomtinagar Vistaar",Provides dialysis services,not_emergency
Dialysis,0.719,Asian Dwarkadas Jalan Super Speciality Hospital,Offers haemodialysis and peritoneal dialysis services,not_emergency
Dialysis,0.712,Jj Plus Hospitals Aurangabad,Kidney Care & Dialysis Center,not_emergency
Dialysis,0.693,GVN Hospital,ICU with dialysis facilities,not_emergency
Dialysis,0.69,Aravind Eye Hospital,Hemodialysis is performed in three units as a regular service,not_emergency
NICU,0.79,Apex Children's Hospital,Has a Neonatal Intensive Care Unit (NICU),not_emergency
NICU,0.79,Narinder Mohan Hospital & Research,Neonatal Intensive Care Unit (NICU),not_emergency


In [0]:
# Optional zero-shot NLI test (requires transformers and a model download).
# These scores are exploratory and should not be treated as verified evidence.
if RUN_NLI and not embedding_results.empty:
    try:
        from transformers import pipeline
        classifier = pipeline('zero-shot-classification',
                              model='typeform/distilbert-base-uncased-mnli', device=-1)
        examples = []
        for _, rec in embedding_results.head(12).iterrows():
            service = str(rec['service'])
            labels = [f'the facility offers {service}',
                      f'the statement mentions {service} without confirming provision',
                      f'the facility does not offer {service}']
            out = classifier(rec['claim'], candidate_labels=labels, multi_label=False)
            examples.append({'service': rec['service'], 'claim': rec['claim'],
                             'model_label': out['labels'][0],
                             'model_score_unvalidated': round(out['scores'][0], 3)})
        display(pd.DataFrame(examples))
    except Exception as exc:
        print('Optional NLI pilot unavailable:', type(exc).__name__, str(exc)[:450])
else:
    print('NLI step skipped. Set RUN_NLI=True after embeddings succeed.')

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

service,claim,model_label,model_score_unvalidated
Dialysis,Provides dialysis treatment,the facility offers Dialysis,0.999
Dialysis,Provides dialysis treatment,the facility offers Dialysis,0.999
Dialysis,Provides dialysis treatment,the facility offers Dialysis,0.999
Dialysis,Provides dialysis services,the facility offers Dialysis,0.999
Dialysis,Offers haemodialysis and peritoneal dialysis services,the facility offers Dialysis,0.999
Dialysis,Kidney Care & Dialysis Center,the facility offers Dialysis,0.999
Dialysis,ICU with dialysis facilities,the facility offers Dialysis,0.999
Dialysis,Hemodialysis is performed in three units as a regular service,the facility offers Dialysis,0.968
NICU,Has a Neonatal Intensive Care Unit (NICU),the facility offers NICU,0.976
NICU,Neonatal Intensive Care Unit (NICU),the facility offers NICU,0.956


In [0]:
print('NEXT: Manually label 50-100 retrieved claims. Compare old regex vs the pilot.')
print('DO NOT change app.py, save scores, or filter live recommendations from this pilot.')

NEXT: Manually label 50-100 retrieved claims. Compare old regex vs the pilot.
DO NOT change app.py, save scores, or filter live recommendations from this pilot.


---

## Phase 4 — Method evaluation

Compare service-matching approaches using provisional labels.

*Source notebook: `care_finder_phase4_evaluation (1) (1).ipynb`*

**Before running this section:** It was tested separately. Restart the Python session or check existing variables if you are rerunning the analysis after an earlier stage.

# Care Finder India — Phase 4: Evidence Evaluation

This work was originally run as a separate Databricks notebook. Generates 84 unlabeled statements and compares methods **only after manual labeling**. Does not write to Unity Catalog or modify the app.

Install `%pip install sentence-transformers transformers` first if necessary; restart the Python environment. Set `EXPORT_PATH` to a writable Unity Catalog Volume path to download the label sheet.

**For reproducible scoring:** use the self-contained evaluation near the beginning, or start with the packaged `data/care_finder_phase4_review_labeled.csv`. The phase-generation code below samples from Databricks and is not needed to reproduce the saved metrics. Do not attach the original labels to a newly sampled set of statements.


In [0]:
# Care Finder India — Phase 4: Evaluation, not live ranking
# Run as a separate Databricks Python notebook.
# First run: create balanced review sample, export CSV, label manually.
# Second run: upload completed CSV to DBFS/Unity Catalog volume; evaluate predictions.

import re
import csv
from collections import Counter
from pathlib import Path

from pyspark.sql import functions as F

SOURCE_TABLE = "workspace.default.care_finder_facilities"
SEED = 615
SAMPLE_PER_SERVICE = 12
RUN_EMBEDDINGS = True
RUN_NLI = False  # Enable only after evaluation sample is labeled / models available.
# Databricks: set to a location you can download, e.g. /Volumes/<catalog>/<schema>/<volume>/phase4_labels.csv
EXPORT_PATH = "/tmp/care_finder_phase4_labels.csv"
# Set to the uploaded, manually labeled CSV on the second run, otherwise leave None.
LABELED_PATH = None

SERVICES = {
    "Dialysis": {
        "query": "The facility provides dialysis treatment for patients.",
        "pattern": r"\b(?:dialysis|ha?emodialysis|peritoneal dialysis)\b",
    },
    "NICU": {
        "query": "The facility has a neonatal intensive care unit for critically ill newborns.",
        "pattern": r"\b(?:nicu|neonatal intensive care|newborn intensive care)\b",
    },
    "MRI": {
        "query": "The facility provides magnetic resonance imaging scans to patients.",
        "pattern": r"\b(?:mri|magnetic resonance imaging)\b",
    },
    "Trauma Care": {
        "query": "The facility provides clinical trauma care for serious injuries.",
        "pattern": r"\b(?:trauma|polytrauma|accident and emergency)\b",
    },
    "Burn Care": {
        "query": "The facility offers treatment for patients with burn injuries.",
        "pattern": r"\b(?:burns?|burn ward|plastic surgery)\b",
    },
    "General Emergency": {
        "query": "The facility operates a general emergency department for urgent medical cases.",
        "pattern": r"\b(?:emergency|casualty|emergencies)\b",
    },
    "Emergency C-section": {
        "query": "The facility can perform emergency cesarean section surgery for urgent deliveries.",
        "pattern": r"\b(?:c[ -]?section|cesarean|caesarean|lscs|obstetric|delivery)\b",
    },
}

# Ground-truth labels describe the evidence *and attribution*, not real-world verification.
LABELS = ["direct", "supporting_only", "unrelated_or_contradicted", "misattributed"]


def cleaned(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def regex_baseline(service, claim):
    """Baseline is intentionally simple and fixed for fair comparison."""
    return bool(re.search(SERVICES[service]["pattern"], claim, re.I))


def emergency_scope(claim):
    text = cleaned(claim).lower()
    if not re.search(r"\bemergenc(?:y|ies)\b|\bcasualty\b", text):
        return "not_emergency"
    if re.search(r"\b(?:dental|dentistry|ophthalmic|ocular|eye)\b.{0,45}\bemergenc(?:y|ies)\b|\bemergenc(?:y|ies)\b.{0,45}\b(?:dental|dentistry|ophthalmic|ocular|eye)\b", text):
        return "specialty_only"
    if re.search(r"\b(?:general emergency|emergency department|accident and emergency|casualty department)\b", text):
        return "general_or_mixed"
    return "unspecified"


# ------------------------------------------------------------
# Step 1: Build a reproducible candidate bank.
# Deduplicate repeated evidence within a facility before sampling.
# Take multiple strata per service, so the sheet isn't all easy positives.
# ------------------------------------------------------------
source_facilities = spark.table(SOURCE_TABLE).select("unique_id", "name", "address_city", "evidence_list")
claims_df = (
    source_facilities
    .select(
        F.col("unique_id").alias("facility_id"),
        F.col("name").alias("facility"),
        F.col("address_city").alias("city"),
        F.explode_outer("evidence_list").alias("claim"),
    )
    .filter(F.col("claim").isNotNull() & (F.length(F.trim("claim")) >= 12))
    .dropDuplicates(["facility_id", "claim"])
)

# Collect a bounded, randomized set to the driver. Avoid collecting all claim rows.
# We use 500 candidates matching each service keyword and 500 random other claims.
# These are retrieval candidates, not assumed true or false labels.
sampled_rows = []
for service, conf in SERVICES.items():
    pat = conf["pattern"]
    matching = claims_df.filter(F.col("claim").rlike("(?i)" + pat))
    nonmatching = claims_df.filter(~F.col("claim").rlike("(?i)" + pat))
    # Limit per pool before collecting to avoid driver overload.
    match_rows = matching.orderBy(F.rand(SEED)).limit(180).collect()
    other_rows = nonmatching.orderBy(F.rand(SEED + 1)).limit(180).collect()
    for source_type, pool in (("keyword", match_rows), ("other", other_rows)):
        for row in pool:
            sampled_rows.append({
                "service": service,
                "facility_id": row["facility_id"],
                "facility": row["facility"],
                "city": row["city"],
                "claim": cleaned(row["claim"]),
                "pool": source_type,
            })

# ------------------------------------------------------------
# Step 2: Precompute semantic similarity on the bounded bank.
# An embedding score is *not* a claim's accuracy probability.
# ------------------------------------------------------------
embeddings_ready = False
if RUN_EMBEDDINGS:
    try:
        import numpy as np
        from sentence_transformers import SentenceTransformer
        embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        bank = list(dict.fromkeys(x["claim"] for x in sampled_rows))
        vectors = embed_model.encode(bank, normalize_embeddings=True, show_progress_bar=False)
        index = {claim: idx for idx, claim in enumerate(bank)}
        for service, conf in SERVICES.items():
            q = embed_model.encode([conf["query"]], normalize_embeddings=True)[0]
            for x in sampled_rows:
                if x["service"] == service:
                    x["similarity"] = float(np.dot(vectors[index[x["claim"]]], q))
        embeddings_ready = True
        print("Sentence embeddings computed for", len(bank), "unique claims")
    except Exception as exc:
        print("Embedding unavailable:", type(exc).__name__, str(exc))

# ------------------------------------------------------------
# Step 3: 12 per service: mix easy and hard examples.
# Select across facility IDs and alternate strata.
# Score-based retrieval includes false positives by design.
# ------------------------------------------------------------
review = []
for service in SERVICES:
    pool = [x for x in sampled_rows if x["service"] == service]
    key = lambda x: x.get("similarity", 1.0 if x["pool"] == "keyword" else 0.0)
    keyword = sorted([x for x in pool if x["pool"] == "keyword"], key=key, reverse=True)
    other = sorted([x for x in pool if x["pool"] == "other"], key=key, reverse=True)
    # Include high and medium retrieval scores, plus distractors.
    groups = [keyword[:4], other[:4], keyword[25:29], other[25:29]]
    selected = []
    seen_facilities = set()
    seen_claims = set()
    for group in groups + [keyword, other]:
        for item in group:
            key2 = (item["facility_id"], item["claim"])
            if key2 in seen_claims or item["facility_id"] in seen_facilities:
                continue
            selected.append(item)
            seen_facilities.add(item["facility_id"])
            seen_claims.add(key2)
            if len(selected) == SAMPLE_PER_SERVICE:
                break
        if len(selected) == SAMPLE_PER_SERVICE:
            break
    for i, item in enumerate(selected, 1):
        review.append({
            "review_id": f"{list(SERVICES).index(service)+1:02d}-{i:02d}",
            "service": service,
            "facility_id": str(item["facility_id"] or ""),
            "facility": str(item["facility"] or ""),
            "city": str(item["city"] or ""),
            "claim": item["claim"],
            "sample_pool": item["pool"],
            "regex_match": int(regex_baseline(service, item["claim"])),
            "embedding_similarity": round(item.get("similarity", -1.0), 4),
            "emergency_scope": emergency_scope(item["claim"]),
            "gold_label": "",  # MUST be reviewed manually.
            "review_notes": "",
        })

assert len(review) == len(SERVICES) * SAMPLE_PER_SERVICE, ("Unexpected sample size", len(review))
print("Created review sheet:", len(review), "claims from", len(set(x["facility_id"] for x in review)), "facilities")
print("Services:", dict(Counter(x["service"] for x in review)))

# ------------------------------------------------------------
# Step 4 (OPTIONAL): NLI predictions, without treating scores as proof.
# This experiment uses a zero-shot classifier; its label scores
# are not calibrated probabilities of service availability.
# ------------------------------------------------------------
if RUN_NLI:
    try:
        from transformers import pipeline
        nli = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=-1)
        for item in review:
            labels = [
                "directly states this facility provides " + item["service"],
                "mentions related care without establishing " + item["service"],
                "does not establish " + item["service"],
            ]
            output = nli(item["claim"], candidate_labels=labels, multi_label=False)
            item["nli_label"] = output["labels"][0]
            item["nli_score_unvalidated"] = round(float(output["scores"][0]), 4)
        print("NLI predictions generated (not validated).")
    except Exception as exc:
        print("NLI unavailable:", type(exc).__name__, str(exc))

# Export after optional predictions so the review CSV includes NLI columns if enabled.
cols = list(dict.fromkeys(k for item in review for k in item))
Path(EXPORT_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(EXPORT_PATH, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=cols)
    writer.writeheader()
    writer.writerows(review)
print("Review CSV saved to:", EXPORT_PATH)
print("Download it using a Databricks-accessible Unity Catalog volume or workspace file location.")
display(spark.createDataFrame(review).select("review_id", "service", "facility", "claim", "regex_match", "embedding_similarity", "gold_label"))

# ------------------------------------------------------------
# Step 5: Evaluate only after human review.
# Positive = a directly attributable service claim.
# 'misattributed' is explicitly negative even if the text is relevant.
# ------------------------------------------------------------
if LABELED_PATH:
    with open(LABELED_PATH, encoding="utf-8-sig", newline="") as f:
        labeled = list(csv.DictReader(f))
    if len(labeled) != 84 or any(row.get("gold_label") not in LABELS for row in labeled):
        raise ValueError("Expected 84 rows, each labeled with exactly one of: " + ", ".join(LABELS))

    def metrics(y_true, y_pred):
        tp = sum(t and p for t, p in zip(y_true, y_pred))
        fp = sum((not t) and p for t, p in zip(y_true, y_pred))
        fn = sum(t and (not p) for t, p in zip(y_true, y_pred))
        tn = sum((not t) and (not p) for t, p in zip(y_true, y_pred))
        return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
                "precision": round(tp / (tp + fp), 3) if tp + fp else None,
                "recall": round(tp / (tp + fn), 3) if tp + fn else None,
                "false_positive_rate": round(fp / (fp + tn), 3) if fp + tn else None}

    true = [r["gold_label"] == "direct" for r in labeled]
    regex_pred = [str(r["regex_match"]) == "1" for r in labeled]
    print("Regex baseline:", metrics(true, regex_pred))
    # Similarity thresholds are exploratory; tune on a different set,
    # not on these 84 labeled examples before reporting test results.
    if all(float(r.get("embedding_similarity") or -1) >= 0 for r in labeled):
        embed_pred = [float(r["embedding_similarity"]) >= 0.65 for r in labeled]
        print("Embedding >= 0.65 (illustrative, untuned):", metrics(true, embed_pred))
    if all(r.get("nli_label") for r in labeled):
        nli_pred = [str(r["nli_label"]).startswith("directly states") for r in labeled]
        print("Zero-shot NLI label selection (unvalidated):", metrics(true, nli_pred))
    print("Review label distribution:", dict(Counter(r["gold_label"] for r in labeled)))
else:
    print("Evaluation is intentionally pending human labels. Fill gold_label and set LABELED_PATH on the second run.")


Embedding unavailable: ModuleNotFoundError No module named 'sentence_transformers'
Created review sheet: 84 claims from 83 facilities
Services: {'Dialysis': 12, 'NICU': 12, 'MRI': 12, 'Trauma Care': 12, 'Burn Care': 12, 'General Emergency': 12, 'Emergency C-section': 12}
Review CSV saved to: /tmp/care_finder_phase4_labels.csv
Download it using a Databricks-accessible Unity Catalog volume or workspace file location.


review_id,service,facility,claim,regex_match,embedding_similarity,gold_label
01-01,Dialysis,Karan Hospital,In-house dialysis facility,1,-1.0,
01-02,Dialysis,Sharanya Multi Speciality Hospital,On-site dialysis unit,1,-1.0,
01-03,Dialysis,"Dr M. L. Dhawale Memorial Trust Hospital, Palghar","Palghar Rotary Dialysis Centre with eight dialysis machines performing over 10,000 dialysis cycles",1,-1.0,
01-04,Dialysis,Garg Child Hospital,Haemodialysis - Peritoneal Dialysis,1,-1.0,
01-05,Dialysis,"Batra Heart & Multispecialty Hospital, Block F, Spring Field Colony, Sector 31, Faridabad, Haryana 121003","Intravascular imaging and physiology capabilities (IVUS, OCT, FFR)",0,-1.0,
01-06,Dialysis,Assa Diagnostics & Nursing Home,Cares for East Siang district and surrounding districts in Arunachal Pradesh,0,-1.0,
01-07,Dialysis,Aster RameshOngole,Performs orthognathic surgery and distraction osteogenesis to correct jaw deformities,0,-1.0,
01-08,Dialysis,J J Eye Hospital,The Department of Ophthalmology at J.J. Hospital is described as a five-star government hospital with full instruments and equipment and specialized staff.,0,-1.0,
01-09,Dialysis,Charak Hospital And Research Centre,Nephrology with 24/7 Advance Dialysis Unit,1,-1.0,
01-10,Dialysis,Sai Super-Speciality Hospital,In-house dialysis services,1,-1.0,


Evaluation is intentionally pending human labels. Fill gold_label and set LABELED_PATH on the second run.


In [0]:

import pandas as pd

labels_path = str(DATA_DIR / "care_finder_phase4_provisional_labels.csv")

labels = pd.read_csv(labels_path)

print("Labels loaded:", len(labels))
display(labels.head(10))


Labels loaded: 84


review_id,gold_label,review_note
01-01,direct,null
01-02,direct,null
01-03,direct,Dialysis center and active treatment cycles explicitly described; check that the centre belongs to the named facility.
01-04,direct,"Explicit dialysis modalities listed, but current service availability is not verified."
01-05,unrelated_or_contradicted,null
01-06,unrelated_or_contradicted,null
01-07,unrelated_or_contradicted,null
01-08,unrelated_or_contradicted,null
01-09,direct,null
01-10,direct,null


In [0]:
# Use the reviewed statements from the original Phase 4 evaluation.
# Regenerating a random sample would change which statements the labels refer to.
review = pd.read_csv(DATA_DIR / "care_finder_phase4_review_labeled.csv")

print("Reviewed statements:", len(review))
print("Missing labels:", review["gold_label"].isna().sum())
display(review[["review_id", "service", "facility", "claim", "regex_match", "gold_label"]].head(10))

Total statements: 84
Missing labels: 0


review_id,service,facility,claim,regex_match,gold_label
01-01,Dialysis,Karan Hospital,In-house dialysis facility,1,direct
01-02,Dialysis,Sharanya Multi Speciality Hospital,On-site dialysis unit,1,direct
01-03,Dialysis,"Dr M. L. Dhawale Memorial Trust Hospital, Palghar","Palghar Rotary Dialysis Centre with eight dialysis machines performing over 10,000 dialysis cycles",1,direct
01-04,Dialysis,Garg Child Hospital,Haemodialysis - Peritoneal Dialysis,1,direct
01-05,Dialysis,"Batra Heart & Multispecialty Hospital, Block F, Spring Field Colony, Sector 31, Faridabad, Haryana 121003","Intravascular imaging and physiology capabilities (IVUS, OCT, FFR)",0,unrelated_or_contradicted
01-06,Dialysis,Assa Diagnostics & Nursing Home,Cares for East Siang district and surrounding districts in Arunachal Pradesh,0,unrelated_or_contradicted
01-07,Dialysis,Aster RameshOngole,Performs orthognathic surgery and distraction osteogenesis to correct jaw deformities,0,unrelated_or_contradicted
01-08,Dialysis,J J Eye Hospital,The Department of Ophthalmology at J.J. Hospital is described as a five-star government hospital with full instruments and equipment and specialized staff.,0,unrelated_or_contradicted
01-09,Dialysis,Charak Hospital And Research Centre,Nephrology with 24/7 Advance Dialysis Unit,1,direct
01-10,Dialysis,Sai Super-Speciality Hospital,In-house dialysis services,1,direct


In [0]:

# The reviewed sample is already included in data/. Keep this export for standalone reruns.
LABELED_PATH = str(DATA_DIR / "care_finder_phase4_review_labeled.csv")

review.to_csv(LABELED_PATH, index=False)

print("Saved to:", LABELED_PATH)
print("Total statements:", len(review))
print("Label counts:")
print(review["gold_label"].value_counts())


Saved to: /Volumes/workspace/default/care_finder_files/care_finder_phase4_review_labeled.csv
Total statements: 84
Label counts:
gold_label
direct                       31
unrelated_or_contradicted    31
supporting_only              21
misattributed                 1
Name: count, dtype: int64


In [0]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Direct evidence = 1
# All other labels = 0
y_true = (review["gold_label"] == "direct").astype(int)

# Existing regex predictions
y_pred = review["regex_match"].astype(int)

# Calculate performance
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

print("REGEX BASELINE RESULTS")
print("----------------------")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 Score:  {f1:.3f}")
print()
print("True Positives:", tp)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Negatives:", tn)


REGEX BASELINE RESULTS
----------------------
Precision: 0.564
Recall:    1.000
F1 Score:  0.721

True Positives: 31
False Positives: 24
False Negatives: 0
True Negatives: 29


In [0]:

# Find claims regex accepted but our provisional labels did not
false_positives = review[
    (review["regex_match"] == 1) &
    (review["gold_label"] != "direct")
].copy()

print("Total false positives:", len(false_positives))

print("\nFalse positives by service:")
print(false_positives["service"].value_counts())

display(
    false_positives[
        [
            "review_id",
            "service",
            "facility",
            "claim",
            "gold_label"
        ]
    ]
)


Total false positives: 24

False positives by service:
service
Emergency C-section    8
Burn Care              7
General Emergency      4
Trauma Care            3
MRI                    2
Name: count, dtype: int64


review_id,service,facility,claim,gold_label
03-02,MRI,Yadgire Super Specialty Hospital,"District Hospital, Amravati has 24x7 CT and MRI imaging services under NHM PPP.",misattributed
03-12,MRI,"Vijaya Diagnostic Centre - Main Road Gopalapatnam, Visakhapatnam",3 Tesla MRI (3T MRI) imaging is available in the Vijaya Diagnostics network,supporting_only
04-01,Trauma Care,Vijaya Health Centre,"Offers a wide range of specialties including Anesthesiology, Cardiology & Cardiothoracic Surgery, Emergency Medicine, ENT, Gastroenterology, General Medicine, General Surgery, Hematology, Hepatology, Infectious Diseases, Intensive Care Medicine, Nephrology, Neurology & Neuro Surgery, Obstetrics & Gynaecology, Oncology, Ophthalmology, Paediatrics & Neonatology, Plastic Surgery, Psychiatry, Pulmonology, Trauma & Orthopaedics, Urology Renal Transplant, Vascular & Endovascular Surgery",supporting_only
04-03,Trauma Care,Anand Hospital,Orthopaedics & Trauma,supporting_only
04-11,Trauma Care,Himalaya Hospital,Orthopedics & Trauma,supporting_only
05-01,Burn Care,Jawahar Memorial Hospital,Facial plastic surgery,supporting_only
05-03,Burn Care,VJ's Transgender Clinic,Offers plastic surgery and gender-affirming surgical procedures,supporting_only
05-04,Burn Care,Aarka Hospitals,"Provides comprehensive healthcare services across multiple medical disciplines, including neurology and plastic surgery.",supporting_only
05-09,Burn Care,Surya Hospital,Plastic surgery procedures,supporting_only
05-10,Burn Care,"Doctor Today IVF and Multispeciality Hospital, Faridabad",Specialities include Plastic Surgery,supporting_only


In [0]:

# Check if embedding scores were computed
print("Total statements:", len(review))

print(
    "Missing embedding scores:",
    (review["embedding_similarity"] == -1).sum()
)

print(
    "Embedding score range:",
    review["embedding_similarity"].min(),
    "to",
    review["embedding_similarity"].max()
)


Total statements: 84
Missing embedding scores: 84
Embedding score range: -1.0 to -1.0


In [0]:

from sentence_transformers import SentenceTransformer
import numpy as np

# Load the same embedding model used in Phase 3
model = SentenceTransformer("all-MiniLM-L6-v2")

# Descriptions of the services we want to find
service_queries = {
    "Dialysis": "Hospital provides dialysis or hemodialysis treatment",
    "NICU": "Hospital has a neonatal intensive care unit for newborns",
    "MRI": "Hospital provides MRI scanning and magnetic resonance imaging",
    "Trauma Care": "Hospital provides trauma treatment and emergency injury care",
    "Burn Care": "Hospital provides specialized treatment for burn injuries",
    "General Emergency": "Hospital operates an emergency department for urgent medical care",
    "Emergency C-section": "Hospital performs emergency cesarean deliveries"
}

# Encode the service queries
query_embeddings = model.encode(
    list(service_queries.values()),
    normalize_embeddings=True
)

query_lookup = {
    service: query_embeddings[i]
    for i, service in enumerate(service_queries)
}

# Encode all 84 evidence statements
claim_embeddings = model.encode(
    review["claim"].fillna("").tolist(),
    normalize_embeddings=True
)

# Compare each statement to its own requested service
scores = []

for i, row in review.reset_index(drop=True).iterrows():
    service = row["service"]
    score = float(
        np.dot(
            claim_embeddings[i],
            query_lookup[service]
        )
    )
    scores.append(score)

review["embedding_similarity"] = scores

print("Statements scored:", len(review))
print(
    "Embedding score range:",
    round(review["embedding_similarity"].min(), 3),
    "to",
    round(review["embedding_similarity"].max(), 3)
)

display(
    review[
        [
            "review_id",
            "service",
            "claim",
            "embedding_similarity",
            "gold_label"
        ]
    ].head(15)
)


/local_disk0/.ephemeral_nfs/envs/pythonEnv-da66ffef-d7dc-4354-af14-4a755e754567/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Statements scored: 84
Embedding score range: 0.003 to 0.87


review_id,service,claim,embedding_similarity,gold_label
01-01,Dialysis,In-house dialysis facility,0.7116078734397888,direct
01-02,Dialysis,On-site dialysis unit,0.6589633822441101,direct
01-03,Dialysis,"Palghar Rotary Dialysis Centre with eight dialysis machines performing over 10,000 dialysis cycles",0.5283347368240356,direct
01-04,Dialysis,Haemodialysis - Peritoneal Dialysis,0.6445626020431519,direct
01-05,Dialysis,"Intravascular imaging and physiology capabilities (IVUS, OCT, FFR)",0.1772761344909668,unrelated_or_contradicted
01-06,Dialysis,Cares for East Siang district and surrounding districts in Arunachal Pradesh,0.08928621560335159,unrelated_or_contradicted
01-07,Dialysis,Performs orthognathic surgery and distraction osteogenesis to correct jaw deformities,0.029561959207057953,unrelated_or_contradicted
01-08,Dialysis,The Department of Ophthalmology at J.J. Hospital is described as a five-star government hospital with full instruments and equipment and specialized staff.,0.390634685754776,unrelated_or_contradicted
01-09,Dialysis,Nephrology with 24/7 Advance Dialysis Unit,0.6041927337646484,direct
01-10,Dialysis,In-house dialysis services,0.6949131488800049,direct


In [0]:

from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

# Direct evidence is our positive class
y_true = (review["gold_label"] == "direct").astype(int)

# Test several embedding similarity thresholds
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

results = []

for threshold in thresholds:

    # Accept a claim if its similarity meets the threshold
    y_pred = (
        review["embedding_similarity"] >= threshold
    ).astype(int)

    results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "false_positives": int(
            ((y_pred == 1) & (y_true == 0)).sum()
        ),
        "false_negatives": int(
            ((y_pred == 0) & (y_true == 1)).sum()
        )
    })

results_df = pd.DataFrame(results)

print("EMBEDDING PERFORMANCE BY THRESHOLD")
display(results_df.round(3))


EMBEDDING PERFORMANCE BY THRESHOLD


threshold,precision,recall,f1,false_positives,false_negatives
0.3,0.6,0.968,0.741,20,1
0.4,0.667,0.903,0.767,14,3
0.5,0.793,0.742,0.767,6,8
0.6,0.833,0.484,0.612,3,16
0.7,0.857,0.194,0.316,1,25
0.8,1.0,0.032,0.062,0,30


In [0]:

# Predictions at similarity threshold 0.50
review["embedding_pred"] = (
    review["embedding_similarity"] >= 0.50
).astype(int)

# Claims embeddings accepted incorrectly
embedding_fp = review[
    (review["embedding_pred"] == 1) &
    (review["gold_label"] != "direct")
].copy()

# Direct claims embeddings missed
embedding_fn = review[
    (review["embedding_pred"] == 0) &
    (review["gold_label"] == "direct")
].copy()

print("EMBEDDING FALSE POSITIVES:", len(embedding_fp))
print("EMBEDDING FALSE NEGATIVES:", len(embedding_fn))

print("\nFALSE POSITIVES:")
display(
    embedding_fp[
        [
            "review_id",
            "service",
            "facility",
            "claim",
            "embedding_similarity",
            "gold_label"
        ]
    ]
)

print("\nFALSE NEGATIVES:")
display(
    embedding_fn[
        [
            "review_id",
            "service",
            "facility",
            "claim",
            "embedding_similarity",
            "gold_label"
        ]
    ]
)


EMBEDDING FALSE POSITIVES: 6
EMBEDDING FALSE NEGATIVES: 8

FALSE POSITIVES:


review_id,service,facility,claim,embedding_similarity,gold_label
03-02,MRI,Yadgire Super Specialty Hospital,"District Hospital, Amravati has 24x7 CT and MRI imaging services under NHM PPP.",0.5926454663276672,misattributed
04-03,Trauma Care,Anand Hospital,Orthopaedics & Trauma,0.5891148447990417,supporting_only
04-11,Trauma Care,Himalaya Hospital,Orthopedics & Trauma,0.6006695032119751,supporting_only
07-01,Emergency C-section,"Sri Hari Sakthivel Hospital , Erode",Provides obstetric/antenatal care,0.5562479496002197,supporting_only
07-04,Emergency C-section,Rainbow Hospital,Pregnancy and delivery care is provided,0.6126322150230408,supporting_only
07-11,Emergency C-section,Dolphin Hospital,Performs cesarean sections,0.7320392727851868,supporting_only



FALSE NEGATIVES:


review_id,service,facility,claim,embedding_similarity,gold_label
02-01,NICU,"Asha Nursing Home – Rampurhat, Birbhum",ICU and NICU facilities,0.47401776909828186,direct
02-03,NICU,Deshmukh Hospital & Research Centre,Level III NICU on site,0.26232433319091797,direct
02-04,NICU,Prem Hospital Super Speciality & Trauma Centre,NICU and PICU facilities,0.3453136086463928,direct
02-10,NICU,Shifaa Hospital,Operates ICU and NICU units,0.45751339197158813,direct
02-12,NICU,Pulse Hospital,ICU/HDU/NICU/PICU units,0.323305606842041,direct
03-04,MRI,Jss Hospital,Provides magnetic resonance imaging (MRI) using Ingenia 3T,0.4438782334327698,direct
03-11,MRI,"Megavision Diagnostic Centres, Pimpri",1.5 Tesla MRI,0.4322115480899811,direct
05-02,Burn Care,Nemcare SuperSpeciality Hospital,Operates 5 operation theatres including a dedicated burn OT,0.4413232207298279,direct


In [0]:

from transformers import pipeline
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
import pandas as pd

# Load a zero-shot NLI model
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1
)

# A specific hypothesis for each service
hypotheses = {
    "Dialysis": "This facility provides dialysis treatment.",
    "NICU": "This facility operates a neonatal intensive care unit.",
    "MRI": "This facility provides MRI scans.",
    "Trauma Care": "This facility provides trauma care.",
    "Burn Care": "This facility provides specialized burn treatment.",
    "General Emergency": "This facility operates a general emergency department.",
    "Emergency C-section": "This facility performs emergency cesarean sections."
}

# Compare a positive and negative hypothesis
nli_scores = []

for _, row in review.iterrows():

    service = row["service"]

    labels = [
        hypotheses[service],
        "The statement does not establish that this facility provides the requested service."
    ]

    result = classifier(
        row["claim"],
        candidate_labels=labels,
        multi_label=False,
        hypothesis_template="{}"
    )

    # Obtain score for the positive hypothesis
    score_map = dict(zip(result["labels"], result["scores"]))
    positive_score = score_map[labels[0]]

    nli_scores.append(positive_score)

review["nli_score"] = nli_scores

# Initial decision threshold (not calibrated)
review["nli_pred"] = (
    review["nli_score"] >= 0.5
).astype(int)

y_true = (
    review["gold_label"] == "direct"
).astype(int)

y_pred = review["nli_pred"]

precision = precision_score(
    y_true, y_pred, zero_division=0
)

recall = recall_score(
    y_true, y_pred, zero_division=0
)

f1 = f1_score(
    y_true, y_pred, zero_division=0
)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

print("NLI EVALUATION RESULTS")
print("----------------------")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 Score:  {f1:.3f}")
print()
print("True Positives:", tp)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Negatives:", tn)

display(
    review[
        [
            "review_id",
            "service",
            "claim",
            "gold_label",
            "nli_score",
            "nli_pred"
        ]
    ]
)


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

NLI EVALUATION RESULTS
----------------------
Precision: 0.705
Recall:    1.000
F1 Score:  0.827

True Positives: 31
False Positives: 13
False Negatives: 0
True Negatives: 40


review_id,service,claim,gold_label,nli_score,nli_pred
01-01,Dialysis,In-house dialysis facility,direct,0.9941635727882385,1
01-02,Dialysis,On-site dialysis unit,direct,0.9957287311553955,1
01-03,Dialysis,"Palghar Rotary Dialysis Centre with eight dialysis machines performing over 10,000 dialysis cycles",direct,0.9982571601867676,1
01-04,Dialysis,Haemodialysis - Peritoneal Dialysis,direct,0.983258843421936,1
01-05,Dialysis,"Intravascular imaging and physiology capabilities (IVUS, OCT, FFR)",unrelated_or_contradicted,0.25430598855018616,0
01-06,Dialysis,Cares for East Siang district and surrounding districts in Arunachal Pradesh,unrelated_or_contradicted,0.3978273570537567,0
01-07,Dialysis,Performs orthognathic surgery and distraction osteogenesis to correct jaw deformities,unrelated_or_contradicted,0.08942489326000214,0
01-08,Dialysis,The Department of Ophthalmology at J.J. Hospital is described as a five-star government hospital with full instruments and equipment and specialized staff.,unrelated_or_contradicted,0.5064031481742859,1
01-09,Dialysis,Nephrology with 24/7 Advance Dialysis Unit,direct,0.9964551329612732,1
01-10,Dialysis,In-house dialysis services,direct,0.9867203235626221,1


In [0]:

# Find NLI predictions that aren't labeled direct
nli_fp = review[
    (review["nli_pred"] == 1) &
    (review["gold_label"] != "direct")
].copy()

print("Total NLI false positives:", len(nli_fp))

print("\nFalse positives by service:")
print(nli_fp["service"].value_counts())

display(
    nli_fp[
        [
            "review_id",
            "service",
            "facility",
            "claim",
            "gold_label",
            "nli_score"
        ]
    ]
)


Total NLI false positives: 13

False positives by service:
service
MRI                    3
Trauma Care            3
General Emergency      3
Emergency C-section    2
Dialysis               1
Burn Care              1
Name: count, dtype: int64


review_id,service,facility,claim,gold_label,nli_score
01-08,Dialysis,J J Eye Hospital,The Department of Ophthalmology at J.J. Hospital is described as a five-star government hospital with full instruments and equipment and specialized staff.,unrelated_or_contradicted,0.5064031481742859
03-02,MRI,Yadgire Super Specialty Hospital,"District Hospital, Amravati has 24x7 CT and MRI imaging services under NHM PPP.",misattributed,0.9947497844696045
03-07,MRI,Modi Hospital,"Located in Mulund, Mumbai, Maharashtra",unrelated_or_contradicted,0.5483870506286621
03-12,MRI,"Vijaya Diagnostic Centre - Main Road Gopalapatnam, Visakhapatnam",3 Tesla MRI (3T MRI) imaging is available in the Vijaya Diagnostics network,supporting_only,0.9934589862823486
04-01,Trauma Care,Vijaya Health Centre,"Offers a wide range of specialties including Anesthesiology, Cardiology & Cardiothoracic Surgery, Emergency Medicine, ENT, Gastroenterology, General Medicine, General Surgery, Hematology, Hepatology, Infectious Diseases, Intensive Care Medicine, Nephrology, Neurology & Neuro Surgery, Obstetrics & Gynaecology, Oncology, Ophthalmology, Paediatrics & Neonatology, Plastic Surgery, Psychiatry, Pulmonology, Trauma & Orthopaedics, Urology Renal Transplant, Vascular & Endovascular Surgery",supporting_only,0.9971433877944946
04-03,Trauma Care,Anand Hospital,Orthopaedics & Trauma,supporting_only,0.9923834204673767
04-11,Trauma Care,Himalaya Hospital,Orthopedics & Trauma,supporting_only,0.991912841796875
05-07,Burn Care,Lokhandwala General Hospital,Listed as a top hospital in Shardanagar on Practo,unrelated_or_contradicted,0.6640523672103882
06-03,General Emergency,Himalayan Hospital,"Offers outpatient services across multiple specialties including Cardiology, Cardiothoracic Surgery, Clinical Immunology, Dental, Dermatology, Emergency Medicine, Endocrinology, ENT, Gastroenterology, Gastrosurgery, General Medicine, General Surgery, Immunohematology and Blood Transfusion, Medical Oncology, Neonatology, Nephrology, Neurology, Neurosurgery, Obstetrics and Gynaecology, Ophthalmology, Orthopaedics, Pediatric Surgery, Pediatrics, Physical Medicine and Rehabilitation, Plastic Surgery, Psychiatry, Pulmonary Medicine, Radiation Oncology, Surgical Oncology, Urology",supporting_only,0.7458509802818298
06-10,General Emergency,SK Hospital,Offers 24-hour emergency booking services,unrelated_or_contradicted,0.8649216294288635


In [0]:

# Print all 13 false positives without truncating the text
for _, row in nli_fp.iterrows():

    print("=" * 65)
    print("Review ID:", row["review_id"])
    print("Service:", row["service"])
    print("Facility:", row["facility"])
    print("Claim:", row["claim"])
    print("Correct label:", row["gold_label"])
    print("NLI score:", round(row["nli_score"], 3))

print("=" * 65)
print("Total false positives:", len(nli_fp))


Review ID: 01-08
Service: Dialysis
Facility: J J Eye Hospital
Claim: The Department of Ophthalmology at J.J. Hospital is described as a five-star government hospital with full instruments and equipment and specialized staff.
Correct label: unrelated_or_contradicted
NLI score: 0.506
Review ID: 03-02
Service: MRI
Facility: Yadgire Super Specialty Hospital
Claim: District Hospital, Amravati has 24x7 CT and MRI imaging services under NHM PPP.
Correct label: misattributed
NLI score: 0.995
Review ID: 03-07
Service: MRI
Facility: Modi Hospital
Claim: Located in Mulund, Mumbai, Maharashtra
Correct label: unrelated_or_contradicted
NLI score: 0.548
Review ID: 03-12
Service: MRI
Facility: Vijaya Diagnostic Centre - Main Road Gopalapatnam, Visakhapatnam
Claim: 3 Tesla MRI (3T MRI) imaging is available in the Vijaya Diagnostics network
Correct label: supporting_only
NLI score: 0.993
Review ID: 04-01
Service: Trauma Care
Facility: Vijaya Health Centre
Claim: Offers a wide range of specialties includ

In [0]:

import re
import pandas as pd

# These rules reject claims that do not sufficiently
# establish the requested service.
# This rule set was tested here but is not used directly in the app.

def safety_check(row):

    service = str(row["service"])
    claim = str(row["claim"]).lower()
    facility = str(row["facility"]).lower()

    # Reject generic directory and promotional listings.
    if re.search(
        r"listed as a top hospital|five.star government hospital",
        claim
    ):
        return "generic hospital listing"

    # Reject clear references to a different facility.
    if (
        "district hospital, amravati" in claim
        and "district hospital" not in facility
    ):
        return "service attributed to another hospital"

    # MRI offered across a network does not establish
    # availability at this particular branch.
    if service == "MRI":
        if re.search(r"\bnetwork\b", claim):
            return "network-wide claim, not branch-specific"

    # A specialty listing alone does not confirm a service.
    # to establish trauma treatment capacity.
    if service == "Trauma Care":
        if re.fullmatch(
            r"\s*orthop[a-z]*\s*&\s*trauma\s*",
            claim
        ):
            return "specialty name without service details"

        if (
            "trauma & orthopaedics" in claim
            and "specialties including" in claim
        ):
            return "general specialty list"

    # Administrative emergency booking is not
    # the same as emergency medical care.
    if service == "General Emergency":

        if "emergency booking" in claim:
            return "booking rather than clinical emergency care"

        if "mass casualty and disaster response" in claim:
            return "emergency preparedness, not established ED"

        if (
            "outpatient services across multiple specialties" in claim
            and "emergency medicine" in claim
        ):
            return "specialty listing does not establish ED"

    # Emergency C-section requires evidence of BOTH
    # cesarean delivery and emergency capability.
    if service == "Emergency C-section":

        has_csection = bool(re.search(
            r"c[ -]?section|cesarean|caesarean|"
            r"\blscs\b",
            claim
        ))

        has_emergency = bool(re.search(
            r"emergency|urgent|24[/x -]?7|"
            r"24.hour",
            claim
        ))

        if not (has_csection and has_emergency):
            return "emergency C-section not established"

    return ""


# Apply safety checks to the NLI predictions.
review["safety_reason"] = review.apply(
    safety_check,
    axis=1
)

review["hybrid_pred"] = (
    (review["nli_pred"] == 1) &
    (review["safety_reason"] == "")
).astype(int)


# Compare the methods against provisional labels.
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_true = (review["gold_label"] == "direct").astype(int)

methods = {
    "Regex": review["regex_match"].astype(int),
    "Embeddings (0.50)": (
        review["embedding_similarity"] >= 0.50
    ).astype(int),
    "NLI": review["nli_pred"].astype(int),
    "Hybrid pilot": review["hybrid_pred"].astype(int)
}

results = []

for name, pred in methods.items():

    tn, fp, fn, tp = confusion_matrix(
        y_true, pred, labels=[0, 1]
    ).ravel()

    results.append({
        "method": name,
        "precision": round(
            precision_score(y_true, pred, zero_division=0), 3
        ),
        "recall": round(
            recall_score(y_true, pred, zero_division=0), 3
        ),
        "f1": round(
            f1_score(y_true, pred, zero_division=0), 3
        ),
        "false_positives": int(fp),
        "false_negatives": int(fn)
    })

print("MODEL COMPARISON")
display(pd.DataFrame(results))

# Inspect all predictions changed by the safety checks.
changed = review[
    review["nli_pred"] != review["hybrid_pred"]
]

print("Predictions changed:", len(changed))

display(
    changed[
        [
            "review_id",
            "service",
            "claim",
            "gold_label",
            "nli_score",
            "safety_reason"
        ]
    ]
)


MODEL COMPARISON


method,precision,recall,f1,false_positives,false_negatives
Regex,0.564,1.0,0.721,24,0
Embeddings (0.50),0.793,0.742,0.767,6,8
NLI,0.705,1.0,0.827,13,0
Hybrid pilot,0.969,1.0,0.984,1,0


Predictions changed: 12


review_id,service,claim,gold_label,nli_score,safety_reason
01-08,Dialysis,The Department of Ophthalmology at J.J. Hospital is described as a five-star government hospital with full instruments and equipment and specialized staff.,unrelated_or_contradicted,0.5064031481742859,generic hospital listing
03-02,MRI,"District Hospital, Amravati has 24x7 CT and MRI imaging services under NHM PPP.",misattributed,0.9947497844696045,service attributed to another hospital
03-12,MRI,3 Tesla MRI (3T MRI) imaging is available in the Vijaya Diagnostics network,supporting_only,0.9934589862823486,"network-wide claim, not branch-specific"
04-01,Trauma Care,"Offers a wide range of specialties including Anesthesiology, Cardiology & Cardiothoracic Surgery, Emergency Medicine, ENT, Gastroenterology, General Medicine, General Surgery, Hematology, Hepatology, Infectious Diseases, Intensive Care Medicine, Nephrology, Neurology & Neuro Surgery, Obstetrics & Gynaecology, Oncology, Ophthalmology, Paediatrics & Neonatology, Plastic Surgery, Psychiatry, Pulmonology, Trauma & Orthopaedics, Urology Renal Transplant, Vascular & Endovascular Surgery",supporting_only,0.9971433877944946,general specialty list
04-03,Trauma Care,Orthopaedics & Trauma,supporting_only,0.9923834204673767,specialty name without service details
04-11,Trauma Care,Orthopedics & Trauma,supporting_only,0.991912841796875,specialty name without service details
05-07,Burn Care,Listed as a top hospital in Shardanagar on Practo,unrelated_or_contradicted,0.6640523672103882,generic hospital listing
06-03,General Emergency,"Offers outpatient services across multiple specialties including Cardiology, Cardiothoracic Surgery, Clinical Immunology, Dental, Dermatology, Emergency Medicine, Endocrinology, ENT, Gastroenterology, Gastrosurgery, General Medicine, General Surgery, Immunohematology and Blood Transfusion, Medical Oncology, Neonatology, Nephrology, Neurology, Neurosurgery, Obstetrics and Gynaecology, Ophthalmology, Orthopaedics, Pediatric Surgery, Pediatrics, Physical Medicine and Rehabilitation, Plastic Surgery, Psychiatry, Pulmonary Medicine, Radiation Oncology, Surgical Oncology, Urology",supporting_only,0.7458509802818298,specialty listing does not establish ED
06-10,General Emergency,Offers 24-hour emergency booking services,unrelated_or_contradicted,0.8649216294288635,booking rather than clinical emergency care
06-11,General Emergency,Mass casualty and disaster response capability,supporting_only,0.5445069074630737,"emergency preparedness, not established ED"


In [0]:

# Find any incorrect claims still accepted by our hybrid
remaining_fp = review[
    (review["hybrid_pred"] == 1) &
    (review["gold_label"] != "direct")
].copy()

print("Remaining hybrid false positives:", len(remaining_fp))

display(
    remaining_fp[
        [
            "review_id",
            "service",
            "facility",
            "claim",
            "gold_label",
            "nli_score",
            "safety_reason"
        ]
    ]
)


Remaining hybrid false positives: 1


review_id,service,facility,claim,gold_label,nli_score,safety_reason
03-07,MRI,Modi Hospital,"Located in Mulund, Mumbai, Maharashtra",unrelated_or_contradicted,0.5483870506286621,


---

## Phase 5 — Additional validation

Evaluate the earlier methods on a separate set of statements.

*Source notebook: `New Notebook 2026-11-22 07_28_33.ipynb`*

**Before running this section:** It was tested separately. Restart the Python session or check existing variables if you are rerunning the analysis after an earlier stage.

In [0]:

import pandas as pd
from pyspark.sql import functions as F

# Load the hospital records.
SOURCE_TABLE = "workspace.default.care_finder_facilities"

# Phase 4 evaluation dataset
OLD_PATH = str(DATA_DIR / "care_finder_phase4_review_labeled.csv")

# Where we'll save the new validation data
NEW_PATH = str(DATA_DIR / "care_finder_phase5_validation.csv")

# Read the previous evaluation set
old = pd.read_csv(OLD_PATH)

# Exclude hospitals used in Phase 4
old_names = set(
    old["facility"].dropna().str.strip().str.lower()
)

# Load original hospital records
df = spark.table(SOURCE_TABLE)

# Extract individual evidence statements
claims = (
    df.select(
        "unique_id",
        F.col("name").alias("facility"),
        F.explode_outer("evidence_list").alias("claim")
    )
    .filter(F.col("claim").isNotNull())
    .filter(F.length(F.trim("claim")) > 10)
    .dropDuplicates(["unique_id", "claim"])
)

# Remove all facilities that appeared in Phase 4
old_names_df = spark.createDataFrame(
    [(name,) for name in old_names],
    ["old_facility"]
)

claims = (
    claims.withColumn(
        "facility_lower",
        F.lower(F.trim("facility"))
    )
    .join(
        F.broadcast(old_names_df),
        F.col("facility_lower") == F.col("old_facility"),
        "left_anti"
    )
    .drop("facility_lower")
)

# Search patterns used for candidate sampling.
# These are NOT evidence-validity labels.
patterns = {
    "Dialysis": r"(?i)dialysis|haemodialysis|hemodialysis|nephrology",
    "NICU": r"(?i)nicu|neonatal|newborn|incubator",
    "MRI": r"(?i)\bmri\b|magnetic resonance|radiology|imaging",
    "Trauma Care": r"(?i)trauma|accident|orthop|injur|fracture",
    "Burn Care": r"(?i)burn|plastic surgery|wound|skin graft",
    "General Emergency": r"(?i)emergency|casualty|urgent|ambulance",
    "Emergency C-section": r"(?i)cesarean|caesarean|c.section|lscs|obstetric|delivery|pregnan"
}

rows = []

for service, pattern in patterns.items():

    # Ten service-related candidates
    related = (
        claims
        .filter(F.col("claim").rlike(pattern))
        .orderBy(F.rand(seed=615))
        .limit(10)
        .collect()
    )

    # Ten candidates without these search terms
    other = (
        claims
        .filter(~F.col("claim").rlike(pattern))
        .orderBy(F.rand(seed=616))
        .limit(10)
        .collect()
    )

    selected = related + other

    for i, row in enumerate(selected, 1):

        rows.append({
            "review_id": f"{list(patterns).index(service)+1:02d}-{i:02d}",
            "service": service,
            "facility_id": row["unique_id"],
            "facility": row["facility"],
            "claim": row["claim"],
            "gold_label": ""
        })

validation = pd.DataFrame(rows)

# Save new statements without assigning labels
validation.to_csv(NEW_PATH, index=False)

print("Total validation statements:", len(validation))
print("\nStatements per service:")
print(validation["service"].value_counts())

print("\nSaved to:", NEW_PATH)

display(validation.head(15))


Total validation statements: 140

Statements per service:
service
Dialysis               20
NICU                   20
MRI                    20
Trauma Care            20
Burn Care              20
General Emergency      20
Emergency C-section    20
Name: count, dtype: int64

Saved to: /Volumes/workspace/default/care_finder_files/care_finder_phase5_validation.csv


review_id,service,facility_id,facility,claim,gold_label
01-01,Dialysis,b9386c11-f799-4fd3-9f1e-54f26b2e7b6b,"Government ESI Medical College, Coimbatore","Provides 24/7 services in nephrology, urology, laboratory, casualty, ICU, NICU, PICU, SICU and labour ICU",
01-02,Dialysis,bd31bab0-5c06-43a1-9766-4ad03f435930,Recovery Nursing Home,On-site dialysis unit,
01-03,Dialysis,77ac3849-daa9-4618-9094-e1d8e92dab5b,Sunrise Hospital,"Operates a wide range of departments across multiple specialties including Urology, Developmental Pediatrics, General Medicine, Nephrology, Maxillofacial Surgery, Psychiatry / Cognitive And Behavioural Sciences, Critical Care Medicine, Radiology / Radiodiagnosis And Imaging, Dentistry, Emergency Medicine, Gastroenterology, Geriatric Medicine, Family Medicine, Laboratory Medicine, Audiology, Occupational Therapy, Physiotherapy, Endocrinology, Minimal Access Surgery, General Surgery, Shoulder And Upper Limb Surgery, Sunrise Assisted Reproduction Centre (IVF), ENT, Minimally Invasive Thoracic Surgery, Paediatrics And Neonatology, Plastic Surgery, Pulmonology, Diabetology, Gynaecology, Obstetrics, and Infertility, Surgical Gastroenterology, Oncology, Kidney Transplant Unit, Neurology, Cardiology, Orthopaedics, Anaesthesiology, Dermatology And Cosmetology",
01-04,Dialysis,60694ed2-190d-43d7-9053-c9981ee74934,Aayush Hospitals,24x7 Hemodialysis,
01-05,Dialysis,1ec90a4b-3757-40ec-af97-33985fcc5996,"GBH American Hospital, Udaipur",On-site dialysis unit,
01-06,Dialysis,2ba1104c-e6a9-4c2e-8327-672a1b6c19e5,Savita Hospital,"Offers cardiology, cardiothoracic surgery, neurology, neurosurgery, orthopedics, nephrology, gynecology, gastroenterology, urology, pediatrics, and diabetology/diabetes care",
01-07,Dialysis,1589ba2e-ae31-4fe0-8931-725339adb118,Vedanta Super Speciality Hospital,Nephrology department,
01-08,Dialysis,9f3c6c60-8512-4a1d-be69-54b262d521eb,Medigo Multispeciality Hospital,"Offers orthopedics & trauma, general medicine, obstetrics and gynecology, pediatrics, urology, dentistry, anesthesiology & critical care, surgical gastroenterology, medical gastroenterology, ENT, dermatology, maxillofacial surgery, neurology, neurosurgery, cardiology, nephrology, dept of radiology, physiotherapy, rheumatology, ophthalmology",
01-09,Dialysis,ea87bf57-7560-41f9-8b03-7a7d33576338,"Marble City Hospital, Kishangarh, Ajmer","Hospital operates departments including Cardiology, General & Laparoscopy Surgery, General Medicine & Nephrology, Orthopedics, Ophthalmology, Obstetrics & Gynecology, Pediatrics & Neonatology, Anesthesiology, Dentistry, Physiotherapy, and Dietetics",
01-10,Dialysis,60d478ce-f2fd-4981-b18a-292785fdc401,Kims Hospital,"Specialisation includes Pathology, Hematology, Immunology, Pulmonology, Transplantation surgery, Cardiology, Nephrology, Hepatology, and Gastroenterology",


In [0]:

import pandas as pd

# Load the Phase 5 validation file
VALIDATION_PATH = str(DATA_DIR / "care_finder_phase5_provisional_labeled.csv")

validation = pd.read_csv(VALIDATION_PATH)

# Check that all statements and labels are present
print("Total statements:", len(validation))
print("Missing labels:", validation["gold_label"].isna().sum())

print("\nLabel distribution:")
print(validation["gold_label"].value_counts())

display(validation.head(10))


Total statements: 140
Missing labels: 0

Label distribution:
gold_label
unrelated_or_contradicted    77
supporting_only              46
direct                       17
Name: count, dtype: int64


review_id,service,facility_id,facility,claim,gold_label,review_note,annotation_source
01-01,Dialysis,b9386c11-f799-4fd3-9f1e-54f26b2e7b6b,"Government ESI Medical College, Coimbatore","Provides 24/7 services in nephrology, urology, laboratory, casualty, ICU, NICU, PICU, SICU and labour ICU",supporting_only,"Nephrology is listed, but dialysis itself is not stated.",AI-assisted provisional text review; requires human verification
01-02,Dialysis,bd31bab0-5c06-43a1-9766-4ad03f435930,Recovery Nursing Home,On-site dialysis unit,direct,null,AI-assisted provisional text review; requires human verification
01-03,Dialysis,77ac3849-daa9-4618-9094-e1d8e92dab5b,Sunrise Hospital,"Operates a wide range of departments across multiple specialties including Urology, Developmental Pediatrics, General Medicine, Nephrology, Maxillofacial Surgery, Psychiatry / Cognitive And Behavioural Sciences, Critical Care Medicine, Radiology / Radiodiagnosis And Imaging, Dentistry, Emergency Medicine, Gastroenterology, Geriatric Medicine, Family Medicine, Laboratory Medicine, Audiology, Occupational Therapy, Physiotherapy, Endocrinology, Minimal Access Surgery, General Surgery, Shoulder And Upper Limb Surgery, Sunrise Assisted Reproduction Centre (IVF), ENT, Minimally Invasive Thoracic Surgery, Paediatrics And Neonatology, Plastic Surgery, Pulmonology, Diabetology, Gynaecology, Obstetrics, and Infertility, Surgical Gastroenterology, Oncology, Kidney Transplant Unit, Neurology, Cardiology, Orthopaedics, Anaesthesiology, Dermatology And Cosmetology",supporting_only,"Nephrology and kidney transplant appear in a long specialty list, not dialysis.",AI-assisted provisional text review; requires human verification
01-04,Dialysis,60694ed2-190d-43d7-9053-c9981ee74934,Aayush Hospitals,24x7 Hemodialysis,direct,null,AI-assisted provisional text review; requires human verification
01-05,Dialysis,1ec90a4b-3757-40ec-af97-33985fcc5996,"GBH American Hospital, Udaipur",On-site dialysis unit,direct,null,AI-assisted provisional text review; requires human verification
01-06,Dialysis,2ba1104c-e6a9-4c2e-8327-672a1b6c19e5,Savita Hospital,"Offers cardiology, cardiothoracic surgery, neurology, neurosurgery, orthopedics, nephrology, gynecology, gastroenterology, urology, pediatrics, and diabetology/diabetes care",supporting_only,null,AI-assisted provisional text review; requires human verification
01-07,Dialysis,1589ba2e-ae31-4fe0-8931-725339adb118,Vedanta Super Speciality Hospital,Nephrology department,supporting_only,null,AI-assisted provisional text review; requires human verification
01-08,Dialysis,9f3c6c60-8512-4a1d-be69-54b262d521eb,Medigo Multispeciality Hospital,"Offers orthopedics & trauma, general medicine, obstetrics and gynecology, pediatrics, urology, dentistry, anesthesiology & critical care, surgical gastroenterology, medical gastroenterology, ENT, dermatology, maxillofacial surgery, neurology, neurosurgery, cardiology, nephrology, dept of radiology, physiotherapy, rheumatology, ophthalmology",supporting_only,"Nephrology appears as a listed specialty, not dialysis.",AI-assisted provisional text review; requires human verification
01-09,Dialysis,ea87bf57-7560-41f9-8b03-7a7d33576338,"Marble City Hospital, Kishangarh, Ajmer","Hospital operates departments including Cardiology, General & Laparoscopy Surgery, General Medicine & Nephrology, Orthopedics, Ophthalmology, Obstetrics & Gynecology, Pediatrics & Neonatology, Anesthesiology, Dentistry, Physiotherapy, and Dietetics",supporting_only,null,AI-assisted provisional text review; requires human verification
01-10,Dialysis,60d478ce-f2fd-4981-b18a-292785fdc401,Kims Hospital,"Specialisation includes Pathology, Hematology, Immunology, Pulmonology, Transplantation surgery, Cardiology, Nephrology, Hepatology, and Gastroenterology",supporting_only,null,AI-assisted provisional text review; requires human verification


In [0]:

import re
import pandas as pd
from transformers import pipeline
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Continue with the validation data loaded above.
validation = validation.copy()

# Same NLI model used in Phase 4
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1
)

# Same hypotheses used in Phase 4
hypotheses = {
    "Dialysis": "This facility provides dialysis treatment.",
    "NICU": "This facility operates a neonatal intensive care unit.",
    "MRI": "This facility provides MRI scans.",
    "Trauma Care": "This facility provides trauma care.",
    "Burn Care": "This facility provides specialized burn treatment.",
    "General Emergency": "This facility operates a general emergency department.",
    "Emergency C-section": "This facility performs emergency cesarean sections."
}

# Original Phase 4 safety rules — unchanged
def safety_check(row):

    service = str(row["service"])
    claim = str(row["claim"]).lower()
    facility = str(row["facility"]).lower()

    if re.search(
        r"listed as a top hospital|five.star government hospital",
        claim
    ):
        return "generic hospital listing"

    if (
        "district hospital, amravati" in claim
        and "district hospital" not in facility
    ):
        return "service attributed to another hospital"

    if service == "MRI":
        if re.search(r"\bnetwork\b", claim):
            return "network-wide claim, not branch-specific"

    if service == "Trauma Care":
        if re.fullmatch(
            r"\s*orthop[a-z]*\s*&\s*trauma\s*",
            claim
        ):
            return "specialty name without service details"

        if (
            "trauma & orthopaedics" in claim
            and "specialties including" in claim
        ):
            return "general specialty list"

    if service == "General Emergency":

        if "emergency booking" in claim:
            return "booking rather than clinical emergency care"

        if "mass casualty and disaster response" in claim:
            return "emergency preparedness, not established ED"

        if (
            "outpatient services across multiple specialties" in claim
            and "emergency medicine" in claim
        ):
            return "specialty listing does not establish ED"

    if service == "Emergency C-section":

        has_csection = bool(re.search(
            r"c[ -]?section|cesarean|caesarean|\blscs\b",
            claim
        ))

        has_emergency = bool(re.search(
            r"emergency|urgent|24[/x -]?7|24.hour",
            claim
        ))

        if not (has_csection and has_emergency):
            return "emergency C-section not established"

    return ""


# Run NLI on all 140 validation statements
nli_scores = []

for _, row in validation.iterrows():

    service = row["service"]

    labels = [
        hypotheses[service],
        "The statement does not establish that this facility provides the requested service."
    ]

    result = classifier(
        str(row["claim"]),
        candidate_labels=labels,
        multi_label=False,
        hypothesis_template="{}"
    )

    score_map = dict(zip(result["labels"], result["scores"]))

    nli_scores.append(
        float(score_map[labels[0]])
    )

validation["nli_score"] = nli_scores

validation["nli_pred"] = (
    validation["nli_score"] >= 0.5
).astype(int)

# Apply exactly the same safety rules
validation["safety_reason"] = validation.apply(
    safety_check,
    axis=1
)

validation["hybrid_pred"] = (
    (validation["nli_pred"] == 1) &
    (validation["safety_reason"] == "")
).astype(int)

# Evaluate against the provisional labels
y_true = (
    validation["gold_label"] == "direct"
).astype(int)

y_pred = validation["hybrid_pred"]

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

print("PHASE 5 — HYBRID VALIDATION RESULTS")
print("-----------------------------------")
print("Total statements:", len(validation))
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.3f}")
print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.3f}")
print(f"F1 Score:  {f1_score(y_true, y_pred, zero_division=0):.3f}")
print()
print("True Positives:", tp)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Negatives:", tn)

# Save predictions without overwriting the original
OUTPUT_PATH = str(DATA_DIR / "care_finder_phase5_predictions.csv")

validation.to_csv(OUTPUT_PATH, index=False)
print("\nPredictions saved to:", OUTPUT_PATH)


/local_disk0/.ephemeral_nfs/envs/pythonEnv-21e66402-bc8e-4787-8ed4-af8a6b692a24/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

PHASE 5 — HYBRID VALIDATION RESULTS
-----------------------------------
Total statements: 140
Precision: 0.400
Recall:    0.941
F1 Score:  0.561

True Positives: 16
False Positives: 24
False Negatives: 1
True Negatives: 99

Predictions saved to: /Volumes/workspace/default/care_finder_files/care_finder_phase5_predictions.csv


In [0]:

import pandas as pd
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Use the original Phase 4 regex patterns.
# First check whether the Phase 5 file already contains
# a regex_match column.

print("Available columns:")
print(validation.columns.tolist())

print(
    "\nRegex predictions available:",
    "regex_match" in validation.columns
)


Available columns:
['review_id', 'service', 'facility_id', 'facility', 'claim', 'gold_label', 'review_note', 'annotation_source', 'nli_score', 'nli_pred', 'safety_reason', 'hybrid_pred']

Regex predictions available: False


In [0]:

import re
import pandas as pd

# Reuse the regex patterns from Phase 4.
patterns = {
    "Dialysis": r"\b(?:dialysis|ha?emodialysis|peritoneal dialysis)\b",
    "NICU": r"\b(?:nicu|neonatal intensive care|newborn intensive care)\b",
    "MRI": r"\b(?:mri|magnetic resonance imaging)\b",
    "Trauma Care": r"\b(?:trauma|polytrauma|accident and emergency)\b",
    "Burn Care": r"\b(?:burns?|burn ward|plastic surgery)\b",
    "General Emergency": r"\b(?:emergency|casualty|emergencies)\b",
    "Emergency C-section": r"\b(?:c[ -]?section|cesarean|caesarean|lscs|obstetric|delivery)\b"
}

# Apply the Phase 4 regex approach again.
def regex_baseline(service, claim):
    return int(bool(re.search(
        patterns[service],
        str(claim),
        flags=re.IGNORECASE
    )))

# Calculate predictions for all 140 statements
validation["regex_match"] = validation.apply(
    lambda row: regex_baseline(
        row["service"],
        row["claim"]
    ),
    axis=1
)

print("Statements evaluated:", len(validation))
print("Regex matches:", validation["regex_match"].sum())

display(
    validation[
        [
            "review_id",
            "service",
            "claim",
            "gold_label",
            "regex_match"
        ]
    ].head(10)
)


Statements evaluated: 140
Regex matches: 26


review_id,service,claim,gold_label,regex_match
01-01,Dialysis,"Provides 24/7 services in nephrology, urology, laboratory, casualty, ICU, NICU, PICU, SICU and labour ICU",supporting_only,0
01-02,Dialysis,On-site dialysis unit,direct,1
01-03,Dialysis,"Operates a wide range of departments across multiple specialties including Urology, Developmental Pediatrics, General Medicine, Nephrology, Maxillofacial Surgery, Psychiatry / Cognitive And Behavioural Sciences, Critical Care Medicine, Radiology / Radiodiagnosis And Imaging, Dentistry, Emergency Medicine, Gastroenterology, Geriatric Medicine, Family Medicine, Laboratory Medicine, Audiology, Occupational Therapy, Physiotherapy, Endocrinology, Minimal Access Surgery, General Surgery, Shoulder And Upper Limb Surgery, Sunrise Assisted Reproduction Centre (IVF), ENT, Minimally Invasive Thoracic Surgery, Paediatrics And Neonatology, Plastic Surgery, Pulmonology, Diabetology, Gynaecology, Obstetrics, and Infertility, Surgical Gastroenterology, Oncology, Kidney Transplant Unit, Neurology, Cardiology, Orthopaedics, Anaesthesiology, Dermatology And Cosmetology",supporting_only,0
01-04,Dialysis,24x7 Hemodialysis,direct,1
01-05,Dialysis,On-site dialysis unit,direct,1
01-06,Dialysis,"Offers cardiology, cardiothoracic surgery, neurology, neurosurgery, orthopedics, nephrology, gynecology, gastroenterology, urology, pediatrics, and diabetology/diabetes care",supporting_only,0
01-07,Dialysis,Nephrology department,supporting_only,0
01-08,Dialysis,"Offers orthopedics & trauma, general medicine, obstetrics and gynecology, pediatrics, urology, dentistry, anesthesiology & critical care, surgical gastroenterology, medical gastroenterology, ENT, dermatology, maxillofacial surgery, neurology, neurosurgery, cardiology, nephrology, dept of radiology, physiotherapy, rheumatology, ophthalmology",supporting_only,0
01-09,Dialysis,"Hospital operates departments including Cardiology, General & Laparoscopy Surgery, General Medicine & Nephrology, Orthopedics, Ophthalmology, Obstetrics & Gynecology, Pediatrics & Neonatology, Anesthesiology, Dentistry, Physiotherapy, and Dietetics",supporting_only,0
01-10,Dialysis,"Specialisation includes Pathology, Hematology, Immunology, Pulmonology, Transplantation surgery, Cardiology, Nephrology, Hepatology, and Gastroenterology",supporting_only,0


In [0]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Provisional labels: only direct counts as positive
y_true = (
    validation["gold_label"] == "direct"
).astype(int)

results = []

for method, col in [
    ("Original Regex", "regex_match"),
    ("NLI", "nli_pred"),
    ("Hybrid", "hybrid_pred")
]:

    y_pred = validation[col].astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    results.append({
        "Method": method,
        "Precision": round(
            precision_score(y_true, y_pred, zero_division=0), 3
        ),
        "Recall": round(
            recall_score(y_true, y_pred, zero_division=0), 3
        ),
        "F1": round(
            f1_score(y_true, y_pred, zero_division=0), 3
        ),
        "False Positives": int(fp),
        "False Negatives": int(fn)
    })

print("PHASE 5 — METHOD COMPARISON")
display(pd.DataFrame(results))


PHASE 5 — METHOD COMPARISON


Method,Precision,Recall,F1,False Positives,False Negatives
Original Regex,0.654,1.0,0.791,9,0
NLI,0.39,0.941,0.552,25,1
Hybrid,0.4,0.941,0.561,24,1


In [0]:

# Find regex false positives on the new validation set
regex_fp = validation[
    (validation["regex_match"] == 1) &
    (validation["gold_label"] != "direct")
].copy()

print("Regex false positives:", len(regex_fp))

display(
    regex_fp[
        [
            "review_id",
            "service",
            "facility",
            "claim",
            "gold_label"
        ]
    ]
)


Regex false positives: 9


review_id,service,facility,claim,gold_label
05-02,Burn Care,Sri Manakula Vinayagar Medical College and Hospital,"Hospital offers Departmental Services in Anesthesiology, Cardiology, Cardiothoracic Surgery, Dental, Dermatology, ENT, General Medicine, Nephrology, Neurology, Neurosurgery, Medical Oncology, Obstetrics and Gynecology, Ophthalmology, Orthopaedics, Paediatrics, Plastic Surgery, Psychiatry, General Surgery, Pulmonary Medicine, Interventional Pulmonology, Radiology, Surgical Oncology, Urology",supporting_only
05-03,Burn Care,"MMI Narayana Multispeciality Hospital, Raipur","Departments include Cardiology, ENT, Neurosurgery, Urology, Gynaecology, Orthopaedics, Ophthalmology, Plastic Surgery, Gastroenterology, Nephrology, Surgical Oncology, General Surgery",supporting_only
05-04,Burn Care,Craftsman Plastic Reconstruction and Cosmetic Surgery Centre,Paediatric plastic surgery is offered,supporting_only
05-05,Burn Care,Yadav Super Speciality Hospital,Plastic surgery department,supporting_only
05-07,Burn Care,Vyas Medicity Super Speciality Hospital,Center of Excellence includes Plastic Surgery,supporting_only
05-08,Burn Care,Medicover Hospital,"Specialties include Cardiology and Cardiothoracic Surgery; Oncology (Medical, Surgical, and Radiation); Orthopedics and Joint Replacement; Pediatrics; Neonatology; Neurology and Neurosurgery; Gynecology and Obstetrics; Plastic Surgery; Critical Care; Emergency Medicine",supporting_only
05-09,Burn Care,Highland Super Speciality Hospital,Offers plastic surgery including cosmetic and reconstructive procedures,supporting_only
06-04,General Emergency,Kumar Dental Hospital,Provides emergency treatment,supporting_only
06-05,General Emergency,Divyadrashti Eye Hospital,Eye Injury & Emergency Eye Treatment,supporting_only


---

## Final application — Streamlit UI 7.8

The code below is the version used for the application. It is provided in full for the project-code submission. To run it, save the code as `app.py` in the repository and deploy it as a Databricks Streamlit app with the required dependencies and environment variables.

It requires `DATABRICKS_WAREHOUSE_ID` and Databricks permissions for `workspace.default.care_finder_facilities`. `GEOAPIFY_API_KEY` is optional for geocoding. Do not put secrets into this notebook or a public GitHub repository.

The app does not use the NLP pilot scores as a probability that care is available.

```python
import ast
import json
import math
import os
import re
from html import escape
from urllib.parse import urlencode, urlparse
from urllib.request import Request, urlopen

import pandas as pd
import pydeck as pdk
import streamlit as st
from databricks import sql
from databricks.sdk.core import Config

try:
    from streamlit_geolocation import streamlit_geolocation
except ImportError:
    streamlit_geolocation = None

# ============================================================
# CARE FINDER INDIA — UI 7.8
# Patient-friendly dark interface, explainable dataset search
# ============================================================

st.set_page_config(
    page_title="Care Finder India",
    page_icon="🏥",
    layout="wide",
    initial_sidebar_state="expanded",
)

VERSION = "UI 7.8"
TABLE = os.getenv("UC_TABLE_NAME", "workspace.default.care_finder_facilities")
RADIUS_KM = 60

SERVICES = [
    "Dialysis", "NICU", "MRI", "Trauma Care", "Burn Care",
    "General Emergency", "Emergency C-section",
]
EMERGENCY_SERVICES = ["General Emergency", "Trauma Care", "Emergency C-section"]
NAV_OPTIONS = [
    "Find Hospitals & Care", "View Hospitals on Map", "Compare Hospitals",
    "Find Emergency Care", "How It Works",
]

# Approximate city reference points; not facility-location verification.
CITY_CENTERS = {
    "patna": (25.5941, 85.1376), "hajipur": (25.6858, 85.2095),
    "jaipur": (26.9124, 75.7873), "lucknow": (26.8467, 80.9462),
    "unnao": (26.5471, 80.4878), "pune": (18.5204, 73.8567),
    "badlapur": (19.1552, 73.2655), "ahmedabad": (23.0225, 72.5714),
    "delhi": (28.6139, 77.2090), "new delhi": (28.6139, 77.2090),
    "noida": (28.5355, 77.3910), "ghaziabad": (28.6692, 77.4538),
    "hyderabad": (17.3850, 78.4867), "mumbai": (19.0760, 72.8777),
    "moradabad": (28.8386, 78.7733), "azamgarh": (26.0686, 83.1836),
}
CITY_STATES = {
    "patna": "bihar", "hajipur": "bihar", "jaipur": "rajasthan",
    "lucknow": "uttar pradesh", "unnao": "uttar pradesh",
    "pune": "maharashtra", "badlapur": "maharashtra",
    "ahmedabad": "gujarat", "delhi": "delhi", "new delhi": "delhi",
    "noida": "uttar pradesh", "ghaziabad": "uttar pradesh",
    "hyderabad": "telangana", "mumbai": "maharashtra",
    "moradabad": "uttar pradesh", "azamgarh": "uttar pradesh",
}
EVIDENCE_FIELDS = ("evidence_list", "procedure_list", "capability_list")
FIELD_LABELS = {
    "evidence_list": "General information",
    "procedure_list": "Procedures recorded",
    "capability_list": "Services recorded",
}

VALID_STATES = {
    "andhra pradesh", "arunachal pradesh", "assam", "bihar", "chhattisgarh",
    "goa", "gujarat", "haryana", "himachal pradesh", "jharkhand",
    "karnataka", "kerala", "madhya pradesh", "maharashtra", "manipur",
    "meghalaya", "mizoram", "nagaland", "odisha", "punjab",
    "rajasthan", "sikkim", "tamil nadu", "telangana", "tripura",
    "uttar pradesh", "uttarakhand", "west bengal",
    "andaman and nicobar islands", "chandigarh",
    "dadra and nagar haveli and daman and diu", "delhi",
    "jammu and kashmir", "ladakh", "lakshadweep", "puducherry",
}
STATE_ALIASES = {
    "up": "uttar pradesh", "u.p": "uttar pradesh",
    "mp": "madhya pradesh", "m.p": "madhya pradesh",
    "mh": "maharashtra", "tn": "tamil nadu", "wb": "west bengal",
    "orissa": "odisha", "uttaranchal": "uttarakhand",
    "nct of delhi": "delhi", "new delhi": "delhi",
    "j&k": "jammu and kashmir", "pondicherry": "puducherry",
}
INVALID_CITY_NAMES = {
    "unknown", "none", "null", "nan", "na", "n a", "not available",
    "not specified", "india", "other", "hospital", "clinic", "dentistry",
    "internalmedicine", "emergency", "dialysis", "mri", "nicu",
}

# ============================================================
# 1. PRESENTATION — dark mode only
# ============================================================

def apply_dark_theme():
    st.markdown("""
    <style>
    :root { color-scheme: dark; }
    .stApp, [data-testid="stAppViewContainer"], [data-testid="stMain"] {
        background: #120f1d !important; color: #f5f1ff !important;
    }
    [data-testid="stSidebar"] { background: #281b48 !important; }
    [data-testid="stSidebar"] * { color: #f5f1ff; }
    .block-container { padding-top: 1.7rem; padding-bottom: 3rem; max-width: 1450px; }
    .hero { background: linear-gradient(115deg,#382062,#7350b6); border-radius:18px;
            padding:29px; margin-bottom:22px; color:white; }
    .hero h1 { color:white !important; margin:0 0 9px; font-size:33px; }
    .hero p { color:#eee7ff !important; margin:0; font-size:15px; }
    .mini-card { background:#211a32; border:1px solid #413653; border-radius:15px;
                 padding:18px; margin-bottom:12px; min-height:115px; }
    .mini-label { color:#c1b8d5; font-size:12px; font-weight:650; text-transform:uppercase; }
    .mini-number { color:white; font-size:25px; font-weight:750; margin:5px 0; }
    .mini-note { color:#bab0cd; font-size:12px; }
    [data-testid="stMetricValue"] { color:#f5f1ff !important; }
    [data-testid="stMetricLabel"] { color:#c4bad5 !important; }
    div[data-baseweb="select"] > div, div[data-baseweb="input"] > div {
        background:#191523 !important; color:#f5f1ff !important; border-color:#514367 !important;
    }
    div[data-baseweb="popover"], div[data-baseweb="menu"], ul[role="listbox"] {
        background:#191523 !important; color:#f5f1ff !important;
    }
    div.stButton > button[kind="primary"],
    div.stFormSubmitButton > button[kind="primary"] {
        background:#ff4b55 !important; color:white !important;
        border:1px solid #ff4b55 !important; font-weight:650;
    }
    [data-testid="stAlert"], [data-testid="stExpander"] { border-radius:12px; }
    .compare-intro { background:#211a32; border:1px solid #413653;
                     border-radius:15px; padding:17px; margin-bottom:15px; }
    .compare-intro p { color:#cbc2dc; margin:0; }
    </style>
    """, unsafe_allow_html=True)


def hero(title, subtitle):
    st.markdown(
        f'<div class="hero"><h1>{escape(str(title))}</h1>'
        f'<p>{escape(str(subtitle))}</p></div>', unsafe_allow_html=True,
    )


def stat(label, value, note):
    st.markdown(
        f'<div class="mini-card"><div class="mini-label">{escape(str(label))}</div>'
        f'<div class="mini-number">{escape(str(value))}</div>'
        f'<div class="mini-note">{escape(str(note))}</div></div>',
        unsafe_allow_html=True,
    )


def distance_note():
    st.caption("Distances are approximate straight-line estimates. Actual travel distance may be longer.")


def safety_note():
    st.info(
        "These are possible care options based on recorded information. "
        "Services and current availability have not been confirmed. Contact facilities directly."
    )

# ============================================================
# 2. DATABRICKS AND RECORDS
# ============================================================

def table_name():
    if not re.fullmatch(r"[A-Za-z0-9_]+(?:\.[A-Za-z0-9_]+){2}", TABLE):
        raise ValueError("UC_TABLE_NAME must be catalog.schema.table")
    return TABLE


def query_rows(query, args=()):
    cfg = Config()
    host = cfg.host.replace("https://", "").replace("http://", "").rstrip("/")
    warehouse = os.environ["DATABRICKS_WAREHOUSE_ID"]
    with sql.connect(
        server_hostname=host,
        http_path=f"/sql/1.0/warehouses/{warehouse}",
        credentials_provider=lambda: cfg.authenticate,
        _use_arrow_native_complex_types=False,
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute(query, list(args))
            columns = [col[0] for col in cursor.description]
            return [dict(zip(columns, row)) for row in cursor.fetchall()]


FIELDS = """
unique_id, name, address_line1, address_city, address_stateOrRegion,
address_zipOrPostcode, phone_numbers, location_status,
latitude, longitude, evidence_list, procedure_list, capability_list,
source_urls_list, officialWebsite, repeated_name
"""


@st.cache_data(ttl=3600, show_spinner=False)
def cities_from_table():
    return query_rows(f"""
        SELECT TRIM(address_city) AS city,
               TRIM(address_stateOrRegion) AS state,
               COUNT(*) AS facility_count
        FROM {table_name()}
        WHERE address_city IS NOT NULL AND address_stateOrRegion IS NOT NULL
          AND TRIM(address_city) <> '' AND TRIM(address_stateOrRegion) <> ''
        GROUP BY TRIM(address_city), TRIM(address_stateOrRegion)
    """)


@st.cache_data(ttl=1800, show_spinner=False)
def nearby_rows(lat, lon, radius=RADIUS_KM):
    lat_delta = radius / 110.57
    lon_delta = radius / max(20, 111.32 * math.cos(math.radians(lat)))
    return query_rows(f"""
        SELECT {FIELDS} FROM {table_name()}
        WHERE latitude BETWEEN ? AND ? AND longitude BETWEEN ? AND ?
    """, [lat-lat_delta, lat+lat_delta, lon-lon_delta, lon+lon_delta])


@st.cache_data(ttl=1800, show_spinner=False)
def city_rows_raw(raw_pairs):
    """Use original database city/state variants, never cleaned display strings."""
    if not raw_pairs:
        return []
    pairs = tuple(raw_pairs)[:50]
    predicates = " OR ".join(
        ["(LOWER(TRIM(address_city)) = ? AND LOWER(TRIM(address_stateOrRegion)) = ?)"]
        * len(pairs)
    )
    args = [value for city, state in pairs for value in (city.lower(), state.lower())]
    return query_rows(
        f"SELECT {FIELDS} FROM {table_name()} WHERE {predicates}", args,
    )

# ============================================================
# 3. DATA HELPERS
# ============================================================

def _quoted_list_fallback(text):
    """Recover separate quoted items only when list syntax is recognizable.

    Do not split on commas: clinical descriptions may contain commas.
    If structural parsing is uncertain, return None and exclude the blob.
    """
    if not (text.startswith("[") and text.endswith("]")):
        return None

    inside = text[1:-1].strip()
    if not inside:
        return []

    token = re.compile(r"\"(?:\\.|[^\"\\])*\"|'(?:\\.|[^'\\])*'")
    matches = list(token.finditer(inside))
    if not matches:
        return None

    leftovers = token.sub("", inside)
    if leftovers.strip(" \t\r\n,"):
        return None

    values = []
    for match in matches:
        part = match.group(0)
        decoded = None
        for parser in (json.loads, ast.literal_eval):
            try:
                decoded = parser(part)
                break
            except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
                continue
        if not isinstance(decoded, str):
            return None
        values.append(decoded)
    return values


def parse_list(value):
    """Flatten real arrays, JSON/Python lists and double-encoded lists.

    Reject malformed array-looking blobs rather than treating them as
    a service statement; a giant list must never create a false match.
    """
    output = []

    def visit(obj, depth=0):
        if depth > 6 or obj is None or len(output) >= 500:
            return
        if isinstance(obj, (list, tuple, set)):
            for part in obj:
                visit(part, depth + 1)
            return
        if not isinstance(obj, str):
            return

        text = obj.strip()
        if not text or text.casefold() in {"none", "null", "nan"}:
            return

        # Try again after decoding a layer; some SQL connector values
        # are a JSON string whose contents are another JSON array.
        if text.startswith(("[", "\"[", "'[")):
            for parser in (json.loads, ast.literal_eval):
                try:
                    decoded = parser(text)
                except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
                    continue
                if decoded != obj:
                    visit(decoded, depth + 1)
                    return

            recovered = _quoted_list_fallback(text)
            if recovered is not None:
                visit(recovered, depth + 1)
                return

            # Unparsed list-like text must not be treated as one claim.
            return

        # Long aggregated claims are not safe evidence for ranking.
        if len(text) > 650:
            return
        output.append(text)

    visit(value)
    return output


def normalize_text(value):
    text = str(value or "")
    for old, new in (("×", "x"), ("✕", "x"), ("–", "-"), ("—", "-"), ("\u00a0", " ")):
        text = text.replace(old, new)
    return re.sub(r"\s+", " ", text).strip()


def claim_key(text):
    return re.sub(r"[^a-z0-9]+", " ", normalize_text(text).casefold()).strip()


def evidence_items(row):
    unique = {}
    for field in EVIDENCE_FIELDS:
        for claim in parse_list(row.get(field)):
            text = normalize_text(claim)
            key = claim_key(text)
            if not key:
                continue
            if key not in unique:
                unique[key] = {"text": text, "fields": [field]}
            elif field not in unique[key]["fields"]:
                unique[key]["fields"].append(field)
    return list(unique.values())


def source_urls(value):
    """Normalize duplicate URLs; cap output for readability."""
    unique = {}
    for entry in parse_list(value):
        for link in re.findall(r"https?://[^\s,)\]\"']+", entry):
            link = link.rstrip(".>;,')")
            parsed = urlparse(link)
            if parsed.scheme not in ("http", "https") or not parsed.netloc:
                continue
            # Equivalent URLs with different trailing slash/case are one link.
            key = (parsed.netloc.lower().removeprefix("www.")
                   + parsed.path.rstrip("/").lower()
                   + ("?" + parsed.query if parsed.query else ""))
            unique.setdefault(key, link)
    return list(unique.values())[:8]


# ============================================================
# 4. LOCATION — clean display labels, keep real database values
# ============================================================

def coordinates(row):
    try:
        lat, lon = float(row["latitude"]), float(row["longitude"])
        if all(map(math.isfinite, (lat, lon))) and 6 <= lat <= 38 and 68 <= lon <= 98:
            return (lat, lon)
    except (ValueError, TypeError, KeyError):
        pass
    return None


def distance_km(p, q):
    lat1, lon1 = map(math.radians, p)
    lat2, lon2 = map(math.radians, q)
    a = (math.sin((lat2-lat1)/2)**2
         + math.cos(lat1)*math.cos(lat2)*math.sin((lon2-lon1)/2)**2)
    return 6371 * 2 * math.asin(min(1, math.sqrt(a)))


def clean_city_name(value):
    city = normalize_text(value).strip(" ,.;:'\"[]")
    if not re.fullmatch(r"[A-Za-z][A-Za-z .'-]{1,59}", city):
        return None
    city = re.sub(r"\s+", " ", city).strip()
    if city.casefold() in INVALID_CITY_NAMES:
        return None
    return city.title()


def clean_state_name(value):
    state = normalize_text(value).strip(" ,.;:'\"[]").casefold()
    state = re.sub(r"\s+", " ", state)
    state = STATE_ALIASES.get(state, state)
    return state.title() if state in VALID_STATES else None


def build_clean_locations(city_data):
    """One displayed option per clean city/state, with original raw variants attached."""
    grouped = {}
    for row in city_data:
        raw_city, raw_state = row.get("city"), row.get("state")
        city = clean_city_name(raw_city)
        state = clean_state_name(raw_state)
        if not city:
            continue
        known_state = CITY_STATES.get(city.casefold())
        if not state:
            continue
        if known_state:
            expected = known_state.title()
            if state != expected:
                continue
            state = expected
        label = f"{city}, {state}"
        if label not in grouped:
            grouped[label] = {"city": city, "state": state, "raw_pairs": []}
        pair = (str(raw_city).strip(), str(raw_state).strip())
        if pair not in grouped[label]["raw_pairs"]:
            grouped[label]["raw_pairs"].append(pair)
    # Keep immutable pairs for cacheability and Streamlit session state.
    return {
        label: {"city": value["city"], "state": value["state"],
                "raw_pairs": tuple(value["raw_pairs"])}
        for label, value in sorted(grouped.items(), key=lambda it: it[0].casefold())
    }


@st.cache_data(ttl=86400, show_spinner=False)
def geocode_address(address):
    """Geoapify search. Only accept results within India."""
    key = os.getenv("GEOAPIFY_API_KEY", "").strip()
    if not key:
        raise ValueError("Address search requires GEOAPIFY_API_KEY.")
    params = urlencode({"text": address, "filter": "countrycode:in",
                        "format": "json", "limit": 3, "apiKey": key})
    request = Request("https://api.geoapify.com/v1/geocode/search?"+params,
                      headers={"User-Agent": "CareFinderIndia/7.8"})
    try:
        with urlopen(request, timeout=12) as response:
            matches = json.load(response).get("results", [])
    except Exception as exc:
        raise ValueError("Location lookup could not be completed.") from exc
    for item in matches:
        try:
            point = (float(item["lat"]), float(item["lon"]))
            country = str(item.get("country_code", "")).casefold()
            if 6 <= point[0] <= 38 and 68 <= point[1] <= 98 and country == "in":
                return point, item.get("formatted", address)
        except (KeyError, ValueError, TypeError):
            pass
    raise ValueError("No sufficiently clear Indian location was found.")


@st.cache_data(ttl=86400, show_spinner=False)
def geocode_city(city, state):
    """A city geocoder, not a hospital location verification service."""
    key = os.getenv("GEOAPIFY_API_KEY", "").strip()
    if not key:
        return None
    params = urlencode({"city": city, "state": state, "country": "India",
                        "filter": "countrycode:in", "format": "json",
                        "limit": 5, "apiKey": key})
    request = Request("https://api.geoapify.com/v1/geocode/search?"+params,
                      headers={"User-Agent": "CareFinderIndia/7.8"})
    try:
        with urlopen(request, timeout=12) as response:
            matches = json.load(response).get("results", [])
    except Exception:
        return None
    for item in matches:
        try:
            returned_city = normalize_text(item.get("city", "")).casefold()
            returned_state = clean_state_name(item.get("state", ""))
            point = (float(item["lat"]), float(item["lon"]))
            if (returned_city == city.casefold() and returned_state
                    and returned_state.casefold() == state.casefold()
                    and str(item.get("country_code", "")).casefold() == "in"
                    and 6 <= point[0] <= 38 and 68 <= point[1] <= 98):
                return point
        except (KeyError, ValueError, TypeError):
            continue
    return None


def search_center(meta):
    if "point" in meta:
        return meta["point"], meta.get("label", "Selected location")
    city, state = meta["city"], meta["state"]
    city_key = city.casefold()
    if city_key in CITY_CENTERS and CITY_STATES.get(city_key) == state.casefold():
        return CITY_CENTERS[city_key], f"Approximate {city} city center"

    # Prefer an independent city center over estimating it from hospital rows.
    mapped = geocode_city(city, state)
    if mapped:
        return mapped, f"Approximate {city} city center (city lookup)"

    # Fall back to original city/state values in Databricks (not display labels).
    rows = city_rows_raw(meta.get("raw_pairs", ()))
    points = [p for row in rows if (p := coordinates(row)) is not None]
    if not points:
        return None, "Not enough valid coordinates to locate this city"
    if len(points) == 1:
        # Available only as a provisional center, never claimed as verified.
        return points[0], f"Unverified {city} location from one facility record"

    clusters = [[p for p in points if distance_km(seed, p) <= 30]
                for seed in points]
    cluster = max(clusters, key=len)
    if len(cluster) < 2:
        return None, "City records have inconsistent coordinates"
    lat = float(pd.Series([p[0] for p in cluster]).median())
    lon = float(pd.Series([p[1] for p in cluster]).median())
    return (lat, lon), f"Estimated {city} center from facility records (unverified)"


def location_concern(row, search_meta=None):
    """Reject known geographically contradictory records; show uncertainty."""
    city = normalize_text(row.get("address_city")).casefold()
    state = clean_state_name(row.get("address_stateOrRegion"))
    state_key = state.casefold() if state else ""
    point = coordinates(row)
    if point is None:
        return "Missing or invalid coordinates."

    # A facility in a reference city cannot claim coordinates far away.
    if city in CITY_CENTERS:
        if CITY_STATES.get(city) != state_key:
            return "Recorded city and state are inconsistent."
        if distance_km(point, CITY_CENTERS[city]) > 35:
            return "Coordinates are far from the recorded city."
        return None

    if search_meta and "city" in search_meta:
        if city == search_meta["city"].casefold() and state_key == search_meta["state"].casefold():
            return None
        # If a different city is not covered by our reference data, only accept
        # the cross-city record when the dataset itself says its CITY is verified.
        status = normalize_text(row.get("location_status")).casefold()
        if "city verified" not in status or "not city verified" in status:
            return "Nearby record from a different city has unverified coordinates."
    return None


def full_address(row):
    pieces = []
    for field in ("address_line1", "address_city", "address_stateOrRegion"):
        part = normalize_text(row.get(field))
        if part and part.casefold() not in {"none", "null", "nan", "unknown"}:
            if not pieces or part.casefold() != pieces[-1].casefold():
                pieces.append(part)
    pin = normalize_text(row.get("address_zipOrPostcode"))
    if re.fullmatch(r"\d{6}", pin):
        pieces.append(pin)
    if pieces:
        pieces.append("India")
    return ", ".join(pieces)


def parsed_phone_numbers(value):
    """Accept plausible Indian dialable numbers; deduplicate a repeated array."""
    raw = parse_list(value)
    numbers = []
    for item in raw:
        for part in re.split(r"[,;]", item):
            digits = re.sub(r"\D", "", str(part))
            if len(digits) == 10:
                digits = "91" + digits
            elif len(digits) == 11 and digits.startswith("0"):
                digits = "91" + digits[1:]
            if len(digits) == 12 and digits.startswith("91"):
                phone = "+" + digits
                if phone not in numbers:
                    numbers.append(phone)
    return numbers[:4]


def location_note(status):
    status = normalize_text(status)
    if not status:
        return "This facility's map location has not been independently verified."
    if "not city verified" in status.casefold():
        return "Location is within India, but the facility's city and map pin are unverified."
    return "Recorded location status: " + status + ". Please confirm before traveling."

# ============================================================
# 5. MEDICAL PATTERNS AND SCREENING
# ============================================================

CSECTION = (r"\b(?:c[\s-]?sections?|caesarean|cesarean|caesarian|cesarian|lscs|"
            r"lower segment caesarean|lower segment cesarean)\b")
EMERGENCY = r"\b(?:emergency|emergencies|casualty|accident and emergency)\b"
HOURS = r"\b(?:24\s*(?:x|/)\s*7|24[\s-]*hours?|round[\s-]*the[\s-]*clock)\b"
PATTERNS = {
    "Dialysis": r"\b(?:dialysis|ha?emodialysis|peritoneal dialysis)\b",
    "NICU": r"\bnicu\b|\bneonatal intensive care\b|\bnewborn intensive care\b",
    "MRI": r"\bmri\b|\bmagnetic resonance imaging\b",
    "Trauma Care": (r"\btrauma (?:care|surgery|unit|cent(?:er|re)|management|services?)\b|"
                    r"\bpolytrauma\b|\bmajor trauma\b"),
    "Burn Care": r"\bburns?\b|\bburn injuries\b",
    "General Emergency": EMERGENCY,
    "Emergency C-section": CSECTION,
}
NEGATIVE = (r"\b(?:not available|not offered|not provided|unavailable|discontinued|"
            r"suspended|no longer|does not offer|does not provide|does not perform|no dedicated)\b")
PLANNED = r"\b(?:planned|proposed|coming soon|under construction|future facility|intends to offer)\b"
ARTIFACTS = [
    r"\?", r"\b(?:only if|if explicitly stated|if mentioned|when stated)\b",
    r"\b(?:provided content|source text|the text|not specified|not explicitly mentioned)\b",
    r"\b(?:check whether|verify whether|confirm whether|should be listed|does the hospital|is there evidence)\b",
    r"\b(?:no procedures listed|no explicit medical procedures)\b",
]


def is_artifact(text):
    return any(re.search(pattern, text, re.I) for pattern in ARTIFACTS)


def service_negative(text, service):
    for clause in re.split(r"[.;\n]", normalize_text(text)):
        matches = list(re.finditer(PATTERNS[service], clause, re.I))
        negs = list(re.finditer(NEGATIVE, clause, re.I))
        if any(abs(m.start()-n.start()) <= 85 for m in matches for n in negs):
            return True
    return False


def general_emergency(text, name):
    specialist = (r"\b(?:dental|dentistry|eye|ophthalmic|ophthalmology|veterinary)\b"
                  r".{0,45}\b(?:emergency|casualty)\b|"
                  r"\b(?:emergency|casualty)\b.{0,45}\b(?:dental|eye|ophthalmic)\b")
    if re.search(specialist, text, re.I):
        return False
    if re.search(r"\b(?:dental|eye|ophthalmic|veterinary)\b", name, re.I):
        return False
    direct = (r"\bemergency (?:department|unit|ward|medicine|services?|care)\b|"
              r"\baccident and emergency\b|\bcasualty (?:department|unit|services?)\b|"
              + HOURS + r".{0,50}\bemergency\b")
    return bool(re.search(direct, text, re.I))


def explicit_emergency_csection(text):
    direct = (
        r"\bemergency\s+(?:lower\s+segment\s+)?"
        r"(?:caesarean|cesarean|caesarian|cesarian|c[\s-]?section|lscs)\b|"
        r"\b(?:caesarean|cesarean|caesarian|cesarian|c[\s-]?section|lscs)"
        r"\s+(?:for\s+)?emergenc(?:y|ies)\b|"
        r"\bemergency\s+(?:obstetric\s+)?(?:surgery|delivery)\s*"
        r"\([^)]*(?:c[\s-]?section|caesarean|cesarean|lscs)[^)]*\)"
    )
    return bool(re.search(direct, normalize_text(text), re.I))


def mri_direct(text):
    text = normalize_text(text)
    without_center_name = re.sub(
        r"\bCT\s*(?:&|and)\s*MRI\s+Cent(?:er|re)\b",
        "diagnostic center", text, flags=re.I,
    )
    if not re.search(r"\b(?:mri|magnetic resonance imaging)\b", without_center_name, re.I):
        return False
    if (re.search(r"\b(?:first|pioneering|introduced|inaugurated|installed in)\b", text, re.I)
            and re.search(r"\b(?:19|20)\d{2}\b", text)):
        return False
    if re.search(r"\b(?:another hospital|other hospital|network hospitals?|partner hospital)\b", text, re.I):
        return False
    direct = (
        r"\b(?:provides?|provided|offers?|offered|performs?|performed|available|operates?|on[ -]?site)\b"
        r".{0,65}\b(?:mri|magnetic resonance imaging)\b|"
        r"\b(?:mri|magnetic resonance imaging)\b.{0,40}"
        r"\b(?:imaging|scans?|scanner|machine|services?|available|provided|performed|facility|cent(?:er|re))\b|"
        r"\b\d(?:\.\d)?\s*tesla\s+(?:digital\s+)?mri\b"
    )
    return bool(re.search(direct, without_center_name, re.I))


def trauma_direct(text, name):
    text = normalize_text(text)
    if name:
        text = re.sub(re.escape(normalize_text(name)), "the facility", text, flags=re.I)
    if re.search(r"\b(?:named|branding|listed as|called)\b", text, re.I):
        return False
    clinical = (
        r"\btrauma\s+(?:care|surgery|unit|ward|treatment|management|services?|team|resuscitation)\b|"
        r"\b(?:treats?|treating|treatment|management|surgery|care for).{0,40}\b(?:trauma|polytrauma)\b|"
        r"\b(?:polytrauma|major trauma)\s+(?:care|treatment|management|surgery|services?)\b"
    )
    return bool(re.search(clinical, text, re.I))


def classify(service, text, name):
    text = normalize_text(text)
    if not re.search(PATTERNS[service], text, re.I):
        return "none"
    if service_negative(text, service):
        return "negative"
    if is_artifact(text) or re.search(PLANNED, text, re.I):
        return "none"
    if service == "General Emergency":
        return "direct" if general_emergency(text, name) else "supporting"
    if service == "Burn Care":
        pattern = (r"\bburns? (?:care|unit|ward|cent(?:er|re)|treatment|management|"
                   r"services?|department|injur(?:y|ies))\b|"
                   r"\b(?:treatment|management|care|treats?|treating)\b.{0,45}\bburns?\b")
        return "direct" if re.search(pattern, text, re.I) else "supporting"
    if service == "Trauma Care":
        return "direct" if trauma_direct(text, name) else "supporting"
    if service == "Emergency C-section":
        return "direct" if explicit_emergency_csection(text) else "supporting"
    if service == "MRI":
        return "direct" if mri_direct(text) else "supporting"
    action = (r"\b(?:provides?|provided|offered|offers?|performs?|performed|available|has|"
              r"facility|department|unit|cent(?:er|re)|services?|equipment|machine|beds?|"
              r"treatment|operates?)\b")
    return "direct" if re.search(action, text, re.I) else "supporting"

# ============================================================
# 6. FACILITY ATTRIBUTION / CONFLICTS
# ============================================================

def normalized_words(value):
    ignored = {"hospital", "hospitals", "medical", "centre", "center", "super",
               "speciality", "specialty", "the", "of", "and", "at", "healthcare", "health"}
    return {word for word in re.findall(r"[a-z0-9]+", str(value).lower())
            if len(word) > 2 and word not in ignored}


def other_hospital_claim(claim, facility):
    pattern = (r"\b(?:at|operated by|provided by|performed at)\s+"
               r"([A-Za-z0-9&.'-]+(?:\s+[A-Za-z0-9&.'-]+){0,8}?\s+"
               r"(?:Hospital|Hospitals|Medical Cent(?:er|re)))\b")
    match = re.search(pattern, claim, re.I)
    if not match:
        return False
    own, other = normalized_words(facility), normalized_words(match.group(1))
    return bool(own and other and own.isdisjoint(other))


def facility_hold(row):
    name = normalize_text(row.get("name")).casefold()
    city = normalize_text(row.get("address_city")).casefold()
    if "aravind eye hospital" in name and "hyderabad" in city:
        return "Known mixed-source record; identity review required."
    if "st.joseph hospital" in name and "lucknow" in city:
        return "Record includes other facilities in Ghaziabad; identity review required."
    return None


def attribution_issue(row, claim):
    name = normalize_text(row.get("name")).casefold()
    city = normalize_text(row.get("address_city")).casefold()
    address = normalize_text(row.get("address_line1")).casefold()
    text = normalize_text(claim).casefold()
    if ("partha dental" in name and "kothapet" in (name+" "+city+" "+address)
            and "attapur" in text):
        return "Claim concerns a different branch."
    if other_hospital_claim(claim, row.get("name") or ""):
        return "Statement may describe another hospital."
    return None


def clean_evidence(row):
    hold = facility_hold(row)
    if hold:
        return [], [hold]
    accepted, cautions = [], []
    for item in evidence_items(row):
        text = item["text"]
        problem = attribution_issue(row, text)
        if problem:
            caution = problem + " This statement was excluded."
            if caution not in cautions:
                cautions.append(caution)
        elif is_artifact(text) or re.search(PLANNED, text, re.I):
            continue
        else:
            accepted.append(item)
    return accepted, cautions


def evaluate(row, service):
    items, warnings = clean_evidence(row)
    if facility_hold(row):
        return {"eligible": False, "level": "Hold for review", "direct": [],
                "supporting": [], "warnings": warnings}
    direct, supporting, negated = [], [], []
    for item in items:
        label = classify(service, item["text"], row.get("name") or "")
        if label == "direct":
            direct.append(item)
        elif label == "supporting":
            supporting.append(item)
        elif label == "negative":
            negated.append(item)
    if negated:
        warnings.append("Conflicting service information; record withheld.")
    level = ("Conflicting" if negated else "Direct wording" if direct
             else "Supporting only" if supporting else "No usable evidence")
    return {"eligible": bool(direct) and not negated, "level": level,
            "direct": direct, "supporting": supporting, "warnings": warnings}

# ============================================================
# 7. EMERGENCY C-SECTION LEADS
# ============================================================

def has_csection(text):
    return bool(re.search(CSECTION, normalize_text(text), re.I))


def strong_24h_emergency(text):
    text = normalize_text(text)
    if not (re.search(HOURS, text, re.I) and re.search(EMERGENCY, text, re.I)):
        return False
    if re.search(r"\b(?:dental|eye|ophthalmic|veterinary)\b.{0,45}\bemergency\b", text, re.I):
        return False
    if re.search(r"\bambulance\b", text, re.I) and not re.search(
        r"\bemergency (?:department|unit|ward|care|services?)\b", text, re.I
    ):
        return False
    return True


def other_emergency(text):
    text = normalize_text(text)
    pattern = (r"\bemergency (?:department|unit|area|ward|care|services?)\b|"
               r"\baccident and emergency\b|\bcasualty (?:department|unit)\b")
    specialist = r"\b(?:dental|eye|ophthalmic|veterinary)\b.{0,45}\bemergency\b"
    return bool(re.search(pattern, text, re.I)) and not bool(re.search(specialist, text, re.I))


def evaluate_csection(row):
    items, warnings = clean_evidence(row)
    output = {"category": "none", "explicit": [], "csection": [],
              "emergency": [], "warnings": warnings}
    if facility_hold(row):
        return output
    explicit, surgery, emergency_24, emergency_other = [], [], [], []
    for item in items:
        text = item["text"]
        if has_csection(text) and service_negative(text, "Emergency C-section"):
            output["warnings"] = warnings + ["Negative C-section wording; review required."]
            return output
        if re.search(EMERGENCY, text, re.I) and re.search(NEGATIVE, text, re.I):
            output["warnings"] = warnings + ["Potential negative emergency wording; review required."]
            return output
        if explicit_emergency_csection(text):
            explicit.append(item)
        else:
            if has_csection(text):
                surgery.append(item)
            if strong_24h_emergency(text):
                emergency_24.append(item)
            elif other_emergency(text):
                emergency_other.append(item)
    if explicit:
        category, emergency = "explicit", []
    elif surgery and emergency_24:
        category, emergency = "priority", emergency_24
    elif surgery and emergency_other:
        category, emergency = "possible", emergency_other
    else:
        category, emergency = "none", []
    output.update(category=category, explicit=explicit,
                  csection=surgery, emergency=emergency)
    return output

# ============================================================
# 8. RANKING AND SEARCH
# ============================================================

def ranking_points(evidence, km):
    text = " ".join(item["text"] for item in evidence["direct"])
    detail = 0
    if re.search(r"\b(?:provides?|provided|offers?|offered|performs?|performed|treatment)\b", text, re.I):
        detail += 7
    if re.search(r"\b(?:dedicated|unit|department|cent(?:er|re)|available)\b", text, re.I):
        detail += 6
    if re.search(r"\b(?:equipment|scanner|machine|beds?|capacity)\b", text, re.I):
        detail += 2
    detail = min(15, detail)
    proximity = round(max(0, 25*(1-km/RADIUS_KM)), 1)
    return {"evidence": 60, "details": detail, "proximity": proximity,
            "total": round(60 + detail + proximity, 1)}


def ranked_facilities(service, meta):
    center, origin_label = search_center(meta)
    counts = {"examined": 0, "outside_radius": 0, "location_hold": 0,
              "insufficient": 0, "conflicting": 0, "eligible": 0}
    if center is None:
        return [], counts, origin_label
    results = []
    for row in nearby_rows(*center):
        counts["examined"] += 1
        point = coordinates(row)
        if not point:
            continue
        km = distance_km(center, point)
        if km > RADIUS_KM:
            counts["outside_radius"] += 1
            continue
        if location_concern(row, meta):
            counts["location_hold"] += 1
            continue
        evidence = evaluate(row, service)
        if evidence["level"] == "Conflicting":
            counts["conflicting"] += 1
            continue
        if not evidence["eligible"]:
            counts["insufficient"] += 1
            continue
        points = ranking_points(evidence, km)
        website = str(row.get("officialWebsite") or "").strip()
        if not website.startswith(("https://", "http://")):
            website = ""
        results.append({
            "id": str(row.get("unique_id") or ""),
            "name": str(row.get("name") or "Unnamed facility"),
            "address": full_address(row),
            "phones": parsed_phone_numbers(row.get("phone_numbers")),
            "location_status": str(row.get("location_status") or ""),
            "city": str(row.get("address_city") or ""),
            "state": str(row.get("address_stateOrRegion") or ""),
            "lat": point[0], "lon": point[1], "km": round(km, 1),
            "level": evidence["level"], "direct": evidence["direct"],
            "supporting": evidence["supporting"], "warnings": evidence["warnings"],
            "sources": source_urls(row.get("source_urls_list")),
            "website": website, "score": points["total"], "parts": points,
        })
        counts["eligible"] += 1
    results.sort(key=lambda h: (-h["score"], h["km"], h["id"]))
    for i, item in enumerate(results, start=1):
        item["rank"] = i
    return results, counts, origin_label

# ============================================================
# 9. EVIDENCE AND FACILITY CARDS — NO NESTED EXPANDERS
# ============================================================

def evidence_display(item, show_fields=False):
    text = str(item.get("text", "")).strip()
    if not text:
        return
    st.markdown("> " + text)
    if show_fields:
        fields = [FIELD_LABELS.get(f, f) for f in item.get("fields", [])]
        st.caption("Recorded under: " + ", ".join(fields))


def score_explanation(item):
    parts = item["parts"]
    st.write(f"Service information: {parts['evidence']}/60")
    st.write(f"Additional service details: {parts['details']}/15")
    st.write(f"Distance: {parts['proximity']}/25")
    st.caption("This is a dataset match score, not a hospital quality or safety rating.")


def sources_display(item):
    st.caption(
        "These are links recorded with the facility. They may not "
        "support every service statement, or even refer to this branch."
    )
    official = item.get("website", "")
    if official:
        st.markdown(f"[Recorded facility website]({official})")

    links = []
    known = {official.rstrip("/").lower()} if official else set()
    for url in item.get("sources", []):
        if url.rstrip("/").lower() not in known:
            links.append(url)
            known.add(url.rstrip("/").lower())
        if len(links) >= 3:
            break

    if links:
        st.markdown("**Other links attached to this record**")
        for index, url in enumerate(links, 1):
            st.markdown(f"[{index}. {urlparse(url).netloc}]({url})")
    elif not official:
        st.write("No source links recorded for this facility.")


def map_link(item):
    return "https://www.google.com/maps/search/?api=1&" + urlencode(
        {"query": f"{item['lat']},{item['lon']}"}
    )


def contact_actions(item):
    """Works with older Streamlit versions: no link_button key argument."""
    phones = item.get("phones", [])
    if phones:
        st.markdown("**Contact numbers recorded for this facility**")
        st.write(" · ".join(phones))
    cols = st.columns(3)
    with cols[0]:
        if phones:
            st.markdown("[📞 Call Facility](tel:" + phones[0] + ")")
        else:
            st.caption("No phone number recorded")
    with cols[1]:
        st.link_button("📍 View on Map", map_link(item))
    with cols[2]:
        website = item.get("website", "")
        if website.startswith(("http://", "https://")):
            st.link_button("🌐 Website", website)
        else:
            st.caption("No website recorded")


def facility_card(item):
    with st.container(border=True):
        left, right = st.columns([3, 1])
        with left:
            st.markdown(f"### {item['rank']}. {item['name']}")
            st.caption(", ".join(x for x in (item["city"], item["state"]) if x))
            if item["address"]:
                st.write("📍 " + item["address"])
            st.caption("Relevant service information found. Current availability is unconfirmed.")
        with right:
            st.metric("Match Score", f"{item['score']:.1f}/100")
            st.metric("Distance", f"{item['km']:.1f} km")
        for warning in item["warnings"]:
            st.warning("Please verify this facility: " + warning)
        # Expanders must be siblings, not nested.
        with st.expander("Why is this facility listed?"):
            st.markdown("**Services listed for this facility**")
            for claim in item["direct"][:3]:
                evidence_display(claim)
            if item["supporting"]:
                st.markdown("**Other available information**")
                for claim in item["supporting"][:2]:
                    evidence_display(claim)
            st.caption("These statements come from the dataset and have not been independently confirmed.")
        with st.expander("How this match was calculated"):
            score_explanation(item)
        with st.expander("Where this information comes from"):
            sources_display(item)
            st.caption("Recorded dataset fields for the main service statements:")
            for claim in item["direct"][:3]:
                evidence_display(claim, show_fields=True)
            st.info("Contact the facility to confirm its address, services and ability to accept patients.")
        st.caption(location_note(item.get("location_status")))
        contact_actions(item)

# ============================================================
# 10. MAP AND COMPARISON
# ============================================================

def result_map(results, center=None):
    if not results:
        st.info("No matching facilities to show on the map.")
        return
    data = pd.DataFrame([{
        "name": h["name"], "lat": h["lat"], "lon": h["lon"],
        "km": h["km"], "score": h["score"], "rank": h["rank"],
    } for h in results[:50]])
    lat = center[0] if center else float(data["lat"].median())
    lon = center[1] if center else float(data["lon"].median())
    layer = pdk.Layer(
        "ScatterplotLayer", data=data, get_position="[lon, lat]", get_radius=550,
        get_fill_color=[117, 74, 197, 190], get_line_color=[255, 255, 255],
        line_width_min_pixels=1, pickable=True,
    )
    deck = pdk.Deck(
        layers=[layer], initial_view_state=pdk.ViewState(latitude=lat, longitude=lon, zoom=10),
        tooltip={"html": "<b>{name}</b><br/>Distance: {km} km<br/>Match: {score}"},
    )
    st.pydeck_chart(deck, use_container_width=True)
    distance_note()
    st.caption("Map pins use recorded coordinates, which may contain errors.")


def comparison_page(results, service, location):
    hero("Compare Hospitals", f"Compare potential {service} options near {location}")
    st.markdown(
        '<div class="compare-intro"><p>Compare how far facilities are and how well their '
        'recorded services match your search. These scores do not measure quality of care.</p></div>',
        unsafe_allow_html=True,
    )
    if not results:
        st.info("No matching facilities are available for comparison.")
        return
    a, b = st.columns([2, 1])
    with a:
        sort_choice = st.selectbox("Compare by", ["Best Match", "Closest First"])
    with b:
        count = st.selectbox("Number of facilities", [3, 5, 10], index=1)
    ordered = sorted(
        results,
        key=(lambda h: (h["km"], -h["score"])) if sort_choice == "Closest First"
        else (lambda h: (-h["score"], h["km"])),
    )
    selected = ordered[:count]
    rows = [{
        "Facility": h["name"], "Distance (km)": h["km"],
        "Match Score": round(h["score"], 1),
        "Additional record warning": "Yes" if h["warnings"] else "No",
    } for h in selected]
    st.dataframe(
        pd.DataFrame(rows), hide_index=True, use_container_width=True,
        column_config={
            "Facility": st.column_config.TextColumn("Facility", width="large"),
            "Distance (km)": st.column_config.NumberColumn("Distance", format="%.1f km"),
            "Match Score": st.column_config.ProgressColumn(
                "Match Score", min_value=0, max_value=100, format="%.1f/100",
            ),
            "Additional record warning": st.column_config.TextColumn(
                "Record warning", help="All services are unverified. 'Yes' means an additional concern was found.",
            ),
        },
    )
    distance_note()
    st.caption("Higher match scores reflect text evidence and distance, not clinical quality or availability.")
    st.subheader("Explore individual facilities")
    st.write("Open a facility to see the recorded information behind its match.")
    for h in selected:
        label = f"{h['name']} · {h['km']:.1f} km · Match {h['score']:.1f}/100"
        with st.expander(label):
            st.markdown("**Services listed for this facility**")
            for claim in h["direct"][:3]:
                evidence_display(claim)
            for warning in h["warnings"]:
                st.warning("Please verify: " + warning)
            st.caption("Recorded service descriptions are not independently verified.")
            st.write("📍 " + h["address"])
            st.caption(location_note(h.get("location_status")))
            contact_actions(h)
    with st.expander("How are hospitals compared?"):
        st.write("Service wording: up to 60 points.")
        st.write("Additional recorded details: up to 15 points.")
        st.write("Distance: up to 25 points.")
        st.caption("This is not a measure of clinical quality, safety or live availability.")

# ============================================================
# 11. EMERGENCY CARE
# ============================================================

def csection_page(meta):
    center, label = search_center(meta)
    if center is None:
        st.warning("We couldn't determine a sufficiently reliable location for this search. " + label)
        return
    grouped = {"explicit": [], "priority": [], "possible": []}
    held_locations = 0
    for row in nearby_rows(*center):
        point = coordinates(row)
        if point is None or distance_km(center, point) > RADIUS_KM:
            continue
        if location_concern(row, meta):
            held_locations += 1
            continue
        evidence = evaluate_csection(row)
        category = evidence["category"]
        if category == "none":
            continue
        grouped[category].append({
            "name": str(row.get("name") or "Unnamed facility"),
            "city": str(row.get("address_city") or ""),
            "address": full_address(row),
            "phones": parsed_phone_numbers(row.get("phone_numbers")),
            "website": str(row.get("officialWebsite") or ""),
            "location_status": str(row.get("location_status") or ""),
            "distance": round(distance_km(center, point), 1),
            "lat": point[0], "lon": point[1], **evidence,
        })
    for values in grouped.values():
        values.sort(key=lambda x: x["distance"])
    st.error("For an active medical emergency, seek emergency services immediately. "
             "This directory cannot confirm operating-room readiness or patient acceptance.")
    st.subheader("Facilities with related service information")
    st.write("We look for C-section and emergency-care statements. "
             "None of these entries establishes current emergency surgical availability.")
    cols = st.columns(3)
    metrics = [
        ("Specific C-section emergency claims", len(grouped["explicit"]), "Still unverified"),
        ("Facilities to check first", len(grouped["priority"]), "C-section + 24-hour emergency statements"),
        ("Other facilities to check", len(grouped["possible"]), "Less specific emergency information"),
    ]
    for col, values in zip(cols, metrics):
        with col:
            stat(*values)
    sections = [
        ("1. Explicit emergency C-section information", "explicit",
         "A specific dataset statement does not confirm current availability."),
        ("2. Facilities with separate C-section and 24-hour emergency claims", "priority",
         "These are verification leads, not confirmed emergency surgery providers."),
        ("3. Other possible facilities to contact", "possible",
         "These records include weaker emergency-related statements."),
    ]
    for title, category, description in sections:
        st.subheader(title)
        st.write(description)
        if not grouped[category]:
            st.info("No matching records in this category.")
        for h in grouped[category][:12]:
            with st.container(border=True):
                left, right = st.columns([3, 1])
                with left:
                    st.markdown("### " + h["name"])
                    st.caption(h["city"])
                    if h["address"]:
                        st.write("📍 " + h["address"])
                with right:
                    st.metric("Distance", f"{h['distance']:.1f} km")
                if category == "explicit":
                    st.markdown("**Emergency C-section statements**")
                    for claim in h["explicit"][:4]:
                        evidence_display(claim)
                else:
                    st.markdown("**C-section services mentioned**")
                    for claim in h["csection"][:3]:
                        evidence_display(claim)
                    st.markdown("**Emergency care mentioned**")
                    for claim in h["emergency"][:3]:
                        evidence_display(claim)
                for warning in h["warnings"]:
                    st.warning("Please verify: " + warning)
                st.info("Confirm emergency obstetric surgery capability directly before relying on this facility.")
                st.caption(location_note(h.get("location_status")))
                contact_actions(h)
    if not any(grouped.values()):
        st.warning("No matching records were found. This does not mean the care is unavailable.")
    if held_locations:
        st.caption(f"{held_locations} location records did not pass the available consistency checks.")
    distance_note()

# ============================================================
# 12. APP / SIDEBAR
# ============================================================

apply_dark_theme()

try:
    city_data = cities_from_table()
except Exception as exc:
    st.error("Could not connect to the Databricks facility database.")
    st.code(str(exc))
    st.stop()

locations = build_clean_locations(city_data)
city_options = list(locations)
if "Patna, Bihar" in city_options:
    city_options.remove("Patna, Bihar")
    city_options.insert(0, "Patna, Bihar")
if not city_options:
    st.error("No usable city/state entries were found.")
    st.stop()

# Migrate old navigation labels and avoid touching instantiated widget state.
old_nav = {
    "Explore care": "Find Hospitals & Care", "Map": "View Hospitals on Map",
    "Compare": "Compare Hospitals", "Emergency evidence": "Find Emergency Care",
    "Our method": "How It Works",
}
if "nav" not in st.session_state:
    st.session_state.nav = NAV_OPTIONS[0]
if st.session_state.nav not in NAV_OPTIONS:
    st.session_state.nav = old_nav.get(st.session_state.nav, NAV_OPTIONS[0])
if "pending_nav" in st.session_state:
    st.session_state.nav = st.session_state.pop("pending_nav")

with st.sidebar:
    st.markdown(
        '<div style="font-size:26px;font-weight:800;color:white">✚ Care Finder</div>'
        '<div style="font-size:12px;color:#d4c6ef;margin-bottom:18px">'
        'HEALTHCARE DISCOVERY · INDIA</div>', unsafe_allow_html=True,
    )
    st.markdown("### Find care")
    search_mode = st.radio("Choose your location", ["City", "Address / landmark", "My location"])
    if search_mode == "City":
        with st.form("city_form"):
            service = st.selectbox("What care do you need?", SERVICES)
            chosen = st.selectbox("Where do you need care?", city_options)
            submitted = st.form_submit_button("Find Nearby Care", type="primary", use_container_width=True)
        if submitted:
            st.session_state.search = (service, chosen, locations[chosen])
            st.session_state.pending_nav = "Find Hospitals & Care"
            st.rerun()
    elif search_mode == "Address / landmark":
        with st.form("address_form"):
            service = st.selectbox("What care do you need?", SERVICES)
            address = st.text_input("Where do you need care?", placeholder="Address, PIN code or landmark")
            submitted = st.form_submit_button("Find Nearby Care", type="primary", use_container_width=True)
        if submitted:
            try:
                if not address.strip():
                    raise ValueError("Please enter an address or landmark.")
                point, label = geocode_address(address.strip())
                st.session_state.search = (service, label, {"point": point, "label": label})
                st.session_state.pending_nav = "Find Hospitals & Care"
                st.rerun()
            except ValueError as exc:
                st.warning(str(exc))
    else:
        shared = streamlit_geolocation() if streamlit_geolocation else None
        with st.form("device_form"):
            service = st.selectbox("What care do you need?", SERVICES)
            submitted = st.form_submit_button("Find Care Near Me", type="primary", use_container_width=True)
        if submitted:
            try:
                point = (float(shared["latitude"]), float(shared["longitude"]))
                if not (6 <= point[0] <= 38 and 68 <= point[1] <= 98):
                    raise ValueError("Location is outside India.")
                st.session_state.search = (service, "My location", {"point": point, "label": "My location"})
                st.session_state.pending_nav = "Find Hospitals & Care"
                st.rerun()
            except (TypeError, KeyError, ValueError):
                st.warning("Share a location within India or choose a city instead.")
        if streamlit_geolocation is None:
            st.caption("Device location is not available. Search by city or address.")
    st.divider()
    st.markdown("### Explore")
    st.radio("Navigation", NAV_OPTIONS, key="nav", label_visibility="collapsed")
    st.divider()
    st.caption("Service availability must be confirmed directly.")

# ============================================================
# 13. PAGES
# ============================================================

nav = st.session_state.nav

if nav == "How It Works":
    hero("How Care Finder Works", "Learn how facilities are matched to your search")
    st.markdown("### How to use Care Finder")
    st.write("**1. Choose the care you need.** Select a service and location.")
    st.write("**2. We search facility information.** The dataset contains descriptions of healthcare services.")
    st.write("**3. We check the statements.** We screen out some unclear, repeated, or conflicting claims.")
    st.write("**4. We show possible facilities.** We consider service wording and recorded distance.")
    st.write("**5. You can inspect each result.** Review the original statements before contacting a facility.")
    st.info("A match score is not a hospital rating or confirmation that treatment is available.")
    with st.expander("Technical approach and exploratory evaluation"):
        st.write("The live application uses explainable service-specific rules and geographic checks.")
        st.dataframe(pd.DataFrame({
            "Method": ["Regex", "NLI", "Hybrid"],
            "Precision": [0.654, 0.390, 0.400],
            "Recall": [1.000, 0.941, 0.941],
            "F1": [0.791, 0.552, 0.561],
        }), hide_index=True, use_container_width=True)
        st.caption("Exploratory Phase 5 results on 140 statements with AI-assisted, manually reviewed labels. "
                   "These figures do not evaluate this deployed version or establish clinical accuracy.")
    st.markdown("### Limitations")
    st.write("Records can be incomplete or misattributed; some city coordinates are uncertain. "
             "Current staffing, emergency readiness, and service availability are not verified.")
    st.stop()

if nav == "Find Emergency Care":
    hero("Find Emergency Care", "Find records mentioning emergency services near your selected city")
    st.error("If someone needs urgent help, contact local emergency services immediately. "
             "Do not delay care while searching this directory.")
    with st.form("emergency_search_form"):
        a, b = st.columns(2)
        with a:
            emergency_service = st.selectbox("What emergency care do you need?", EMERGENCY_SERVICES)
        with b:
            emergency_city = st.selectbox("Where do you need care?", city_options)
        submitted = st.form_submit_button("Find Emergency Care Options", type="primary")
    if submitted:
        st.session_state.emergency_query_data = (emergency_service, emergency_city)
    if "emergency_query_data" in st.session_state:
        selected_service, selected_city = st.session_state.emergency_query_data
        if selected_city not in locations:
            st.warning("This location has changed. Please choose it again.")
            st.stop()
        meta = locations[selected_city]
        if selected_service == "Emergency C-section":
            csection_page(meta)
        else:
            results, audit, origin = ranked_facilities(selected_service, meta)
            if origin.startswith("Not enough"):
                st.warning(origin + ". Search a nearby city or use an address if available.")
            st.write(f"**{len(results)} potential care options** near {selected_city}")
            st.warning("Emergency readiness and patient acceptance are not confirmed.")
            for item in results[:8]:
                facility_card(item)
            if not results:
                st.info("No matching records found here. This does not mean the service is unavailable.")
            distance_note()
    st.stop()

if "search" not in st.session_state:
    hero("Find the Care You Need, Near You.",
         "Find healthcare facilities in India and understand why they match your search")
    cols = st.columns(3)
    for col, metric in zip(cols, [
        ("Types of Care", len(SERVICES), "Searchable medical services"),
        ("Search Area", "60 km", "Around your selected location"),
        ("Availability", "Not Confirmed", "Contact facilities directly"),
    ]):
        with col:
            stat(*metric)
    st.info("Choose a service and location on the left, then select Find Nearby Care.")
    st.stop()

service, location, meta = st.session_state.search

if service == "Emergency C-section":
    hero("Emergency C-section Care Options", f"{location} · recorded facility information")
    csection_page(meta)
    st.stop()

try:
    results, audit, center_label = ranked_facilities(service, meta)
except Exception as exc:
    st.error("We couldn't retrieve facility information.")
    st.code(str(exc))
    st.stop()

# Give the selected page its own heading, avoiding duplicate 'Explore care' banners.
if nav == "Find Hospitals & Care":
    hero("Hospitals & Care Options Near You", f"{service} · {location} · within {RADIUS_KM} km")
    nearest = min((h["km"] for h in results), default=None)
    cols = st.columns(3)
    metrics = [
        ("Potential Care Options", len(results), "Based on recorded service information"),
        ("Closest Facility", f"{nearest:.1f} km" if nearest is not None else "—", "Approximate distance"),
        ("Search Area", f"{RADIUS_KM} km", "From the selected location"),
    ]
    for col, metric in zip(cols, metrics):
        with col:
            stat(*metric)
    st.caption("Search location: " + center_label)
    if "unverified" in center_label.lower():
        st.warning("The city center is estimated from facility coordinates, not independently verified. "
                   "Distances and nearby results may be inaccurate. Try an address search for a more precise starting point.")
    safety_note()
    distance_note()
    if not results:
        if center_label.startswith(("Not enough", "City records")):
            st.warning("We couldn't reliably locate the selected city using its facility records. "
                       "Try a nearby larger city or use an address search.")
        else:
            st.warning("No matching facility information was found within this search area. "
                       "Some records may lack service details or have unverified coordinates. "
                       "This doesn't prove that care is unavailable.")
        st.stop()
    st.subheader("Care Options for You")
    a, b = st.columns([3, 1])
    with a:
        order = st.selectbox("Sort by", ["Best Evidence Match", "Closest First"])
    with b:
        count = st.selectbox("Show", [3, 5, 10, "All"])
    ordered = (sorted(results, key=lambda h: (h["km"], -h["score"]))
               if order == "Closest First" else results)
    displayed = ordered if count == "All" else ordered[:int(count)]
    for item in displayed:
        facility_card(item)
    with st.expander("Search Details"):
        st.write("Facility records checked:", audit["examined"])
        st.write("Outside search area:", audit["outside_radius"])
        st.write("Withheld due to location uncertainty:", audit["location_hold"])
        st.write("Insufficient service information / identity concerns:", audit["insufficient"])
        st.write("Conflicting service information:", audit["conflicting"])
        st.write("Potential options listed:", audit["eligible"])

elif nav == "View Hospitals on Map":
    hero("View Hospitals on Map", f"{service} · {location}")
    if not results:
        st.info("No matching facilities to map for this search.")
    else:
        center, _ = search_center(meta)
        result_map(results, center)

elif nav == "Compare Hospitals":
    comparison_page(results, service, location)

st.caption(f"Care Finder India · {VERSION} · Virtue Foundation dataset · "
           "Databricks + Streamlit · Healthcare discovery prototype")
```

## What this project does not verify

- The dataset does not guarantee that a facility currently provides the listed service.
- A 24-hour emergency-care statement and a separate C-section statement do not establish emergency C-section readiness.
- The distance calculation uses recorded coordinates and straight-line distance, not road travel time.
- The text-label evaluations do not establish clinical accuracy or current service availability.

**Possible next steps:** better location reference data, verified hospital updates, road travel times, language support, and improved device-location integration.